# AI-Powered SME Decision Support Platform

منصة لدعم قرارات المشاريع الصغيرة والمتوسطة في الأردن.

**ملاحظات منهجية**
- النظام أداة **Decision Support** وليس ضمانًا لنجاح المشروع.
- `Location Suitability Score` هو **Screening score** وليس احتمال نجاح.
- مؤشر الطلب الأسري من **HEIS 2017** تاريخي، ويُستخدم فقط عندما توجد مطابقة مناسبة للنشاط.
- أوزان/حدود الـPrototype وقواعد التنبيه قابلة للتعديل وليست ثوابت علمية عالمية.


In [ ]:
# ============================================
# 1) INSTALL + IMPORTS
# ============================================

!pip -q install sentence-transformers transformers sentencepiece scikit-learn

import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import torch

from google.colab import files
from IPython.display import display, HTML, clear_output
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Libraries ready")


✅ Libraries ready


In [ ]:
# ============================================
# 2) LOAD MASTER WORKBOOK
# ============================================

DEFAULT_FILE = "SME_AI_Master_Data_v4_Location_Scoring.xlsx"

if os.path.exists(DEFAULT_FILE):
    FILE_NAME = DEFAULT_FILE
else:
    print("📁 ارفعي ملف SME_AI_Master_Data_v4_Location_Scoring.xlsx")
    uploaded = files.upload()

    xlsx_files = [
        name for name in uploaded.keys()
        if name.lower().endswith(".xlsx")
    ]

    if not xlsx_files:
        raise FileNotFoundError(
            "لم يتم رفع ملف Excel. ارفعي ملف البيانات الرئيسي ثم أعيدي تشغيل الخلية."
        )

    FILE_NAME = (
        DEFAULT_FILE
        if DEFAULT_FILE in xlsx_files
        else xlsx_files[0]
    )

governorates = pd.read_excel(
    FILE_NAME,
    sheet_name="Governorate Master"
)

table4 = pd.read_excel(
    FILE_NAME,
    sheet_name="Table4 Structured"
)

demand = pd.read_excel(
    FILE_NAME,
    sheet_name="Demand Proxy 2017",
    usecols="A:P",
    nrows=28
)

print("✅ Data loaded")
print("Governorates:", governorates.shape)
print("Activities:", table4.shape)
print("Demand categories:", demand.shape)


📁 ارفعي ملف SME_AI_Master_Data_v4_Location_Scoring.xlsx


Saving SME_AI_Master_Data_v4_Location_Scoring.xlsx to SME_AI_Master_Data_v4_Location_Scoring.xlsx
✅ Data loaded
Governorates: (12, 18)
Activities: (382, 18)
Demand categories: (28, 16)


In [ ]:
import os

files_to_check = [
    "/content/app_BUNYA.py",
    "/content/SME_AI_Master_Data_v4_Location_Scoring.xlsx"
]

for f in files_to_check:
    print("موجود" if os.path.exists(f) else "مش موجود", ":", f)

مش موجود : /content/app_BUNYA.py
موجود : /content/SME_AI_Master_Data_v4_Location_Scoring.xlsx


In [ ]:
# ============================================
# 3) CORE DATA PREPARATION
# ============================================

def normalize_isic4(value):
    if pd.isna(value):
        return None

    code = str(value).strip()

    if code.endswith(".0"):
        code = code[:-2]

    if code.isdigit():
        code = code.zfill(4)

    return code


table4["ISIC Code"] = table4["ISIC Code"].apply(
    normalize_isic4
)

activities = table4[
    table4["Level"] == "Detailed activity (4-digit)"
].copy()

activities["ISIC Code"] = activities[
    "ISIC Code"
].apply(normalize_isic4)


governorate_map = {
    "عمان": "Amman",
    "البلقاء": "Balqa",
    "الزرقاء": "Zarqa",
    "مادبا": "Madaba",
    "إربد": "Irbid",
    "المفرق": "Mafraq",
    "جرش": "Jarash",
    "عجلون": "Ajlun",
    "الكرك": "Karak",
    "الطفيلة": "Tafiela",
    "معان": "Ma'an",
    "العقبة": "Aqaba"
}


def normalize(series, reverse=False):
    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            [50] * len(series),
            index=series.index
        )

    score = (
        (series - minimum)
        / (maximum - minimum)
        * 100
    )

    if reverse:
        score = 100 - score

    return score


def get_business_family(isic_code):
    code = normalize_isic4(isic_code)

    if not code or not code[:2].isdigit():
        return "other"

    division = int(code[:2])

    if 1 <= division <= 3:
        return "agriculture"
    elif 5 <= division <= 9:
        return "mining"
    elif 10 <= division <= 33:
        return "manufacturing"
    elif 35 <= division <= 39:
        return "utilities_environment"
    elif 41 <= division <= 43:
        return "construction"
    elif 45 <= division <= 47:
        return "retail_trade"
    elif 49 <= division <= 53:
        return "transport_logistics"
    elif 55 <= division <= 56:
        return "hospitality_food"
    elif 58 <= division <= 63:
        return "information_technology"
    elif 64 <= division <= 66:
        return "financial_services"
    elif division == 68:
        return "real_estate"
    elif 69 <= division <= 75:
        return "professional_services"
    elif 77 <= division <= 82:
        return "business_support"
    elif division == 85:
        return "education"
    elif 86 <= division <= 88:
        return "health_social"
    elif 90 <= division <= 93:
        return "entertainment"
    elif 94 <= division <= 96:
        return "personal_services"
    return "other"


business_family_labels = {
    "agriculture": "الزراعة",
    "mining": "التعدين",
    "manufacturing": "التصنيع",
    "utilities_environment": "المياه والطاقة والبيئة",
    "construction": "الإنشاءات",
    "retail_trade": "التجارة والبيع",
    "transport_logistics": "النقل والخدمات اللوجستية",
    "hospitality_food": "المطاعم والطعام والضيافة",
    "information_technology": "تكنولوجيا المعلومات",
    "financial_services": "الخدمات المالية",
    "real_estate": "العقارات",
    "professional_services": "الخدمات المهنية",
    "business_support": "خدمات الأعمال",
    "education": "التعليم",
    "health_social": "الصحة والخدمات الاجتماعية",
    "entertainment": "الترفيه",
    "personal_services": "الخدمات الشخصية",
    "other": "أخرى"
}


activity_catalog = activities[
    ["ISIC Code", "Economic Activity"]
].dropna().drop_duplicates(
    subset=["ISIC Code"]
).reset_index(drop=True)

activity_catalog["Business Family"] = (
    activity_catalog["ISIC Code"]
    .apply(get_business_family)
)

# Search text is kept local to the workbook for a clean/reproducible final file.
activity_catalog["Search Text"] = (
    activity_catalog["Economic Activity"]
    .fillna("")
    .astype(str)
)

print("✅ Core data prepared")
print("Detailed activities:", len(activity_catalog))


✅ Core data prepared
Detailed activities: 304


In [ ]:
# ============================================
# 4) AI MODELS: SEMANTIC SEARCH + TRANSLATION + SENTIMENT
# ============================================

semantic_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-ar-en"

translation_tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL
)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

translation_model = translation_model.to(device)

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

demand_categories = (
    demand["Category (EN)"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

demand_vectors = semantic_model.encode(
    demand_categories,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("✅ AI models ready")
print("Device:", device)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

✅ AI models ready
Device: cpu


In [ ]:
# ============================================
# 5) ACTIVITY SEARCH + PROJECT INPUT
# ============================================

def translate_search_text(text):
    text = str(text).strip()

    if text == "":
        return ""

    inputs = translation_tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        generated = translation_model.generate(
            **inputs,
            max_length=128
        )

    return translation_tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )


def search_activities_in_family(
    business_family,
    search_text,
    top_k=5
):
    search_text = str(search_text).strip()

    if search_text == "":
        return pd.DataFrame()

    translated_text = translate_search_text(
        search_text
    )

    candidates = activity_catalog[
        activity_catalog["Business Family"]
        == business_family
    ].copy()

    if candidates.empty:
        return pd.DataFrame()

    documents = (
        candidates["Search Text"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    document_vectors = semantic_model.encode(
        documents,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    arabic_vector = semantic_model.encode(
        search_text,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    english_vector = semantic_model.encode(
        translated_text,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    arabic_semantic = util.cos_sim(
        arabic_vector,
        document_vectors
    )[0].cpu().numpy()

    english_semantic = util.cos_sim(
        english_vector,
        document_vectors
    )[0].cpu().numpy()

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        lowercase=True
    )

    char_matrix = char_vectorizer.fit_transform(
        [translated_text] + documents
    )

    char_scores = cosine_similarity(
        char_matrix[0:1],
        char_matrix[1:]
    )[0]

    candidates["Arabic Semantic"] = arabic_semantic
    candidates["English Semantic"] = english_semantic
    candidates["Text Match"] = char_scores

    # Prototype hybrid weights; not empirically calibrated.
    candidates["Final Search Score"] = (
        0.40 * candidates["Arabic Semantic"]
        + 0.30 * candidates["English Semantic"]
        + 0.30 * candidates["Text Match"]
    )

    return (
        candidates
        .sort_values(
            "Final Search Score",
            ascending=False
        )
        .head(top_k)[
            [
                "ISIC Code",
                "Economic Activity",
                "Business Family",
                "Final Search Score"
            ]
        ]
        .reset_index(drop=True)
    )


def match_demand_category(activity_name):
    query_vector = semantic_model.encode(
        str(activity_name),
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    scores = util.cos_sim(
        query_vector,
        demand_vectors
    )[0]

    best_index = int(scores.argmax())

    return {
        "Demand Category":
            demand_categories[best_index],
        "Similarity":
            float(scores[best_index])
    }


available_families = (
    activity_catalog["Business Family"]
    .dropna()
    .unique()
    .tolist()
)

sector_options = [
    (
        business_family_labels.get(
            family,
            family
        ),
        family
    )
    for family in available_families
]

sector_dropdown = widgets.Dropdown(
    options=sector_options,
    description="القطاع:",
    style={"description_width": "initial"}
)

activity_search_box = widgets.Text(
    placeholder="مثال: مطعم، صيدلية، تصنيع ملابس...",
    description="ابحث عن النشاط:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px")
)

search_button = widgets.Button(
    description="بحث 🔎",
    button_style="info"
)

smart_activity_dropdown = widgets.Dropdown(
    options=[],
    description="النشاط:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px")
)

approve_activity_button = widgets.Button(
    description="اعتماد النشاط ✅",
    button_style="success"
)

search_output = widgets.Output()
approval_output = widgets.Output()

selected_project = {}
new_project_profile = {}


def run_activity_search(button=None):
    with search_output:
        clear_output()

        search_text = (
            activity_search_box.value.strip()
        )

        if not search_text:
            print("⚠️ اكتبي اسم النشاط أولًا.")
            return

        results = search_activities_in_family(
            sector_dropdown.value,
            search_text,
            top_k=5
        )

        if results.empty:
            print("لم يتم العثور على نشاط مناسب.")
            smart_activity_dropdown.options = []
            return

        print("🔎 أفضل الأنشطة المقترحة:")
        display(
            results[
                [
                    "ISIC Code",
                    "Economic Activity",
                    "Final Search Score"
                ]
            ].round(3)
        )

        options = []

        for _, row in results.iterrows():
            label = (
                f"{row['ISIC Code']} - "
                f"{row['Economic Activity']} "
                f"(Score: {row['Final Search Score']:.3f})"
            )

            options.append(
                (label, row["ISIC Code"])
            )

        smart_activity_dropdown.options = options

        if options:
            smart_activity_dropdown.value = (
                options[0][1]
            )

        display(smart_activity_dropdown)


def approve_selected_activity(button=None):
    with approval_output:
        clear_output()

        selected_isic = (
            smart_activity_dropdown.value
        )

        if selected_isic is None:
            print("⚠️ ابحثي واختاري نشاط أولًا.")
            return

        row = activity_catalog[
            activity_catalog["ISIC Code"]
            == selected_isic
        ].iloc[0]

        selected_project.clear()

        selected_project.update({
            "ISIC Code":
                selected_isic,
            "Economic Activity":
                row["Economic Activity"],
            "Business Family":
                row["Business Family"],
            "Sector Arabic":
                business_family_labels.get(
                    row["Business Family"],
                    row["Business Family"]
                )
        })

        print("✅ تم اعتماد النشاط")
        print("ISIC:", selected_project["ISIC Code"])
        print("Activity:", selected_project["Economic Activity"])
        print("Sector:", selected_project["Sector Arabic"])


search_button.on_click(run_activity_search)
approve_activity_button.on_click(
    approve_selected_activity
)


new_preferred_governorate = widgets.Dropdown(
    options=list(governorates["Governorate"]),
    description="المحافظة المفضلة:",
    style={"description_width": "initial"}
)

new_can_relocate = widgets.ToggleButtons(
    options=[
        ("نعم", True),
        ("لا", False)
    ],
    description="يمكن تغيير الموقع؟",
    style={"description_width": "initial"}
)

new_operating_model = widgets.Dropdown(
    options=[
        ("من المنزل", "home"),
        ("أونلاين", "online"),
        ("من المنزل + أونلاين", "home_online"),
        ("محل/موقع صغير", "small_physical"),
        ("موقع تجاري كامل", "full_physical"),
        ("موقع إنتاج/مصنع", "production")
    ],
    description="طريقة التشغيل:",
    style={"description_width": "initial"}
)

new_available_capital = widgets.FloatText(
    value=0,
    description="رأس المال المتوفر - اختياري (JD):",
    style={"description_width": "initial"}
)

save_new_project_button = widgets.Button(
    description="حفظ بيانات المشروع ✅",
    button_style="success"
)

new_project_output = widgets.Output()


def save_new_project_profile():
    if not selected_project:
        raise ValueError(
            "اختاري النشاط الاقتصادي أولًا."
        )

    available_capital = float(
        new_available_capital.value
    )

    if available_capital <= 0:
        available_capital = None

    new_project_profile.clear()

    new_project_profile.update({
        "ISIC Code":
            selected_project["ISIC Code"],
        "Economic Activity":
            selected_project["Economic Activity"],
        "Business Family":
            selected_project["Business Family"],
        "Project Status":
            "new",
        "Preferred Governorate":
            new_preferred_governorate.value,
        "Can Relocate":
            new_can_relocate.value,
        "Operating Model":
            new_operating_model.value,
        "Available Capital":
            available_capital
    })


def run_save_new_project(button=None):
    with new_project_output:
        clear_output()

        try:
            save_new_project_profile()
            print("✅ تم حفظ بيانات المشروع الجديد")

            for key, value in new_project_profile.items():
                if (
                    key == "Available Capital"
                    and value is None
                ):
                    print(
                        "Available Capital: "
                        "لم يتم تحديده"
                    )
                else:
                    print(f"{key}: {value}")

            if (
                "open_new_business_section" in globals()
                and globals().get("current_main_section") == "new"
            ):
                open_new_business_section()

        except Exception as e:
            print("⚠️", e)


save_new_project_button.on_click(
    run_save_new_project
)


In [ ]:
# ============================================
# 6) DYNAMIC CAPITAL PLANNER
# ============================================

capital_major_asset = widgets.FloatText(
    value=0,
    description="شراء أصل رئيسي:",
    style={"description_width": "190px"}
)

capital_equipment = widgets.FloatText(
    value=0,
    description="معدات وأجهزة:",
    style={"description_width": "190px"}
)

capital_inventory = widgets.FloatText(
    value=0,
    description="مخزون أولي:",
    style={"description_width": "190px"}
)

capital_premises = widgets.FloatText(
    value=0,
    description="موقع / إيجار / تأمين:",
    style={"description_width": "190px"}
)

capital_setup = widgets.FloatText(
    value=0,
    description="تجهيز وديكور:",
    style={"description_width": "190px"}
)

capital_licensing = widgets.FloatText(
    value=0,
    description="ترخيص وقانوني:",
    style={"description_width": "190px"}
)

capital_packaging = widgets.FloatText(
    value=0,
    description="تغليف وتجهيز منتجات:",
    style={"description_width": "190px"}
)

capital_delivery_setup = widgets.FloatText(
    value=0,
    description="توصيل / لوجستيات أولية:",
    style={"description_width": "190px"}
)

capital_launch_marketing = widgets.FloatText(
    value=0,
    description="تسويق الإطلاق:",
    style={"description_width": "190px"}
)

capital_technology = widgets.FloatText(
    value=0,
    description="موقع / أنظمة / تقنية:",
    style={"description_width": "190px"}
)

capital_other_startup = widgets.FloatText(
    value=0,
    description="تكاليف تأسيس أخرى:",
    style={"description_width": "190px"}
)

monthly_salaries_general = widgets.FloatText(
    value=0,
    description="رواتب شهرية:",
    style={"description_width": "190px"}
)

monthly_rent_general = widgets.FloatText(
    value=0,
    description="إيجار شهري:",
    style={"description_width": "190px"}
)

monthly_utilities_general = widgets.FloatText(
    value=0,
    description="خدمات وفواتير:",
    style={"description_width": "190px"}
)

monthly_marketing_general = widgets.FloatText(
    value=0,
    description="تسويق شهري:",
    style={"description_width": "190px"}
)

monthly_logistics_general = widgets.FloatText(
    value=0,
    description="نقل / توصيل شهري:",
    style={"description_width": "190px"}
)

monthly_other_general = widgets.FloatText(
    value=0,
    description="مصاريف شهرية أخرى:",
    style={"description_width": "190px"}
)

working_capital_months_general = widgets.IntText(
    value=3,
    description="أشهر التشغيل:",
    style={"description_width": "190px"}
)

reserve_percentage_general = widgets.FloatText(
    value=10,
    description="احتياطي %:",
    style={"description_width": "190px"}
)

show_all_capital_fields = widgets.Checkbox(
    value=False,
    description="عرض جميع بنود رأس المال"
)

startup_capital_box = widgets.VBox()
monthly_capital_box = widgets.VBox()


def update_capital_input_fields(change=None):
    family = new_project_profile.get(
        "Business Family",
        "other"
    )

    model = new_project_profile.get(
        "Operating Model",
        "unknown"
    )

    if family == "real_estate":
        capital_major_asset.description = (
            "شراء عقار / أصل:"
        )
    elif family == "transport_logistics":
        capital_major_asset.description = (
            "مركبة / أصل رئيسي:"
        )
    else:
        capital_major_asset.description = (
            "شراء أصل رئيسي:"
        )

    startup_fields = [
        capital_licensing,
        capital_launch_marketing,
        capital_technology,
        capital_other_startup
    ]

    monthly_fields = [
        monthly_marketing_general,
        monthly_other_general
    ]

    if model in [
        "small_physical",
        "full_physical"
    ]:
        startup_fields += [
            capital_equipment,
            capital_premises,
            capital_setup
        ]

        monthly_fields += [
            monthly_rent_general,
            monthly_utilities_general,
            monthly_salaries_general
        ]

    elif model == "production":
        startup_fields += [
            capital_major_asset,
            capital_equipment,
            capital_premises,
            capital_setup,
            capital_inventory
        ]

        monthly_fields += [
            monthly_rent_general,
            monthly_utilities_general,
            monthly_salaries_general
        ]

    elif model == "home":
        startup_fields += [
            capital_equipment
        ]

        monthly_fields += [
            monthly_utilities_general
        ]

    elif model in [
        "online",
        "home_online"
    ]:
        startup_fields += [
            capital_equipment,
            capital_technology
        ]

    if family in [
        "retail_trade",
        "manufacturing",
        "hospitality_food",
        "agriculture"
    ]:
        startup_fields += [
            capital_inventory,
            capital_packaging,
            capital_delivery_setup
        ]

        monthly_fields += [
            monthly_logistics_general
        ]

    if family == "real_estate":
        startup_fields += [
            capital_major_asset
        ]

    if family == "transport_logistics":
        startup_fields += [
            capital_major_asset,
            capital_delivery_setup
        ]

        monthly_fields += [
            monthly_logistics_general
        ]

    if family in [
        "professional_services",
        "information_technology",
        "business_support"
    ]:
        startup_fields += [
            capital_equipment,
            capital_technology
        ]

    if show_all_capital_fields.value:
        startup_fields = [
            capital_major_asset,
            capital_equipment,
            capital_inventory,
            capital_premises,
            capital_setup,
            capital_licensing,
            capital_packaging,
            capital_delivery_setup,
            capital_launch_marketing,
            capital_technology,
            capital_other_startup
        ]

        monthly_fields = [
            monthly_salaries_general,
            monthly_rent_general,
            monthly_utilities_general,
            monthly_marketing_general,
            monthly_logistics_general,
            monthly_other_general
        ]

    startup_capital_box.children = tuple(
        dict.fromkeys(startup_fields)
    )

    monthly_capital_box.children = tuple(
        dict.fromkeys(monthly_fields)
    )


show_all_capital_fields.observe(
    update_capital_input_fields,
    names="value"
)


def calculate_capital_phases():
    startup_costs = {
        "Major Asset":
            float(capital_major_asset.value),
        "Equipment":
            float(capital_equipment.value),
        "Initial Inventory":
            float(capital_inventory.value),
        "Premises / Deposit":
            float(capital_premises.value),
        "Setup / Decoration":
            float(capital_setup.value),
        "Licensing / Legal":
            float(capital_licensing.value),
        "Packaging":
            float(capital_packaging.value),
        "Delivery Setup":
            float(capital_delivery_setup.value),
        "Launch Marketing":
            float(capital_launch_marketing.value),
        "Technology":
            float(capital_technology.value),
        "Other Startup":
            float(capital_other_startup.value)
    }

    phase1_launch = sum(
        startup_costs.values()
    )

    monthly_costs = {
        "Salaries":
            float(monthly_salaries_general.value),
        "Rent":
            float(monthly_rent_general.value),
        "Utilities":
            float(monthly_utilities_general.value),
        "Marketing":
            float(monthly_marketing_general.value),
        "Logistics":
            float(monthly_logistics_general.value),
        "Other Monthly":
            float(monthly_other_general.value)
    }

    monthly_operating_cost = sum(
        monthly_costs.values()
    )

    months = max(
        0,
        int(working_capital_months_general.value)
    )

    phase2_operating = (
        monthly_operating_cost
        * months
    )

    phase2_cumulative = (
        phase1_launch
        + phase2_operating
    )

    reserve_percentage = max(
        0,
        float(reserve_percentage_general.value)
    )

    reserve_amount = (
        phase2_cumulative
        * reserve_percentage
        / 100
    )

    total_required = (
        phase2_cumulative
        + reserve_amount
    )

    result = {
        "Phase 1 - Launch":
            phase1_launch,
        "Monthly Operating Cost":
            monthly_operating_cost,
        "Phase 2 - Operating Capital":
            phase2_operating,
        "Phase 2 - Cumulative":
            phase2_cumulative,
        "Phase 3 - Reserve":
            reserve_amount,
        "Total Estimated Capital":
            total_required
    }

    available = new_project_profile.get(
        "Available Capital"
    )

    if (
        available is not None
        and float(available) > 0
    ):
        result["Available Capital"] = (
            float(available)
        )

        result["Capital Difference"] = (
            float(available)
            - total_required
        )

    return result


capital_phases = {}
capital_estimate_valid = False

capital_calculate_button = widgets.Button(
    description="احسب رأس المال 💰",
    button_style="success",
    layout=widgets.Layout(
        width="220px",
        height="42px"
    )
)

capital_calculation_output = widgets.Output()


def run_universal_capital_calculation(
    button=None
):
    global capital_phases
    global capital_estimate_valid

    with capital_calculation_output:
        clear_output()

        capital_phases = (
            calculate_capital_phases()
        )

        total = capital_phases[
            "Total Estimated Capital"
        ]

        if total <= 0:
            capital_estimate_valid = False

            print(
                "⚠️ لم يتم إدخال تكاليف كافية "
                "لحساب رأس المال المطلوب."
            )
            return

        capital_estimate_valid = True

        print("💰 خطة رأس المال")
        print("=" * 45)
        print(
            "Phase 1 - Launch:",
            f"{capital_phases['Phase 1 - Launch']:.2f} JD"
        )
        print(
            "Monthly Operating Cost:",
            f"{capital_phases['Monthly Operating Cost']:.2f} JD"
        )
        print(
            "Operating Capital:",
            f"{capital_phases['Phase 2 - Operating Capital']:.2f} JD"
        )
        print(
            "Reserve:",
            f"{capital_phases['Phase 3 - Reserve']:.2f} JD"
        )
        print(
            "Total Estimated Capital:",
            f"{capital_phases['Total Estimated Capital']:.2f} JD"
        )

        if "Available Capital" in capital_phases:
            difference = capital_phases[
                "Capital Difference"
            ]

            print(
                "Available Capital:",
                f"{capital_phases['Available Capital']:.2f} JD"
            )

            if difference >= 0:
                print(
                    "✅ فائض تقريبي:",
                    f"{difference:.2f} JD"
                )
            else:
                print(
                    "⚠️ فجوة تمويل تقريبية:",
                    f"{abs(difference):.2f} JD"
                )

    if (
        "open_new_business_section" in globals()
        and globals().get("current_main_section") == "new"
    ):
        open_new_business_section()


capital_calculate_button.on_click(
    run_universal_capital_calculation
)


In [ ]:
# ============================================
# 7) NEW-BUSINESS ANALYSIS ENGINE
# ============================================

location_strategy = {
    "manufacturing": "cluster",
    "construction": "cluster",
    "transport_logistics": "cluster",

    # Similar-business density may indicate either market activity or competition.
    "real_estate": "ambiguous_market",

    "retail_trade": "competition",
    "hospitality_food": "competition",
    "personal_services": "competition",
    "professional_services": "competition",
    "business_support": "competition",
    "education": "competition",
    "entertainment": "competition",
    "information_technology": "competition",
    "financial_services": "competition",
    "health_social": "competition",

    "agriculture": "resource_based",
    "mining": "resource_based",
    "utilities_environment": "resource_based",

    "other": "competition"
}

current_demand_category = None


def universal_location_model(
    isic_code,
    business_family,
    demand_category=None
):

    isic_code = str(isic_code)

    business = activities[
        activities["ISIC Code"] == isic_code
    ]

    if business.empty:
        raise ValueError(
            "لم يتم العثور على النشاط في بيانات المنشآت."
        )

    business = business.iloc[0]

    strategy = location_strategy.get(
        business_family,
        "competition"
    )

    demand_row = None

    if demand_category is not None:

        demand_match_row = demand[
            demand["Category (EN)"]
            == demand_category
        ]

        if not demand_match_row.empty:
            demand_row = demand_match_row.iloc[0]


    rows = []

    for arabic_name, english_name in governorate_map.items():

        gov_row = governorates[
            governorates["Governorate"]
            == english_name
        ].iloc[0]

        population = float(
            gov_row["Population 2025"]
        )

        registration_growth = float(
            gov_row["H1 Growth 2023→2024"]
        )

        similar_businesses = business[
            english_name
        ]

        if pd.isna(similar_businesses):
            similar_businesses = 0

        similar_businesses = float(
            similar_businesses
        )

        businesses_per_10k = (
            similar_businesses
            / population
            * 10000
        )

        if demand_row is not None:
            historical_demand = float(
                demand_row[arabic_name]
            )
        else:
            historical_demand = np.nan


        rows.append({
            "Governorate": english_name,
            "Population": population,
            "Historical Demand": historical_demand,
            "Similar Businesses": similar_businesses,
            "Businesses per 10k": businesses_per_10k,
            "Registration Growth": registration_growth
        })


    result = pd.DataFrame(rows)


    # 1) Market size
    result["Log Population"] = np.log1p(
        result["Population"]
    )

    result["Market Score"] = normalize(
        result["Log Population"]
    )


    # 2) Historical demand
    if result["Historical Demand"].notna().any():

        result["Demand Score"] = normalize(
            result["Historical Demand"]
        )

    else:

        # Missing evidence stays missing; it is not replaced by an invented 50.
        result["Demand Score"] = np.nan


    # 3) Similar-business activity indicator
    result["Sector Activity Score"] = normalize(
        result["Businesses per 10k"]
    )

    if strategy == "cluster":

        result["Sector Score"] = (
            result["Sector Activity Score"]
        )

    elif strategy == "competition":

        result["Sector Score"] = normalize(
            result["Businesses per 10k"],
            reverse=True
        )

    else:

        # ambiguous_market and resource_based:
        # keep the activity indicator for context,
        # but do not force an assumed positive/negative direction.
        result["Sector Score"] = np.nan


    # 4) Registration trend
    result["Trend Score"] = normalize(
        result["Registration Growth"]
    )


    # 5) Dynamic final score using only supported components
    score_components = [
        "Market Score",
        "Trend Score"
    ]

    if result["Demand Score"].notna().any():
        score_components.append(
            "Demand Score"
        )

    if (
        strategy in ["cluster", "competition"]
        and result["Sector Score"].notna().any()
    ):
        score_components.append(
            "Sector Score"
        )


    result[
        "Location Suitability Score"
    ] = result[
        score_components
    ].mean(axis=1)


    result = result.sort_values(
        "Location Suitability Score",
        ascending=False
    ).reset_index(drop=True)


    result["Rank"] = (
        result.index + 1
    )

    result["Strategy"] = strategy

    result["Components Used"] = (
        ", ".join(score_components)
    )

    return result


# ============================================
# DYNAMIC & DEFENSIBLE LOCATION SCORING
# ============================================

def apply_dynamic_location_scoring(
    location_df,
    business_family,
    demand_category=None
):

    result = location_df.copy()

    strategy = location_strategy.get(
        business_family,
        "competition"
    )

    score_components = [
        "Market Score",
        "Trend Score"
    ]


    # ----------------------------------------
    # Demand
    # ----------------------------------------

    if (
        demand_category is not None
        and "Demand Score" in result.columns
        and result["Demand Score"].notna().any()
    ):

        score_components.append(
            "Demand Score"
        )


    # ----------------------------------------
    # Sector / Competition Indicator
    # ----------------------------------------

    if strategy in [
        "competition",
        "cluster"
    ]:

        if (
            "Sector Score" in result.columns
            and result["Sector Score"].notna().any()
        ):

            score_components.append(
                "Sector Score"
            )


    # ambiguous_market:
    # نعرض بيانات نشاط القطاع،
    # لكن لا ندخلها في النتيجة النهائية
    # لأن اتجاه تأثيرها غير محسوم.

    # resource_based:
    # لا نستخدم Sector Score العام
    # حتى تتوفر بيانات موارد متخصصة.


    # ----------------------------------------
    # Final Score
    # ----------------------------------------

    result[
        "Location Suitability Score"
    ] = (
        result[
            score_components
        ].mean(axis=1)
    )


    result = result.sort_values(
        "Location Suitability Score",
        ascending=False
    ).reset_index(drop=True)


    result["Rank"] = (
        result.index + 1
    )


    result["Components Used"] = (
        ", ".join(score_components)
    )


    return result

# ============================================
# IMPROVED LOCATION EXPLANATION
# ============================================

def explain_selected_location(
    location_df,
    selected_governorate,
    demand_category=None
):

    selected_rows = location_df[
        location_df["Governorate"]
        == selected_governorate
    ]

    if selected_rows.empty:
        raise ValueError(
            "المحافظة المختارة غير موجودة في النتائج."
        )

    selected = selected_rows.iloc[0]
    top_location = location_df.iloc[0]

    strengths = []
    weaknesses = []
    notes = []

    family = new_project_profile.get(
        "Business Family",
        "other"
    )

    strategy = location_strategy.get(
        family,
        "competition"
    )

    components_text = str(
        selected.get(
            "Components Used",
            ""
        )
    )

    # --------------------------------
    # Market
    # --------------------------------

    market_median = location_df[
        "Market Score"
    ].median()

    if selected["Market Score"] > market_median:

        strengths.append(
            "حجم السوق النسبي أعلى من وسيط المحافظات."
        )

    elif selected["Market Score"] < market_median:

        weaknesses.append(
            "حجم السوق النسبي أقل من وسيط المحافظات."
        )


    # --------------------------------
    # Historical Demand Proxy
    # --------------------------------

    if (
        demand_category is not None
        and "Demand Score" in location_df.columns
        and "Demand Score" in components_text
    ):

        demand_median = location_df[
            "Demand Score"
        ].median()

        if selected["Demand Score"] > demand_median:

            strengths.append(
                "مؤشر الإنفاق الأسري التاريخي المرتبط "
                "بالنشاط (HEIS 2017) أعلى من وسيط المحافظات."
            )

        elif selected["Demand Score"] < demand_median:

            weaknesses.append(
                "مؤشر الإنفاق الأسري التاريخي المرتبط "
                "بالنشاط (HEIS 2017) أقل من وسيط المحافظات."
            )

        notes.append(
            "مؤشر الطلب المستخدم تاريخي من HEIS 2017 "
            "ولا يمثل الطلب الحالي بشكل مباشر."
        )

    else:

        notes.append(
            "لم يتم استخدام مؤشر طلب أسري مباشر "
            "لهذا النشاط لعدم توفر Proxy موثوق."
        )


    # --------------------------------
    # Sector / Competition
    # --------------------------------

    if (
        "Sector Score" in location_df.columns
        and "Sector Score" in components_text
    ):

        sector_median = location_df[
            "Sector Score"
        ].median()

        if strategy == "competition":

            if selected["Sector Score"] > sector_median:

                strengths.append(
                    "مؤشر الفرصة التنافسية المبني على "
                    "كثافة الأنشطة المشابهة أفضل نسبيًا "
                    "من وسيط المحافظات."
                )

            elif selected["Sector Score"] < sector_median:

                weaknesses.append(
                    "كثافة الأنشطة المشابهة تشير إلى "
                    "ضغط تنافسي أعلى نسبيًا."
                )

        elif strategy == "cluster":

            if selected["Sector Score"] > sector_median:

                strengths.append(
                    "مؤشر تجمع الأنشطة المشابهة "
                    "أعلى من وسيط المحافظات."
                )

            elif selected["Sector Score"] < sector_median:

                weaknesses.append(
                    "مؤشر تجمع الأنشطة المشابهة "
                    "أقل من وسيط المحافظات."
                )


    # --------------------------------
    # Registration Trend
    # --------------------------------

    trend_median = location_df[
        "Trend Score"
    ].median()

    if selected["Trend Score"] > trend_median:

        strengths.append(
            "اتجاه تسجيل الأعمال أقوى نسبيًا "
            "من وسيط المحافظات."
        )

    elif selected["Trend Score"] < trend_median:

        weaknesses.append(
            "اتجاه تسجيل الأعمال أضعف نسبيًا "
            "من وسيط المحافظات."
        )


    # --------------------------------
    # Ranking
    # --------------------------------

    if selected["Rank"] == 1:

        notes.append(
            "المحافظة المختارة حصلت على أعلى "
            "نتيجة Screening في النموذج الحالي."
        )

    else:

        gap = (
            top_location[
                "Location Suitability Score"
            ]
            - selected[
                "Location Suitability Score"
            ]
        )

        notes.append(
            f"أعلى نتيجة Screening كانت في "
            f"{top_location['Governorate']} "
            f"بفارق {gap:.2f} نقطة."
        )


    return {

        "Governorate":
            selected_governorate,

        "Rank":
            int(selected["Rank"]),

        "Score":
            round(
                selected[
                    "Location Suitability Score"
                ],
                2
            ),

        "Strengths":
            strengths,

        "Weaknesses":
            weaknesses,

        "Notes":
            notes,

        "Components Used":
            components_text
    }


def generate_target_customer_profile(
    project_profile
):

    family = project_profile.get(
        "Business Family",
        "other"
    )

    operating_model = project_profile.get(
        "Operating Model",
        "unknown"
    )

    activity = project_profile.get(
        "Economic Activity",
        ""
    )

    isic_code = str(
        project_profile.get(
            "ISIC Code",
            ""
        )
    )

    segments = []
    needs = []
    channels = []
    notes = []


    # ============================
    # Customer Segments
    # ============================

    if family == "retail_trade":

        segments = [
            "مستهلكون أفراد يبحثون عن المنتج للاستخدام الشخصي.",
            "عملاء محليون ضمن منطقة المشروع.",
            "عملاء يقارنون بين السعر والجودة والتوفر."
        ]

        needs = [
            "سعر مناسب",
            "توفر المنتج",
            "سهولة الشراء",
            "جودة موثوقة"
        ]


    elif family == "hospitality_food":

        segments = [
            "عملاء أفراد يبحثون عن الطعام أو خدمات الضيافة.",
            "عملاء محليون ومتكررون.",
            "عملاء التوصيل والطلبات الرقمية."
        ]

        needs = [
            "الجودة",
            "السعر",
            "سرعة الخدمة",
            "النظافة",
            "التوصيل"
        ]


    elif family == "manufacturing":

        segments = [
            "تجار ومحلات قد تشترى بالجملة.",
            "شركات تحتاج المنتجات ضمن سلسلة التوريد.",
            "عملاء أفراد إذا كان المشروع يبيع مباشرة."
        ]

        needs = [
            "ثبات الجودة",
            "السعر",
            "القدرة على التوريد",
            "الالتزام بالمواعيد"
        ]


    elif family == "real_estate":

        if isic_code == "6810":

            segments = [
                "أفراد يبحثون عن شراء أو استئجار عقار.",
                "أسر تبحث عن سكن مناسب.",
                "مستثمرون يبحثون عن عقارات للاستثمار."
            ]

            needs = [
                "السعر",
                "الموقع",
                "حالة العقار",
                "وضوح المعلومات",
                "الثقة والشفافية"
            ]


        elif isic_code == "6820":

            segments = [
                "أفراد يبحثون عن شراء أو استئجار عقار.",
                "أصحاب عقارات يرغبون ببيعها أو تأجيرها.",
                "مستثمرون يبحثون عن فرص عقارية."
            ]

            needs = [
                "الثقة",
                "سهولة الوصول للعروض",
                "وضوح العمولة",
                "معلومات دقيقة عن العقار",
                "سرعة التواصل"
            ]


        else:

            segments = [
                "أفراد يحتاجون خدمات عقارية.",
                "مستثمرون في القطاع العقاري."
            ]

            needs = [
                "الثقة",
                "السعر",
                "الموقع",
                "وضوح المعلومات"
            ]


    elif family == "information_technology":

        segments = [
            "مشاريع صغيرة ومتوسطة تحتاج حلولًا تقنية.",
            "شركات تبحث عن تطوير أو دعم أنظمة رقمية.",
            "عملاء أفراد حسب طبيعة الخدمة."
        ]

        needs = [
            "حل واضح للمشكلة",
            "سهولة الاستخدام",
            "الدعم الفني",
            "السعر"
        ]


    elif family == "professional_services":

        segments = [
            "أفراد يحتاجون خدمات متخصصة.",
            "شركات ومؤسسات تحتاج خدمات مهنية.",
            "مشاريع صغيرة تحتاج خبرات خارجية."
        ]

        needs = [
            "الثقة",
            "الخبرة",
            "وضوح الخدمة",
            "سرعة الاستجابة"
        ]


    elif family == "education":

        segments = [
            "طلاب أو متعلمون يحتاجون الخدمة التعليمية.",
            "أولياء أمور عند ارتباط الخدمة بالتعليم المدرسي.",
            "مؤسسات تحتاج تدريبًا أو تطوير مهارات."
        ]

        needs = [
            "جودة التعليم",
            "النتائج",
            "المرونة",
            "السعر"
        ]


    elif family == "health_social":

        segments = [
            "أفراد يحتاجون الخدمة الصحية أو الاجتماعية.",
            "أسر تبحث عن خدمات مناسبة لأفرادها.",
            "جهات تحتاج خدمات متخصصة."
        ]

        needs = [
            "الثقة",
            "سهولة الوصول",
            "جودة الخدمة",
            "الاستجابة"
        ]


    else:

        segments = [
            "عملاء أفراد محتملون حسب طبيعة النشاط.",
            "شركات أو مؤسسات قد تستفيد من الخدمة.",
            "عملاء ضمن السوق المحلي للمشروع."
        ]

        needs = [
            "القيمة مقابل السعر",
            "الجودة",
            "الثقة",
            "سهولة الوصول"
        ]


    # ============================
    # Marketing Channels
    # ============================

    if operating_model in [
        "online",
        "home_online"
    ]:

        channels = [
            "وسائل التواصل الاجتماعي",
            "الإعلانات الرقمية",
            "التواصل المباشر عبر الإنترنت"
        ]

    elif operating_model == "home":

        channels = [
            "وسائل التواصل الاجتماعي",
            "التوصيات والإحالات",
            "التسويق المحلي"
        ]

    elif operating_model in [
        "small_physical",
        "full_physical"
    ]:

        channels = [
            "الموقع الفعلي",
            "Google Maps والمنصات المحلية",
            "وسائل التواصل الاجتماعي",
            "التسويق المحلي"
        ]

    elif operating_model == "production":

        channels = [
            "التواصل مع التجار والشركات",
            "المعارض والفعاليات التجارية",
            "التسويق الرقمي B2B",
            "المبيعات المباشرة"
        ]

    else:

        channels = [
            "التسويق الرقمي",
            "التسويق المحلي"
        ]


    notes.append(
        "هذا الجمهور المستهدف تقدير أولي مبني على "
        "نوع النشاط وطريقة التشغيل، ويحتاج إلى التحقق "
        "من خلال بيانات سوق وعملاء فعلية."
    )


    return {
        "Activity": activity,
        "Customer Segments": segments,
        "Customer Needs": needs,
        "Recommended Channels": channels,
        "Notes": notes
    }

def analyze_prelaunch_risks(
    project_profile,
    location_explanation,
    capital_phases=None,
    demand_category=None
):

    high_priority = []
    watch_points = []
    data_gaps = []
    positive_factors = []


    # =================================
    # 1. Capital Risk
    # =================================

    available = project_profile.get(
        "Available Capital"
    )

    required = None

    if capital_phases is not None:
        required = capital_phases.get(
            "Total Estimated Capital"
        )


    if (
        available is not None
        and required is not None
    ):

        gap = (
            float(available)
            - float(required)
        )

        if gap < 0:

            high_priority.append(
                f"رأس المال المتوفر أقل من "
                f"التقدير المطلوب بحوالي "
                f"{abs(gap):.2f} JD."
            )

        else:

            positive_factors.append(
                "رأس المال المتوفر يغطي "
                "التقدير الحالي لتكاليف الإطلاق."
            )

    elif available is None:

        data_gaps.append(
            "لم يتم تحديد رأس المال المتوفر، "
            "لذلك لا يمكن تقييم فجوة التمويل حاليًا."
        )


    # =================================
    # 2. Location Risk
    # =================================

    rank = int(
        location_explanation["Rank"]
    )

    if rank <= 3:

        positive_factors.append(
            f"الموقع المختار ضمن أعلى "
            f"{rank} نتائج في Screening الموقع."
        )

    elif rank >= 8:

        watch_points.append(
            f"الموقع المختار ترتيبه "
            f"{rank} من 12 في Screening الموقع."
        )

        if not project_profile.get(
            "Can Relocate",
            False
        ):

            high_priority.append(
                "الموقع الحالي منخفض نسبيًا في "
                "Screening والمستخدم لا يخطط "
                "لتغيير الموقع."
            )


    # =================================
    # 3. Demand Evidence
    # =================================

    if demand_category is None:

        data_gaps.append(
            "لا يوجد حاليًا مؤشر طلب مباشر "
            "موثوق لهذا النشاط، لذلك يجب دعم "
            "قرار الإطلاق ببيانات سوق إضافية."
        )


    # =================================
    # 4. Operating Model
    # =================================

    operating_model = project_profile.get(
        "Operating Model"
    )

    if operating_model == "home":

        watch_points.append(
            "نموذج التشغيل من المنزل يحتاج "
            "التحقق من ملاءمته لطبيعة النشاط "
            "ومتطلبات الترخيص قبل الإطلاق."
        )


    # =================================
    # 5. Sector interpretation
    # =================================

    family = project_profile.get(
        "Business Family"
    )

    strategy = location_strategy.get(
        family,
        "competition"
    )

    if strategy == "cluster":

        watch_points.append(
            "ارتفاع نشاط القطاع قد يدل على "
            "سوق نشط أو تجمع أعمال، لكنه قد "
            "يعني أيضًا منافسة أعلى؛ لذلك يجب "
            "عدم تفسيره كميزة مؤكدة وحده."
        )

    elif strategy == "resource_based":

        data_gaps.append(
            "هذا القطاع يحتاج بيانات موارد "
            "ومتغيرات متخصصة قبل إصدار تحليل "
            "موقع أكثر دقة."
        )


    return {
        "High Priority Risks":
            high_priority,

        "Watch Points":
            watch_points,

        "Data Gaps":
            data_gaps,

        "Positive Factors":
            positive_factors
    }

def generate_prelaunch_recommendations(
    project_profile,
    location_explanation,
    prelaunch_risks,
    target_customer_profile,
    capital_result=None
):

    priorities = []
    recommendations = []
    validation_steps = []


    # ==============================
    # 1. رأس المال
    # ==============================

    available = project_profile.get(
        "Available Capital"
    )

    required = None

    if capital_result is not None:

        required = capital_result.get(
            "Total Estimated Capital"
        )

        if required is None:

            required = capital_result.get(
                "Estimated Required Capital"
            )


    if (
        available is not None
        and required is not None
    ):

        gap = (
            float(available)
            - float(required)
        )

        if gap < 0:

            priorities.append(
                f"يوجد نقص في رأس المال يقدّر بحوالي "
                f"{abs(gap):.2f} JD. "
                "يُنصح بتقليل تكاليف البداية "
                "أو تقسيم الإنفاق إلى مراحل."
            )

        else:

            recommendations.append(
                "رأس المال المتوفر يغطي "
                "التقدير الحالي للمشروع."
            )


    # ==============================
    # 2. الموقع
    # ==============================

    rank = int(
        location_explanation["Rank"]
    )

    if rank == 1:

        recommendations.append(
            "الموقع المختار حصل على أعلى نتيجة "
            "في Screening الموقع الحالي."
        )

    elif rank <= 3:

        recommendations.append(
            f"الموقع المختار ضمن أعلى {rank} "
            "نتائج في Screening الموقع."
        )

    else:

        validation_steps.append(
            "مقارنة الموقع المختار مع المحافظات "
            "الأعلى ترتيبًا قبل اتخاذ القرار النهائي."
        )


    # ==============================
    # 3. المخاطر
    # ==============================

    for risk in prelaunch_risks[
        "High Priority Risks"
    ]:

        # منع تكرار تحذير رأس المال
        if (
            "رأس المال" in risk
            and any(
                "رأس المال" in item
                for item in priorities
            )
        ):
            continue

        if risk not in priorities:
            priorities.append(risk)


    for point in prelaunch_risks[
        "Watch Points"
    ]:

        if point not in validation_steps:
            validation_steps.append(point)


    for gap in prelaunch_risks[
        "Data Gaps"
    ]:

        if gap not in validation_steps:
            validation_steps.append(gap)


    # ==============================
    # 4. الجمهور المستهدف
    # ==============================

    if target_customer_profile[
        "Customer Segments"
    ]:

        recommendations.append(
            "اختبار الجمهور المستهدف مبدئيًا "
            "قبل الإطلاق للتأكد من وجود اهتمام فعلي."
        )


    if target_customer_profile[
        "Recommended Channels"
    ]:

        recommendations.append(
            "بدء تجربة تسويقية صغيرة باستخدام "
            "قنوات الوصول المقترحة قبل زيادة "
            "الإنفاق التسويقي."
        )


    # ==============================
    # 5. التحقق من السوق
    # ==============================

    validation_steps.append(
        "جمع بيانات فعلية من السوق قبل الالتزام "
        "الكامل بالمشروع، مثل استبيان أو مقابلات "
        "مع عملاء محتملين أو تجربة أولية صغيرة."
    )


    return {
        "Priority Actions":
            priorities,

        "Recommendations":
            recommendations,

        "Validation Steps":
            validation_steps
    }

def refresh_new_business_analysis():

    global capital_phases
    global capital_estimate_valid
    global prelaunch_risks
    global prelaunch_recommendations
    global target_customer_profile
    global new_location_result
    global location_explanation
    global current_demand_category


    if (
        "new_project_profile" not in globals()
        or not isinstance(new_project_profile, dict)
        or not new_project_profile.get("Business Family")
        or not new_project_profile.get("ISIC Code")
    ):
        raise ValueError(
            "يجب حفظ بيانات المشروع الجديد واختيار النشاط أولًا."
        )


    # =========================================
    # 1. Available Capital
    # =========================================

    available = new_project_profile.get(
        "Available Capital"
    )

    if (
        available is None
        or float(available) <= 0
    ):
        new_project_profile[
            "Available Capital"
        ] = None


    # =========================================
    # 2. Demand proxy selection
    # =========================================

    family = new_project_profile[
        "Business Family"
    ]

    activity_name = new_project_profile.get(
        "Economic Activity",
        ""
    )

    consumer_families = [
        "retail_trade",
        "hospitality_food",
        "personal_services",
        "education",
        "health_social",
        "entertainment"
    ]

    current_demand_category = None

    if (
        family in consumer_families
        and "match_demand_category" in globals()
    ):

        try:

            demand_match = (
                match_demand_category(
                    activity_name
                )
            )

            similarity = float(
                demand_match[
                    "Similarity"
                ]
            )

            # Prototype confidence threshold.
            if similarity >= 0.45:
                current_demand_category = (
                    demand_match[
                        "Demand Category"
                    ]
                )

        except Exception:
            current_demand_category = None


    # =========================================
    # 3. Location analysis
    # =========================================

    new_location_result = (
        universal_location_model(
            new_project_profile[
                "ISIC Code"
            ],
            family,
            current_demand_category
        )
    )


    if (
        "apply_dynamic_location_scoring"
        in globals()
    ):

        new_location_result = (
            apply_dynamic_location_scoring(
                new_location_result,
                family,
                current_demand_category
            )
        )


    location_explanation = (
        explain_selected_location(
            new_location_result,
            new_project_profile[
                "Preferred Governorate"
            ],
            current_demand_category
        )
    )


    # =========================================
    # 4. Target customer
    # =========================================

    target_customer_profile = (
        generate_target_customer_profile(
            new_project_profile
        )
    )


    # Keep capital fields aligned with the latest project profile.
    if (
        "update_capital_input_fields"
        in globals()
    ):
        update_capital_input_fields()


    # =========================================
    # 5. Capital
    # =========================================

    capital_phases = (
        calculate_capital_phases()
    )

    total_capital = capital_phases.get(
        "Total Estimated Capital",
        0
    )

    capital_estimate_valid = (
        total_capital > 0
    )


    if capital_estimate_valid:
        capital_for_analysis = (
            capital_phases
        )
    else:
        capital_for_analysis = None


    # =========================================
    # 6. Risks
    # =========================================

    prelaunch_risks = (
        analyze_prelaunch_risks(
            new_project_profile,
            location_explanation,
            capital_for_analysis,
            current_demand_category
        )
    )


    # =========================================
    # 7. Recommendations
    # =========================================

    prelaunch_recommendations = (
        generate_prelaunch_recommendations(
            new_project_profile,
            location_explanation,
            prelaunch_risks,
            target_customer_profile,
            capital_for_analysis
        )
    )


    validation_steps = (
        prelaunch_recommendations[
            "Validation Steps"
        ]
    )

    validation_steps[:] = [
        item
        for item in validation_steps
        if (
            "رأس المال المتوفر" not in item
            and "تكاليف كافية" not in item
        )
    ]


    if not capital_estimate_valid:

        validation_steps.insert(
            0,
            "لم يتم إدخال تكاليف كافية "
            "لحساب رأس المال المطلوب."
        )

    elif (
        new_project_profile[
            "Available Capital"
        ] is None
    ):

        validation_steps.insert(
            0,
            "تم تقدير رأس المال المطلوب، "
            "لكن لم يتم تحديد رأس المال المتوفر؛ "
            "لذلك لا يمكن تقييم فجوة التمويل حاليًا."
        )


In [ ]:
# ============================================
# 8) FREE / PREMIUM PRESENTATION
# ============================================

plan_selector = widgets.ToggleButtons(
    options=[
        ("🆓 Free", "free"),
        ("⭐ Premium", "premium")
    ],
    value="free",
    description="الخطة:",
    style={
        "description_width": "70px"
    }
)

plan_features = {
    "free": {
        "project_profile": True,
        "basic_capital": True,
        "basic_location": True,
        "location_explanation": False,
        "detailed_risks": False,
        "target_customer": False,
        "recommendations": False,
        "validation_steps": False
    },
    "premium": {
        "project_profile": True,
        "basic_capital": True,
        "basic_location": True,
        "location_explanation": True,
        "detailed_risks": True,
        "target_customer": True,
        "recommendations": True,
        "validation_steps": True
    }
}


def feature_available(feature):
    return plan_features[
        plan_selector.value
    ].get(
        feature,
        False
    )


def show_new_business_dashboard():

    # ==================================
    # Current Plan
    # ==================================

    current_plan = plan_selector.value

    if current_plan == "free":
        plan_name = "🆓 FREE PLAN"
    else:
        plan_name = "⭐ PREMIUM PLAN"


    # ==================================
    # Title
    # ==================================

    display(
        HTML(
            f"""
            <h2>🚀 New Business Decision Support</h2>
            <p>Pre-Launch Business Analysis Dashboard</p>
            <h4>{plan_name}</h4>
            """
        )
    )


    # ==================================
    # Project Profile
    # FREE + PREMIUM
    # ==================================

    print("🏢 PROJECT PROFILE")
    print("=" * 55)

    print(
        "Activity:",
        new_project_profile.get(
            "Economic Activity",
            "Not specified"
        )
    )

    print(
        "ISIC Code:",
        new_project_profile.get(
            "ISIC Code",
            "Not specified"
        )
    )

    print(
        "Business Family:",
        new_project_profile.get(
            "Business Family",
            "Not specified"
        )
    )

    print(
        "Preferred Governorate:",
        new_project_profile.get(
            "Preferred Governorate",
            "Not specified"
        )
    )

    print(
        "Operating Model:",
        new_project_profile.get(
            "Operating Model",
            "Not specified"
        )
    )


    available = new_project_profile.get(
        "Available Capital"
    )

    if available is None:

        print(
            "Available Capital: Not specified"
        )

    else:

        print(
            f"Available Capital: "
            f"{available:.2f} JD"
        )


    # ==================================
    # Capital Plan
    # FREE + PREMIUM
    # ==================================

    print("\n💰 CAPITAL PLAN")
    print("=" * 55)

    if (
        "capital_estimate_valid" not in globals()
        or not capital_estimate_valid
    ):

        print(
            "Capital Estimate: "
            "Not calculated yet"
        )

        print(
            "أدخل تكاليف المشروع ثم اضغط "
            "احسب رأس المال 💰."
        )

    else:

        print(
            f"Phase 1 - Launch: "
            f"{capital_phases['Phase 1 - Launch']:.2f} JD"
        )

        print(
            f"Monthly Operating Cost: "
            f"{capital_phases['Monthly Operating Cost']:.2f} JD"
        )

        print(
            f"Total Estimated Capital: "
            f"{capital_phases['Total Estimated Capital']:.2f} JD"
        )


        # Premium يعطي تفاصيل إضافية
        if current_plan == "premium":

            print(
                f"Phase 2 - Operating Capital: "
                f"{capital_phases['Phase 2 - Operating Capital']:.2f} JD"
            )

            print(
                f"Phase 3 - Reserve: "
                f"{capital_phases['Phase 3 - Reserve']:.2f} JD"
            )


        if "Available Capital" in capital_phases:

            difference = capital_phases[
                "Capital Difference"
            ]

            print(
                f"Available Capital: "
                f"{capital_phases['Available Capital']:.2f} JD"
            )

            if difference >= 0:

                print(
                    f"Capital Surplus: "
                    f"{difference:.2f} JD"
                )

            else:

                print(
                    f"Capital Gap: "
                    f"{abs(difference):.2f} JD"
                )


    # ==================================
    # Basic Location
    # FREE + PREMIUM
    # ==================================

    print("\n📍 LOCATION ANALYSIS")
    print("=" * 55)

    print(
        "Selected Governorate:",
        location_explanation[
            "Governorate"
        ]
    )

    print(
        "Rank:",
        location_explanation[
            "Rank"
        ],
        "/ 12"
    )

    print(
        "Location Suitability Score:",
        round(
            location_explanation[
                "Score"
            ],
            2
        )
    )


    # ==================================
    # Premium Location Explanation
    # ==================================

    if feature_available(
        "location_explanation"
    ):

        print("\n✅ Strengths")

        if location_explanation[
            "Strengths"
        ]:

            for item in location_explanation[
                "Strengths"
            ]:

                print("•", item)

        else:

            print(
                "• لا توجد نقاط قوة واضحة "
                "ضمن المؤشرات الحالية."
            )


        print("\n⚠️ Areas to Review")

        if location_explanation[
            "Weaknesses"
        ]:

            for item in location_explanation[
                "Weaknesses"
            ]:

                print("•", item)

        else:

            print(
                "• لم تظهر نقاط ضعف واضحة "
                "ضمن المؤشرات الحالية."
            )

        notes = location_explanation.get(
            "Notes",
            []
        )

        if notes:
            print("\nℹ️ Location Notes")
            for item in notes:
                print("•", item)

    else:

        print(
            "\n🔒 تفسير نقاط قوة وضعف الموقع "
            "متاح في Premium."
        )


    # ==================================
    # Detailed Risks
    # PREMIUM
    # ==================================

    if feature_available(
        "detailed_risks"
    ):

        print("\n🚨 PRE-LAUNCH RISKS")
        print("=" * 55)

        if prelaunch_risks[
            "High Priority Risks"
        ]:

            for item in prelaunch_risks[
                "High Priority Risks"
            ]:

                print("•", item)

        else:

            print(
                "• لا توجد مخاطر عالية الأولوية "
                "ضمن البيانات الحالية."
            )

    else:

        print("\n🚨 PRE-LAUNCH RISKS")
        print("=" * 55)

        print(
            "🔒 تحليل المخاطر التفصيلي "
            "متاح في Premium."
        )


    # ==================================
    # Target Customer
    # PREMIUM
    # ==================================

    if feature_available(
        "target_customer"
    ):

        print("\n🎯 TARGET CUSTOMER")
        print("=" * 55)

        for item in target_customer_profile[
            "Customer Segments"
        ]:

            print("•", item)


        print("\n💡 Customer Needs")

        for item in target_customer_profile[
            "Customer Needs"
        ]:

            print("•", item)


        print("\n📣 Suggested Channels")

        for item in target_customer_profile[
            "Recommended Channels"
        ]:

            print("•", item)

    else:

        print("\n🎯 TARGET CUSTOMER")
        print("=" * 55)

        print(
            "🔒 تحليل الجمهور المستهدف "
            "وقنوات الوصول متاح في Premium."
        )


    # ==================================
    # Priority Actions + Recommendations
    # PREMIUM
    # ==================================

    if feature_available(
        "recommendations"
    ):

        print("\n🚨 PRIORITY ACTIONS")
        print("=" * 55)

        if prelaunch_recommendations[
            "Priority Actions"
        ]:

            for item in prelaunch_recommendations[
                "Priority Actions"
            ]:

                print("•", item)

        else:

            print(
                "• لا توجد إجراءات عاجلة "
                "ضمن التحليل الحالي."
            )


        print("\n💡 RECOMMENDATIONS")
        print("=" * 55)

        for item in prelaunch_recommendations[
            "Recommendations"
        ]:

            print("•", item)

    else:

        print("\n💡 RECOMMENDATIONS")
        print("=" * 55)

        print(
            "🔒 التوصيات المخصصة وخطة العمل "
            "متاحة في Premium."
        )


    # ==================================
    # Validation Steps
    # PREMIUM
    # ==================================

    if feature_available(
        "validation_steps"
    ):

        print("\n🔎 VALIDATION BEFORE LAUNCH")
        print("=" * 55)

        for item in prelaunch_recommendations[
            "Validation Steps"
        ]:

            print("•", item)


    # ==================================
    # Free Upgrade Message
    # ==================================

    if current_plan == "free":

        print("\n⭐ PREMIUM ANALYSIS")
        print("=" * 55)

        print(
            "افتح Premium للحصول على:"
        )

        print(
            "• تفسير نقاط القوة والضعف"
        )

        print(
            "• تحليل المخاطر بالتفصيل"
        )

        print(
            "• الجمهور المستهدف"
        )

        print(
            "• التوصيات وخطوات ما قبل الإطلاق"
        )


    # ==================================
    # Disclaimer
    # ==================================

    print(
        "\nℹ️ هذا التحليل أداة لدعم القرار، "
        "ولا يمثل ضمانًا لنجاح المشروع."
    )

In [ ]:
# ============================================
# 9) EXISTING-BUSINESS PERFORMANCE ENGINE
# ============================================

performance_history = []
performance_df = pd.DataFrame()


def calculate_business_kpis(snapshot):
    revenue = float(snapshot["Revenue"])
    expenses = float(snapshot["Expenses"])
    marketing = float(
        snapshot["Marketing Spend"]
    )
    customers = int(snapshot["Customers"])
    transactions = int(
        snapshot["Transactions"]
    )
    complaints = int(snapshot["Complaints"])

    profit = revenue - expenses

    profit_margin = (
        profit / revenue * 100
        if revenue > 0
        else 0
    )

    average_order_value = (
        revenue / transactions
        if transactions > 0
        else 0
    )

    complaint_rate = (
        complaints / customers * 100
        if customers > 0
        else 0
    )

    marketing_ratio = (
        marketing / revenue * 100
        if revenue > 0
        else 0
    )

    return {
        "Profit":
            round(profit, 2),
        "Profit Margin %":
            round(profit_margin, 2),
        "Average Order Value":
            round(average_order_value, 2),
        "Complaint Rate %":
            round(complaint_rate, 2),
        "Marketing / Revenue %":
            round(marketing_ratio, 2)
    }


def add_or_update_month(
    month_name,
    snapshot
):
    global performance_history
    global performance_df

    month_name = (
        str(month_name).strip()
        or "Month"
    )

    kpis = calculate_business_kpis(
        snapshot
    )

    month_record = {
        "Month":
            month_name,
        **snapshot,
        **kpis
    }

    performance_history = [
        row
        for row in performance_history
        if row["Month"] != month_name
    ]

    performance_history.append(
        month_record
    )

    performance_df = pd.DataFrame(
        performance_history
    )

    return month_record


def get_month_number(month_name):
    match = re.search(
        r"\d+",
        str(month_name)
    )

    if match:
        return int(match.group())

    return -1


def get_valid_performance_df():
    if (
        performance_df is None
        or performance_df.empty
    ):
        return pd.DataFrame()

    temp = performance_df.copy()

    required = [
        "Revenue",
        "Expenses",
        "Customers",
        "Transactions"
    ]

    for col in required:
        if col not in temp.columns:
            temp[col] = 0

    actual_mask = (
        temp[required]
        .fillna(0)
        .abs()
        .sum(axis=1)
        > 0
    )

    temp = temp[
        actual_mask
    ].copy()

    if temp.empty:
        return temp

    temp["Month Order"] = (
        temp["Month"]
        .apply(get_month_number)
    )

    return (
        temp
        .sort_values("Month Order")
        .reset_index(drop=True)
    )


def get_latest_two_months():
    temp = get_valid_performance_df()

    if len(temp) < 2:
        raise ValueError(
            "يجب توفر بيانات فعلية لشهرين على الأقل لإجراء المقارنة."
        )

    return (
        temp.iloc[-2],
        temp.iloc[-1]
    )


def safe_percent_change(
    current,
    previous
):
    try:
        current = float(current)
        previous = float(previous)
    except Exception:
        return None

    if previous == 0 and current == 0:
        return 0.0

    if previous == 0:
        return None

    return (
        (current - previous)
        / abs(previous)
        * 100
    )


def format_change(value):
    if value is None or pd.isna(value):
        return "N/A"

    return f"{value:+.1f}%"


def build_current_month_comparison(
    previous,
    latest
):
    indicators = [
        "Revenue",
        "Expenses",
        "Profit",
        "Profit Margin %",
        "Customers",
        "Transactions",
        "Average Order Value",
        "Complaints",
        "Complaint Rate %",
        "Marketing Spend",
        "Marketing / Revenue %",
        "Customer Rating"
    ]

    rows = []

    for indicator in indicators:
        old_value = float(
            previous[indicator]
        )
        new_value = float(
            latest[indicator]
        )

        difference = (
            new_value - old_value
        )

        change = safe_percent_change(
            new_value,
            old_value
        )

        rows.append({
            "Indicator":
                indicator,
            "Previous":
                old_value,
            "Current":
                new_value,
            "Difference":
                difference,
            "Change %":
                np.nan
                if change is None
                else change
        })

    return pd.DataFrame(rows)


def get_metric_change(
    comparison_df,
    indicator
):
    row = comparison_df[
        comparison_df["Indicator"]
        == indicator
    ]

    if row.empty:
        return None

    value = row.iloc[0]["Change %"]

    if pd.isna(value):
        return None

    return float(value)


def get_metric_difference(
    comparison_df,
    indicator
):
    row = comparison_df[
        comparison_df["Indicator"]
        == indicator
    ]

    if row.empty:
        return None

    value = row.iloc[0]["Difference"]

    if pd.isna(value):
        return None

    return float(value)


def analyze_business_changes(
    comparison_df
):
    warnings = []
    observations = []
    positives = []

    revenue = get_metric_change(
        comparison_df,
        "Revenue"
    )
    profit = get_metric_change(
        comparison_df,
        "Profit"
    )
    customers = get_metric_change(
        comparison_df,
        "Customers"
    )
    transactions = get_metric_change(
        comparison_df,
        "Transactions"
    )
    complaints = get_metric_change(
        comparison_df,
        "Complaint Rate %"
    )
    rating_diff = get_metric_difference(
        comparison_df,
        "Customer Rating"
    )
    margin_diff = get_metric_difference(
        comparison_df,
        "Profit Margin %"
    )
    expenses = get_metric_change(
        comparison_df,
        "Expenses"
    )
    average_order = get_metric_change(
        comparison_df,
        "Average Order Value"
    )

    # Prototype thresholds; adjustable.
    if revenue is not None:
        if revenue <= -20:
            warnings.append(
                f"الإيرادات انخفضت بنسبة "
                f"{abs(revenue):.1f}%، "
                "وهو تغير يحتاج مراجعة ذات أولوية."
            )
        elif revenue <= -10:
            observations.append(
                f"الإيرادات انخفضت بنسبة "
                f"{abs(revenue):.1f}%."
            )
        elif revenue >= 10:
            positives.append(
                f"الإيرادات ارتفعت بنسبة "
                f"{revenue:.1f}%."
            )

    if profit is not None:
        if profit <= -20:
            warnings.append(
                f"الربح انخفض بنسبة "
                f"{abs(profit):.1f}%، "
                "ويحتاج تحليل أسباب الانخفاض."
            )
        elif profit <= -10:
            observations.append(
                f"الربح انخفض بنسبة "
                f"{abs(profit):.1f}%."
            )
        elif profit >= 10:
            positives.append(
                f"الربح ارتفع بنسبة "
                f"{profit:.1f}%."
            )

    if margin_diff is not None:
        if margin_diff <= -5:
            warnings.append(
                f"هامش الربح انخفض بحوالي "
                f"{abs(margin_diff):.1f} نقطة مئوية."
            )
        elif margin_diff >= 3:
            positives.append(
                f"هامش الربح تحسن بحوالي "
                f"{margin_diff:.1f} نقطة مئوية."
            )

    if customers is not None:
        if customers <= -15:
            warnings.append(
                f"عدد العملاء انخفض بنسبة "
                f"{abs(customers):.1f}%."
            )
        elif customers <= -5:
            observations.append(
                f"عدد العملاء انخفض بنسبة "
                f"{abs(customers):.1f}%."
            )
        elif customers >= 10:
            positives.append(
                f"عدد العملاء ارتفع بنسبة "
                f"{customers:.1f}%."
            )

    if transactions is not None:
        if transactions <= -20:
            warnings.append(
                f"عدد عمليات البيع انخفض بنسبة "
                f"{abs(transactions):.1f}%."
            )
        elif transactions <= -10:
            observations.append(
                f"عدد عمليات البيع انخفض بنسبة "
                f"{abs(transactions):.1f}%."
            )
        elif transactions >= 10:
            positives.append(
                f"عدد عمليات البيع ارتفع بنسبة "
                f"{transactions:.1f}%."
            )

    if complaints is not None:
        if complaints >= 25:
            warnings.append(
                f"معدل الشكاوى ارتفع بنسبة "
                f"{complaints:.1f}%."
            )
        elif complaints >= 10:
            observations.append(
                f"معدل الشكاوى ارتفع بنسبة "
                f"{complaints:.1f}%."
            )
        elif complaints <= -20:
            positives.append(
                f"معدل الشكاوى انخفض بنسبة "
                f"{abs(complaints):.1f}%."
            )

    if rating_diff is not None:
        if rating_diff <= -0.5:
            warnings.append(
                f"تقييم العملاء انخفض بمقدار "
                f"{abs(rating_diff):.1f} نقطة."
            )
        elif rating_diff >= 0.3:
            positives.append(
                f"تقييم العملاء تحسن بمقدار "
                f"{rating_diff:.1f} نقطة."
            )

    if (
        transactions is not None
        and average_order is not None
        and transactions < 0
        and average_order > 0
    ):
        observations.append(
            "انخفض عدد عمليات البيع بينما ارتفع "
            "متوسط قيمة الطلب؛ قد يكون التغير ناتجًا "
            "عن عدد أقل من العمليات بقيم أعلى."
        )

    if (
        expenses is not None
        and revenue is not None
        and expenses > 10
        and revenue <= 0
    ):
        observations.append(
            "المصاريف ارتفعت بينما الإيرادات لم ترتفع، "
            "لذلك يُنصح بمراجعة بنود التكلفة."
        )

    return {
        "Warnings":
            warnings,
        "Observations":
            observations,
        "Positives":
            positives
    }


def generate_business_recommendations(
    comparison_df
):
    recommendations = []
    investigation = []

    revenue = get_metric_change(
        comparison_df,
        "Revenue"
    )
    profit = get_metric_change(
        comparison_df,
        "Profit"
    )
    transactions = get_metric_change(
        comparison_df,
        "Transactions"
    )
    customers = get_metric_change(
        comparison_df,
        "Customers"
    )
    complaints = get_metric_change(
        comparison_df,
        "Complaint Rate %"
    )
    margin_diff = get_metric_difference(
        comparison_df,
        "Profit Margin %"
    )

    if revenue is not None and revenue <= -10:
        recommendations.append(
            "مراجعة مصادر انخفاض الإيرادات، "
            "مثل حجم الطلب، قنوات البيع، الأسعار "
            "والعروض التسويقية."
        )

    if profit is not None and profit <= -10:
        recommendations.append(
            "مراجعة هيكل التكاليف ومصادر الربح "
            "لتحديد البنود الأكثر تأثيرًا على انخفاض الربحية."
        )

    if (
        margin_diff is not None
        and margin_diff <= -3
    ):
        recommendations.append(
            "تحليل هامش الربح حسب المنتج أو الخدمة "
            "ومراجعة التسعير والتكاليف المباشرة."
        )

    if (
        transactions is not None
        and transactions <= -10
    ):
        recommendations.append(
            "فحص أسباب انخفاض عدد عمليات البيع، "
            "مع مراجعة التحويل وإعادة الشراء "
            "وسهولة عملية الشراء."
        )

    if (
        customers is not None
        and customers <= -5
    ):
        recommendations.append(
            "مراجعة اكتساب العملاء والاحتفاظ بهم "
            "وتحديد ما إذا كان الانخفاض من عملاء "
            "جدد أو عملاء متكررين."
        )

    if (
        complaints is not None
        and complaints >= 10
    ):
        recommendations.append(
            "مراجعة أسباب الشكاوى وتصنيفها "
            "حسب الموضوع لتحديد المشكلات الأكثر تكرارًا."
        )

        investigation.append(
            "مقارنة ارتفاع الشكاوى مع نتائج تحليل "
            "آراء العملاء، دون افتراض علاقة سببية."
        )

    return {
        "Recommendations":
            list(dict.fromkeys(recommendations)),
        "Needs Investigation":
            list(dict.fromkeys(investigation))
    }


month_name_input = widgets.Text(
    value="Month 1",
    description="اسم الشهر:",
    style={"description_width": "initial"}
)

new_revenue = widgets.FloatText(
    value=0,
    description="الإيرادات (JD):",
    style={"description_width": "initial"}
)

new_expenses = widgets.FloatText(
    value=0,
    description="المصاريف (JD):",
    style={"description_width": "initial"}
)

new_marketing = widgets.FloatText(
    value=0,
    description="التسويق (JD):",
    style={"description_width": "initial"}
)

new_customers = widgets.IntText(
    value=0,
    description="عدد العملاء:",
    style={"description_width": "initial"}
)

new_transactions = widgets.IntText(
    value=0,
    description="عدد عمليات البيع:",
    style={"description_width": "initial"}
)

new_employees = widgets.IntText(
    value=0,
    description="عدد الموظفين:",
    style={"description_width": "initial"}
)

new_complaints = widgets.IntText(
    value=0,
    description="عدد الشكاوى:",
    style={"description_width": "initial"}
)

new_rating = widgets.FloatSlider(
    value=4.0,
    min=1,
    max=5,
    step=0.1,
    description="تقييم العملاء:",
    style={"description_width": "initial"}
)

save_new_month_button = widgets.Button(
    description="حفظ الشهر ✅",
    button_style="success"
)

new_month_output = widgets.Output()


In [ ]:
# ============================================
# 10) CUSTOMER REVIEW NLP
# ============================================

aspect_keywords = {
    "الجودة": [
        "جودة",
        "الخامة",
        "خامة",
        "القماش",
        "قماش",
        "تشطيب",
        "خياطة",
        "quality",
        "fabric"
    ],
    "المقاس": [
        "مقاس",
        "قياس",
        "ضيق",
        "واسع",
        "size"
    ],
    "السعر": [
        "سعر",
        "السعر",
        "غالي",
        "رخيص",
        "مرتفع",
        "price"
    ],
    "التأخير": [
        "تأخر",
        "تأخير",
        "متأخر",
        "التوصيل",
        "توصيل",
        "delivery",
        "late"
    ],
    "الخدمة": [
        "خدمة",
        "الخدمة",
        "تعامل",
        "موظف",
        "موظفين",
        "استجابة",
        "service"
    ],
    "التغليف": [
        "تغليف",
        "التغليف",
        "باكج",
        "package",
        "packaging"
    ]
}


def analyze_review_sentiment(review):
    result = sentiment_model(
        str(review)
    )[0]

    label_map = {
        "negative": "سلبي",
        "neutral": "محايد",
        "positive": "إيجابي"
    }

    return {
        "Review":
            review,
        "Sentiment":
            label_map.get(
                result["label"].lower(),
                result["label"]
            ),
        "Confidence":
            round(
                float(result["score"]),
                3
            )
    }


def detect_review_aspects(review):
    text = str(review).lower()
    found_aspects = []

    for aspect, keywords in (
        aspect_keywords.items()
    ):
        for keyword in keywords:
            if keyword.lower() in text:
                found_aspects.append(
                    aspect
                )
                break

    if not found_aspects:
        found_aspects.append(
            "غير محدد"
        )

    return found_aspects


def split_review_into_clauses(review):
    text = str(review).strip()

    clauses = re.split(
        r"\s*(?:بس|لكن|ولكن|،|,|؛|;)\s*|\s+و(?=ال)",
        text
    )

    return [
        clause.strip()
        for clause in clauses
        if clause.strip()
    ]


def analyze_aspect_sentiment(review):
    results = []

    for clause in split_review_into_clauses(
        review
    ):
        aspects = detect_review_aspects(
            clause
        )

        if aspects == ["غير محدد"]:
            continue

        sentiment = analyze_review_sentiment(
            clause
        )

        for aspect in aspects:
            results.append({
                "Review":
                    review,
                "Aspect":
                    aspect,
                "Context":
                    clause,
                "Aspect Sentiment":
                    sentiment["Sentiment"],
                "Confidence":
                    sentiment["Confidence"]
            })

    return results


def build_aspect_summary(reviews):
    aspect_results = []

    for review in reviews:
        aspect_results.extend(
            analyze_aspect_sentiment(
                review
            )
        )

    if not aspect_results:
        return pd.DataFrame()

    df = pd.DataFrame(
        aspect_results
    )

    summary = (
        df
        .groupby(
            ["Aspect", "Aspect Sentiment"]
        )
        .size()
        .unstack(fill_value=0)
    )

    for sentiment in [
        "إيجابي",
        "محايد",
        "سلبي"
    ]:
        if sentiment not in summary.columns:
            summary[sentiment] = 0

    summary = summary[
        ["إيجابي", "محايد", "سلبي"]
    ]

    summary["Total Mentions"] = (
        summary[
            ["إيجابي", "محايد", "سلبي"]
        ]
        .sum(axis=1)
    )

    summary["Positive %"] = (
        summary["إيجابي"]
        / summary["Total Mentions"]
        * 100
    )

    summary["Negative %"] = (
        summary["سلبي"]
        / summary["Total Mentions"]
        * 100
    )

    confidence_summary = (
        df
        .groupby("Aspect")[
            "Confidence"
        ]
        .mean()
    )

    summary["Average Confidence"] = (
        confidence_summary
    )

    return (
        summary
        .sort_values(
            ["Negative %", "Total Mentions"],
            ascending=[False, False]
        )
        .reset_index()
    )


customer_reviews = []
aspect_summary = pd.DataFrame()

review_input = widgets.Textarea(
    value="",
    placeholder="اكتب رأي العميل هنا...",
    description="رأي العميل:",
    layout=widgets.Layout(
        width="700px",
        height="90px"
    ),
    style={
        "description_width": "initial"
    }
)

add_review_button = widgets.Button(
    description="إضافة وتحليل الرأي 🤖",
    button_style="info"
)

review_output = widgets.Output()


def add_review_and_analyze(
    button=None
):
    global aspect_summary

    with review_output:
        clear_output()

        review = review_input.value.strip()

        if not review:
            print("⚠️ اكتبي رأي العميل أولًا.")
            return

        customer_reviews.append(
            review
        )

        sentiment = analyze_review_sentiment(
            review
        )

        aspect_summary = build_aspect_summary(
            customer_reviews
        )

        print(
            "Sentiment:",
            sentiment["Sentiment"]
        )
        print(
            "Confidence:",
            f"{sentiment['Confidence'] * 100:.1f}%"
        )

        aspects = detect_review_aspects(
            review
        )

        print(
            "Aspects:",
            "، ".join(aspects)
        )

        print(
            f"\n✅ عدد المراجعات المحللة: "
            f"{len(customer_reviews)}"
        )

    if (
        "open_existing_business_section" in globals()
        and globals().get("current_main_section") == "existing"
    ):
        open_existing_business_section()


add_review_button.on_click(
    add_review_and_analyze
)


def generate_integrated_business_insights(
    comparison_df,
    aspect_summary
):
    base = analyze_business_changes(
        comparison_df
    )

    priority_alerts = list(
        base["Warnings"]
    )
    business_context = list(
        base["Observations"]
    )
    positive_signals = list(
        base["Positives"]
    )

    if (
        aspect_summary is not None
        and not aspect_summary.empty
    ):
        for _, row in aspect_summary.iterrows():
            aspect = row["Aspect"]
            mentions = int(
                row["Total Mentions"]
            )
            positive = float(
                row["Positive %"]
            )
            negative = float(
                row["Negative %"]
            )

            if (
                mentions >= 2
                and negative >= 60
            ):
                priority_alerts.append(
                    f"آراء العملاء حول '{aspect}' "
                    f"تظهر اتجاهًا سلبيًا "
                    f"({negative:.0f}% من الإشارات، "
                    f"{mentions} إشارات)."
                )

            if (
                mentions >= 2
                and positive >= 60
            ):
                positive_signals.append(
                    f"آراء العملاء حول '{aspect}' "
                    f"تميل إلى الإيجابية "
                    f"({positive:.0f}% من الإشارات)."
                )

    return {
        "Priority Alerts":
            list(dict.fromkeys(priority_alerts)),
        "Business Context":
            list(dict.fromkeys(business_context)),
        "Positive Signals":
            list(dict.fromkeys(positive_signals))
    }


In [ ]:
# ============================================
# 11) EXISTING-BUSINESS DASHBOARD + SAVE
# ============================================

def show_existing_business_dashboard():
    valid_history = (
        get_valid_performance_df()
    )

    if len(valid_history) < 2:
        display(
            HTML(
                """
                <h2>📊 Business Performance Dashboard</h2>
                <p>
                ⚠️ يلزم إدخال بيانات فعلية لشهرين على الأقل
                قبل إنشاء المقارنة ولوحة الأداء.
                </p>
                """
            )
        )

        if len(valid_history) == 1:
            print(
                f"✅ الشهر المحفوظ حاليًا: "
                f"{valid_history.iloc[0]['Month']}"
            )
        else:
            print(
                "لا توجد أشهر تشغيل فعلية محفوظة حاليًا."
            )

        return

    previous, latest = (
        get_latest_two_months()
    )

    current_comparison = (
        build_current_month_comparison(
            previous,
            latest
        )
    )

    current_analysis = (
        analyze_business_changes(
            current_comparison
        )
    )

    current_recommendations = (
        generate_business_recommendations(
            current_comparison
        )
    )

    feedback_available = (
        isinstance(
            aspect_summary,
            pd.DataFrame
        )
        and not aspect_summary.empty
        and "Aspect" in aspect_summary.columns
    )

    if feedback_available:
        current_integrated = (
            generate_integrated_business_insights(
                current_comparison,
                aspect_summary
            )
        )
    else:
        current_integrated = {
            "Priority Alerts":
                current_analysis["Warnings"],
            "Business Context":
                current_analysis["Observations"],
            "Positive Signals":
                current_analysis["Positives"]
        }

    revenue_change = safe_percent_change(
        latest["Revenue"],
        previous["Revenue"]
    )

    profit_change = safe_percent_change(
        latest["Profit"],
        previous["Profit"]
    )

    display(
        HTML(
            f"""
            <h2>📊 Business Performance Dashboard</h2>
            <h4>
            Latest period: {latest['Month']}
            &nbsp; | &nbsp;
            Compared with: {previous['Month']}
            </h4>
            """
        )
    )

    display(
        HTML(
            f"""
            <div style="
                display:flex;
                gap:15px;
                flex-wrap:wrap;
                margin-top:15px;
            ">
                <div style="border:1px solid #ddd;padding:15px;border-radius:10px;width:190px;">
                    <b>Revenue</b><br>
                    {latest['Revenue']:.2f} JD<br>
                    Change: {format_change(revenue_change)}
                </div>

                <div style="border:1px solid #ddd;padding:15px;border-radius:10px;width:190px;">
                    <b>Profit</b><br>
                    {latest['Profit']:.2f} JD<br>
                    Change: {format_change(profit_change)}
                </div>

                <div style="border:1px solid #ddd;padding:15px;border-radius:10px;width:190px;">
                    <b>Profit Margin</b><br>
                    {latest['Profit Margin %']:.2f}%
                </div>

                <div style="border:1px solid #ddd;padding:15px;border-radius:10px;width:190px;">
                    <b>Customers</b><br>
                    {int(latest['Customers'])}
                </div>

                <div style="border:1px solid #ddd;padding:15px;border-radius:10px;width:190px;">
                    <b>Complaint Rate</b><br>
                    {latest['Complaint Rate %']:.2f}%
                </div>

                <div style="border:1px solid #ddd;padding:15px;border-radius:10px;width:190px;">
                    <b>Customer Rating</b><br>
                    {latest['Customer Rating']:.1f} / 5
                </div>
            </div>
            """
        )
    )

    print("\n🚨 Priority Alerts")
    print("----------------")

    if current_integrated[
        "Priority Alerts"
    ]:
        for item in current_integrated[
            "Priority Alerts"
        ]:
            print("•", item)
    else:
        print(
            "لا توجد إشارات ذات أولوية مرتفعة."
        )

    print("\n📊 Business Context")
    print("----------------")

    if current_integrated[
        "Business Context"
    ]:
        for item in current_integrated[
            "Business Context"
        ]:
            print("•", item)
    else:
        print(
            "• لا توجد ملاحظات سياقية إضافية حاليًا."
        )

    print("\n✅ Positive Signals")
    print("----------------")

    if current_integrated[
        "Positive Signals"
    ]:
        for item in current_integrated[
            "Positive Signals"
        ]:
            print("•", item)
    else:
        print(
            "• لا توجد إشارة إيجابية كافية "
            "ضمن المقارنة الحالية."
        )

    print("\n💡 Recommended Actions")
    print("----------------")

    if current_recommendations[
        "Recommendations"
    ]:
        for item in current_recommendations[
            "Recommendations"
        ]:
            print("•", item)
    else:
        print(
            "• لا توجد توصيات تلقائية إضافية "
            "ضمن المؤشرات الحالية."
        )

    if current_recommendations[
        "Needs Investigation"
    ]:
        print("\n🔎 Needs Investigation")
        print("----------------")

        for item in current_recommendations[
            "Needs Investigation"
        ]:
            print("•", item)

    print("\n🗣️ Customer Feedback Summary")
    print("----------------")

    if feedback_available:
        print(
            "ملاحظة: ملخص الآراء مرتبط بعينة "
            "المراجعات الحالية، وليس بشهر محدد "
            "ما لم يتم ربط المراجعات زمنيًا."
        )

        display(
            aspect_summary[
                [
                    "Aspect",
                    "Total Mentions",
                    "Positive %",
                    "Negative %",
                    "Average Confidence"
                ]
            ].round(2)
        )
    else:
        print(
            "لا توجد بيانات مراجعات محللة متاحة حاليًا."
        )


def save_new_month(
    button=None
):
    with new_month_output:
        clear_output()

        snapshot = {
            "Revenue":
                float(new_revenue.value),
            "Expenses":
                float(new_expenses.value),
            "Marketing Spend":
                float(new_marketing.value),
            "Customers":
                int(new_customers.value),
            "Transactions":
                int(new_transactions.value),
            "Employees":
                int(new_employees.value),
            "Complaints":
                int(new_complaints.value),
            "Customer Rating":
                float(new_rating.value)
        }

        core_values = [
            snapshot["Revenue"],
            snapshot["Expenses"],
            snapshot["Customers"],
            snapshot["Transactions"]
        ]

        if all(
            value == 0
            for value in core_values
        ):
            print(
                "⚠️ لم يتم حفظ الشهر لأن "
                "البيانات الأساسية فارغة."
            )
            return

        month_name = (
            month_name_input.value.strip()
            or "Month"
        )

        add_or_update_month(
            month_name,
            snapshot
        )

        print(
            f"تم حفظ {month_name} بنجاح ✅"
        )

    if (
        "current_main_section" in globals()
        and current_main_section == "existing"
    ):
        open_existing_business_section()


save_new_month_button.on_click(
    save_new_month
)


## Final prototype notes

- **New Business:** activity selection → project profile → dynamic capital plan → location screening → risks → target customer → recommendations.
- **Existing Business:** monthly performance tracking → KPI comparison → alerts → recommendations → customer-review NLP.
- Historical demand evidence is only used when a suitable proxy is available.
- Real-estate similar-business density is shown contextually and is not automatically treated as positive.
- ML success-probability prediction is intentionally **not enabled** until an appropriate historical dataset and defensible target are available.


In [ ]:
# ============================================
# 12) FINAL APPLICATION INTERFACE
# ============================================

new_business_button = widgets.Button(
    description="🚀 مشروع جديد",
    button_style="success",
    layout=widgets.Layout(
        width="220px",
        height="50px"
    )
)

existing_business_button = widgets.Button(
    description="📊 مشروع قائم",
    button_style="info",
    layout=widgets.Layout(
        width="220px",
        height="50px"
    )
)

back_to_menu_button = widgets.Button(
    description="⬅️ العودة للقائمة الرئيسية",
    layout=widgets.Layout(
        width="230px",
        height="40px"
    )
)

refresh_new_analysis_button = widgets.Button(
    description="تحديث التحليل 🔄",
    button_style="warning"
)

main_app_output = widgets.Output()
current_main_section = "menu"


def show_main_menu():
    global current_main_section
    current_main_section = "menu"

    with main_app_output:
        clear_output(wait=True)

        display(
            HTML(
                """
                <div style="text-align:center; padding:20px;">
                    <h1>AI-Powered SME Decision Support</h1>
                    <p>
                    منصة ذكية لدعم قرارات المشاريع
                    الصغيرة والمتوسطة في الأردن
                    </p>
                    <h3>اختر نوع المشروع</h3>
                </div>
                """
            )
        )

        display(
            widgets.HBox(
                [
                    new_business_button,
                    existing_business_button
                ],
                layout=widgets.Layout(
                    justify_content="center",
                    gap="20px"
                )
            )
        )


def display_new_business_setup():
    display(
        HTML(
            """
            <h4>1️⃣ اختيار النشاط الاقتصادي</h4>
            """
        ),
        sector_dropdown,
        activity_search_box,
        search_button,
        search_output,
        approve_activity_button,
        approval_output
    )

    display(
        HTML(
            """
            <h4>2️⃣ بيانات المشروع</h4>
            """
        ),
        new_preferred_governorate,
        new_can_relocate,
        new_operating_model,
        new_available_capital,
        save_new_project_button,
        new_project_output
    )

    if new_project_profile:
        update_capital_input_fields()

        display(
            HTML(
                """
                <h4>3️⃣ تخطيط رأس المال</h4>
                """
            ),
            startup_capital_box,
            HTML(
                "<b>التكاليف الشهرية</b>"
            ),
            monthly_capital_box,
            working_capital_months_general,
            reserve_percentage_general,
            show_all_capital_fields,
            capital_calculate_button,
            capital_calculation_output
        )


def open_new_business_section(
    button=None
):
    global current_main_section
    current_main_section = "new"

    with main_app_output:
        clear_output(wait=True)

        display(back_to_menu_button)

        display(
            HTML(
                """
                <hr>
                <h3>🚀 تحليل مشروع جديد</h3>
                """
            )
        )

        display_new_business_setup()

        if not new_project_profile:
            print(
                "\nℹ️ اختاري النشاط واحفظي بيانات "
                "المشروع لعرض التحليل."
            )
            return

        try:
            refresh_new_business_analysis()

            display(
                plan_selector,
                refresh_new_analysis_button
            )

            show_new_business_dashboard()

        except Exception as e:
            print(
                "⚠️ تعذر تحديث تحليل المشروع الجديد:"
            )
            print(e)


def display_existing_month_entry():
    display(
        HTML(
            """
            <h4>➕ إضافة أو تحديث شهر</h4>
            <p>
            استخدمي نفس اسم الشهر لتحديثه بدل تكراره.
            </p>
            """
        ),
        month_name_input,
        new_revenue,
        new_expenses,
        new_marketing,
        new_customers,
        new_transactions,
        new_employees,
        new_complaints,
        new_rating,
        save_new_month_button,
        new_month_output
    )


def display_review_entry():
    display(
        HTML(
            """
            <h4>🗣️ تحليل رأي عميل</h4>
            """
        ),
        review_input,
        add_review_button,
        review_output
    )


def open_existing_business_section(
    button=None
):
    global current_main_section
    current_main_section = "existing"

    with main_app_output:
        clear_output(wait=True)

        display(back_to_menu_button)

        display(
            HTML(
                """
                <hr>
                <h3>📊 تحليل مشروع قائم</h3>
                """
            )
        )

        display_existing_month_entry()

        display(
            HTML("<hr>")
        )

        display_review_entry()

        display(
            HTML("<hr>")
        )

        show_existing_business_dashboard()


def return_to_main_menu(
    button=None
):
    show_main_menu()


def update_plan_dashboard(change):
    if (
        change["name"] == "value"
        and current_main_section == "new"
        and new_project_profile
    ):
        open_new_business_section()


def refresh_new_analysis(
    button=None
):
    if (
        current_main_section == "new"
        and new_project_profile
    ):
        open_new_business_section()


# Defensive cleanup prevents duplicated callbacks after re-running this cell.
for button in [
    new_business_button,
    existing_business_button,
    back_to_menu_button,
    refresh_new_analysis_button
]:
    try:
        button._click_handlers.callbacks = []
    except Exception:
        pass

new_business_button.on_click(
    open_new_business_section
)

existing_business_button.on_click(
    open_existing_business_section
)

back_to_menu_button.on_click(
    return_to_main_menu
)

refresh_new_analysis_button.on_click(
    refresh_new_analysis
)

try:
    plan_selector.unobserve(
        update_plan_dashboard,
        names="value"
    )
except Exception:
    pass

plan_selector.observe(
    update_plan_dashboard,
    names="value"
)

display(main_app_output)
show_main_menu()


Output()

In [ ]:
# ============================================
# DEMO VERSION — CLOTHING STORE
# نسخة تجريبية فقط
# ============================================

DEMO_MODE = True

if DEMO_MODE:

    # ========================================
    # 1) NEW BUSINESS DEMO
    # ========================================

    new_project_profile.clear()

    new_project_profile.update({
        "ISIC Code": "4771",

        "Economic Activity":
            "Retail sale of clothing, footwear and leather articles "
            "in specialized stores",

        "Business Family": "retail_trade",

        "Project Status": "new",

        "Preferred Governorate": "Irbid",

        "Can Relocate": True,

        "Operating Model": "small_physical",

        "Available Capital": None
    })


    # تحديث حقول رأس المال حسب نوع المشروع
    try:
        update_capital_input_fields()
    except Exception:
        pass


    # إعادة حساب تحليل المشروع الجديد
    try:
        refresh_new_business_analysis()

    except Exception as e:
        print(
            "⚠️ New Business Demo:",
            e
        )


    # ========================================
    # 2) EXISTING BUSINESS DEMO
    # ========================================

    performance_history = []
    performance_df = pd.DataFrame()


    # Month 1
    add_or_update_month(
        "Month 1",
        {
            "Revenue": 2500,
            "Expenses": 1645,
            "Marketing Spend": 390,
            "Customers": 33,
            "Transactions": 65,
            "Employees": 3,
            "Complaints": 5,
            "Customer Rating": 4.0
        }
    )


    # Month 2
    add_or_update_month(
        "Month 2",
        {
            "Revenue": 3350,
            "Expenses": 2005,
            "Marketing Spend": 350,
            "Customers": 36,
            "Transactions": 41,
            "Employees": 3,
            "Complaints": 7,
            "Customer Rating": 4.0
        }
    )


    # ========================================
    # 3) CUSTOMER REVIEWS DEMO
    # ========================================

    customer_reviews = [
        "الجودة ممتازة والخامة مرتبة.",
        "المقاس لم يكن مناسبًا والتبديل تأخر.",
        "الخدمة ممتازة والموظفون متعاونون.",
        "السعر مرتفع قليلًا مقارنة بالجودة.",
        "التغليف جميل والمقاس مناسب.",
        "التوصيل تأخر لكن القطعة جودتها ممتازة."
    ]


    try:
        aspect_summary = build_aspect_summary(
            customer_reviews
        )

    except Exception as e:
        aspect_summary = pd.DataFrame()

        print(
            "⚠️ Customer Reviews Demo:",
            e
        )


    # ========================================
    # DONE
    # ========================================

    print("✅ Demo data loaded successfully")
    print()
    print("🛍️ New Business Demo:")
    print("Clothing Store")
    print("ISIC Code: 4771")
    print("Governorate: Irbid")
    print()
    print("📊 Existing Business:")
    print("Month 1 + Month 2 loaded")
    print()
    print("🗣️ Customer Reviews:")
    print("Demo reviews loaded")

✅ Demo data loaded successfully

🛍️ New Business Demo:
Clothing Store
ISIC Code: 4771
Governorate: Irbid

📊 Existing Business:
Month 1 + Month 2 loaded

🗣️ Customer Reviews:
Demo reviews loaded


In [ ]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 30.9 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st

# ============================================
# PAGE SETTINGS
# ============================================

st.set_page_config(
    page_title="SME Decision Support",
    page_icon="📊",
    layout="wide"
)

# ============================================
# SESSION STATE
# ============================================

if "page" not in st.session_state:
    st.session_state.page = "home"


# ============================================
# HOME PAGE
# ============================================

def show_home():

    st.markdown(
        """
        <div style="text-align:center; padding:40px 10px;">
            <h1>📊 منصة دعم قرارات المشاريع</h1>
            <p style="font-size:19px;">
                تحليل ودعم قرارات المشاريع الصغيرة والمتوسطة في الأردن
            </p>
        </div>
        """,
        unsafe_allow_html=True
    )

    st.markdown("### اختر نوع المشروع")

    col1, col2 = st.columns(2)

    with col1:

        if st.button(
            "🚀 مشروع جديد",
            use_container_width=True
        ):
            st.session_state.page = "new"
            st.rerun()

    with col2:

        if st.button(
            "📊 مشروع قائم",
            use_container_width=True
        ):
            st.session_state.page = "existing"
            st.rerun()


# ============================================
# NEW BUSINESS PAGE
# ============================================

def show_new_business():

    if st.button("⬅️ العودة للقائمة الرئيسية"):
        st.session_state.page = "home"
        st.rerun()

    st.divider()

    st.header("🚀 تحليل مشروع جديد")

    st.info(
        "هنا سنضيف اختيار النشاط، الموقع، رأس المال "
        "وتحليل Free / Premium."
    )


# ============================================
# EXISTING BUSINESS PAGE
# ============================================

def show_existing_business():

    if st.button("⬅️ العودة للقائمة الرئيسية"):
        st.session_state.page = "home"
        st.rerun()

    st.divider()

    st.header("📊 تحليل مشروع قائم")

    st.info(
        "هنا سنضيف البيانات الشهرية، مؤشرات الأداء، "
        "التنبيهات وتحليل آراء العملاء."
    )


# ============================================
# APP ROUTER
# ============================================

if st.session_state.page == "home":
    show_home()

elif st.session_state.page == "new":
    show_new_business()

elif st.session_state.page == "existing":
    show_existing_business()

Writing app.py


In [ ]:
# ============================================
# RUN STREAMLIT APP
# ============================================

import subprocess
import time
from google.colab import output

# تشغيل Streamlit بالخلفية
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port=8501",
        "--server.headless=true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# نعطيه ثواني حتى يشتغل
time.sleep(4)

print("✅ Streamlit app is running")

# فتح التطبيق
output.serve_kernel_port_as_window(8501)

✅ Streamlit app is running
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output

In [ ]:
import subprocess
import time
import re
import requests
import shutil
from pathlib import Path

# 1) تأكد إن Streamlit شغال
r = requests.get("http://127.0.0.1:8501", timeout=5)
print("✅ Streamlit شغال")

# 2) ثبّت cloudflared إذا مش موجود
if shutil.which("cloudflared") is None:
    subprocess.run([
        "wget",
        "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O",
        "/usr/local/bin/cloudflared"
    ], check=True)

    subprocess.run([
        "chmod",
        "+x",
        "/usr/local/bin/cloudflared"
    ], check=True)

# 3) وقف أي رابط قديم
subprocess.run(
    ["pkill", "-f", "cloudflared"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

# 4) اعمل رابط جديد
log_file = open("cloudflare.log", "w")

cloudflare_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("🌐 جاري إنشاء الرابط...")

url = None

for _ in range(30):
    time.sleep(2)
    log_file.flush()

    text = Path("cloudflare.log").read_text(
        encoding="utf-8",
        errors="ignore"
    )

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        text
    )

    if match:
        url = match.group(0)
        break

if url:
    print("\n✅ افتحي هذا الرابط:")
    print(url)
else:
    print("❌ ما طلع الرابط")

✅ Streamlit شغال
🌐 جاري إنشاء الرابط...

✅ افتحي هذا الرابط:
https://compact-chronicles-nightlife-syndicate.trycloudflare.com


In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

In [ ]:
%%writefile app.py

import streamlit as st

# ============================================
# PAGE SETTINGS
# ============================================

st.set_page_config(
    page_title="SME Decision Support",
    page_icon="📊",
    layout="wide"
)

# ============================================
# SESSION STATE
# ============================================

if "page" not in st.session_state:
    st.session_state.page = "home"

if "new_project" not in st.session_state:
    st.session_state.new_project = {}


# ============================================
# HOME PAGE
# ============================================

def show_home():

    st.markdown(
        """
        <div style="text-align:center; padding:40px 10px;">
            <h1>📊 منصة دعم قرارات المشاريع</h1>
            <p style="font-size:19px;">
                دعم وتحليل قرارات المشاريع الصغيرة والمتوسطة في الأردن
            </p>
        </div>
        """,
        unsafe_allow_html=True
    )

    st.subheader("اختر نوع المشروع")

    col1, col2 = st.columns(2)

    with col1:
        if st.button(
            "🚀 مشروع جديد",
            use_container_width=True
        ):
            st.session_state.page = "new"
            st.rerun()

    with col2:
        if st.button(
            "📊 مشروع قائم",
            use_container_width=True
        ):
            st.session_state.page = "existing"
            st.rerun()


# ============================================
# NEW BUSINESS
# ============================================

def show_new_business():

    if st.button("⬅️ العودة"):
        st.session_state.page = "home"
        st.rerun()

    st.title("🚀 تحليل مشروع جديد")

    st.markdown("### 1️⃣ بيانات المشروع")

    activity = st.selectbox(
        "النشاط الاقتصادي",
        [
            "4771 - Retail sale of clothing, footwear and leather articles in specialized stores"
        ]
    )

    governorate = st.selectbox(
        "المحافظة المفضلة",
        [
            "Amman",
            "Balqa",
            "Zarqa",
            "Madaba",
            "Irbid",
            "Mafraq",
            "Jarash",
            "Ajlun",
            "Karak",
            "Tafiela",
            "Ma'an",
            "Aqaba"
        ],
        index=4
    )

    can_relocate = st.radio(
        "هل يمكن تغيير الموقع؟",
        ["نعم", "لا"],
        horizontal=True
    )

    operating_model = st.selectbox(
        "نموذج التشغيل",
        [
            "small_physical",
            "online",
            "hybrid",
            "home_based"
        ]
    )

    available_capital = st.number_input(
        "رأس المال المتوفر - اختياري (JD)",
        min_value=0.0,
        value=0.0,
        step=100.0
    )

    plan = st.radio(
        "نوع التحليل",
        ["Free", "Premium"],
        horizontal=True
    )

    if st.button(
        "حفظ بيانات المشروع ✅",
        use_container_width=True
    ):

        st.session_state.new_project = {
            "ISIC Code": "4771",
            "Economic Activity":
                "Retail sale of clothing, footwear and leather articles in specialized stores",
            "Business Family": "retail_trade",
            "Preferred Governorate": governorate,
            "Can Relocate": can_relocate == "نعم",
            "Operating Model": operating_model,
            "Available Capital":
                available_capital if available_capital > 0 else None,
            "Plan": plan
        }

        st.success("✅ تم حفظ بيانات المشروع")

    # ========================================
    # SHOW SAVED PROFILE
    # ========================================

    if st.session_state.new_project:

        st.divider()

        st.markdown("### 📋 ملف المشروع")

        project = st.session_state.new_project

        col1, col2, col3 = st.columns(3)

        with col1:
            st.metric(
                "ISIC",
                project["ISIC Code"]
            )

        with col2:
            st.metric(
                "المحافظة",
                project["Preferred Governorate"]
            )

        with col3:
            st.metric(
                "الخطة",
                project["Plan"]
            )

        st.write(
            "**النشاط:**",
            project["Economic Activity"]
        )

        st.info(
            "الخطوة التالية ستكون ربط هذه البيانات "
            "بنموذج تحليل الموقع ورأس المال والمخاطر."
        )


# ============================================
# EXISTING BUSINESS
# ============================================

def show_existing_business():

    if st.button("⬅️ العودة"):
        st.session_state.page = "home"
        st.rerun()

    st.title("📊 تحليل مشروع قائم")

    st.info(
        "سنربط البيانات الشهرية والتحليل في الخطوة التالية."
    )


# ============================================
# ROUTER
# ============================================

if st.session_state.page == "home":
    show_home()

elif st.session_state.page == "new":
    show_new_business()

elif st.session_state.page == "existing":
    show_existing_business()

Overwriting app.py


In [ ]:
import subprocess
import time
import requests

# إيقاف أي نسخة قديمة من Streamlit
subprocess.run(
    ["pkill", "-f", "streamlit run app.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

# تشغيل Streamlit من جديد
streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

# فحص السيرفر
try:
    response = requests.get(
        "http://127.0.0.1:8501",
        timeout=5
    )

    print("✅ Streamlit server is working")
    print("Status:", response.status_code)

except Exception as e:
    print("❌ Streamlit did not start")
    print(e)

    print("\n--- LOG ---")

    with open("streamlit.log", "r") as f:
        print(f.read())

✅ Streamlit server is working
Status: 200


In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=900
)

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=900
)

<IPython.core.display.Javascript object>

In [ ]:
# ============================================
# OPEN STREAMLIT THROUGH CLOUDFLARE
# ============================================

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess
import re
import time

cloudflare_process = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("جاري إنشاء رابط التطبيق...")

for _ in range(60):

    line = cloudflare_process.stdout.readline()

    match = re.search(
        r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com",
        line
    )

    if match:
        print("\n✅ افتحي هذا الرابط:")
        print(match.group(0))
        break

جاري إنشاء رابط التطبيق...

✅ افتحي هذا الرابط:
https://repair-dicke-greeting-automobiles.trycloudflare.com


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")

old = '''        st.info(
            "الخطوة التالية ستكون ربط هذه البيانات "
            "بنموذج تحليل الموقع ورأس المال والمخاطر."
        )
'''

new = '''        st.caption(
            "سيتم استخدام هذه البيانات لتخصيص التحليل والنتائج."
        )
'''

code = code.replace(old, new)

path.write_text(code, encoding="utf-8")

print("✅ تم تنظيف النص الظاهر للمستخدم")

✅ تم تنظيف النص الظاهر للمستخدم


In [ ]:
%%writefile app.py

import os
import numpy as np
import pandas as pd
import streamlit as st

# ============================================
# PAGE SETTINGS
# ============================================

st.set_page_config(
    page_title="SME Decision Support",
    page_icon="📊",
    layout="wide"
)

st.markdown(
    """
    <style>
    html, body, [class*="css"] {
        direction: rtl;
        text-align: right;
    }
    </style>
    """,
    unsafe_allow_html=True
)

# ============================================
# DATA
# ============================================

DATA_FILE = "SME_AI_Master_Data_v4_Location_Scoring.xlsx"

if not os.path.exists(DATA_FILE):
    st.error(
        "ملف البيانات الرئيسي غير موجود في جلسة Colab."
    )
    st.stop()

governorates = pd.read_excel(
    DATA_FILE,
    sheet_name="Governorate Master"
)

table4 = pd.read_excel(
    DATA_FILE,
    sheet_name="Table4 Structured"
)

demand = pd.read_excel(
    DATA_FILE,
    sheet_name="Demand Proxy 2017",
    usecols="A:P",
    nrows=28
)


# ============================================
# HELPERS
# ============================================

def normalize_isic4(value):

    if pd.isna(value):
        return None

    code = str(value).strip()

    if code.endswith(".0"):
        code = code[:-2]

    if code.isdigit():
        code = code.zfill(4)

    return code


def normalize(series, reverse=False):

    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            [50] * len(series),
            index=series.index
        )

    score = (
        (series - minimum)
        / (maximum - minimum)
        * 100
    )

    if reverse:
        score = 100 - score

    return score


table4["ISIC Code"] = (
    table4["ISIC Code"]
    .apply(normalize_isic4)
)

activities = table4[
    table4["Level"] == "Detailed activity (4-digit)"
].copy()


governorate_map = {
    "عمان": "Amman",
    "البلقاء": "Balqa",
    "الزرقاء": "Zarqa",
    "مادبا": "Madaba",
    "إربد": "Irbid",
    "المفرق": "Mafraq",
    "جرش": "Jarash",
    "عجلون": "Ajlun",
    "الكرك": "Karak",
    "الطفيلة": "Tafiela",
    "معان": "Ma'an",
    "العقبة": "Aqaba"
}


# ============================================
# LOCATION ANALYSIS
# ============================================

def find_clothing_demand_category():

    categories = demand["Category (EN)"].dropna()

    matches = categories[
        categories.astype(str)
        .str.lower()
        .str.contains("clothing|footwear")
    ]

    if not matches.empty:
        return matches.iloc[0]

    return None


def calculate_location_scores(
    isic_code,
    demand_category=None
):

    business = activities[
        activities["ISIC Code"] == isic_code
    ]

    if business.empty:
        raise ValueError(
            "لم يتم العثور على النشاط في بيانات المنشآت."
        )

    business = business.iloc[0]

    demand_row = None

    if demand_category is not None:

        match = demand[
            demand["Category (EN)"]
            == demand_category
        ]

        if not match.empty:
            demand_row = match.iloc[0]

    rows = []

    for arabic_name, english_name in governorate_map.items():

        gov_match = governorates[
            governorates["Governorate"]
            == english_name
        ]

        if gov_match.empty:
            continue

        gov_row = gov_match.iloc[0]

        population = float(
            gov_row["Population 2025"]
        )

        registration_growth = float(
            gov_row["H1 Growth 2023→2024"]
        )

        similar_businesses = pd.to_numeric(
            business.get(
                english_name,
                0
            ),
            errors="coerce"
        )

        if pd.isna(similar_businesses):
            similar_businesses = 0

        similar_businesses = float(
            similar_businesses
        )

        businesses_per_10k = (
            similar_businesses
            / population
            * 10000
        )

        if demand_row is not None:

            historical_demand = pd.to_numeric(
                demand_row.get(
                    arabic_name,
                    np.nan
                ),
                errors="coerce"
            )

        else:

            historical_demand = np.nan

        rows.append({
            "Governorate": english_name,
            "Population": population,
            "Historical Demand": historical_demand,
            "Similar Businesses": similar_businesses,
            "Businesses per 10k": businesses_per_10k,
            "Registration Growth": registration_growth
        })


    result = pd.DataFrame(rows)

    # حجم السوق
    result["Log Population"] = np.log1p(
        result["Population"]
    )

    result["Market Score"] = normalize(
        result["Log Population"]
    )

    # الطلب التاريخي
    if result["Historical Demand"].notna().any():

        result["Demand Score"] = normalize(
            result["Historical Demand"]
        )

    else:

        result["Demand Score"] = np.nan

    # البيع بالتجزئة:
    # كثافة أقل للأنشطة المشابهة = فرصة تنافسية أعلى نسبيًا
    result["Sector Score"] = normalize(
        result["Businesses per 10k"],
        reverse=True
    )

    # اتجاه التسجيل
    result["Trend Score"] = normalize(
        result["Registration Growth"]
    )

    components = [
        "Market Score",
        "Trend Score",
        "Sector Score"
    ]

    if result["Demand Score"].notna().any():
        components.append(
            "Demand Score"
        )

    result["Location Suitability Score"] = (
        result[components]
        .mean(axis=1)
    )

    result = result.sort_values(
        "Location Suitability Score",
        ascending=False
    ).reset_index(drop=True)

    result["Rank"] = result.index + 1

    return result, components


# ============================================
# SESSION STATE
# ============================================

if "page" not in st.session_state:
    st.session_state.page = "home"

if "new_project" not in st.session_state:
    st.session_state.new_project = {}


# ============================================
# HOME
# ============================================

def show_home():

    st.markdown(
        """
        <div style="text-align:center; padding:35px;">
            <h1>📊 منصة دعم قرارات المشاريع</h1>
            <p>
            دعم وتحليل قرارات المشاريع الصغيرة والمتوسطة في الأردن
            </p>
        </div>
        """,
        unsafe_allow_html=True
    )

    col1, col2 = st.columns(2)

    with col1:

        if st.button(
            "🚀 مشروع جديد",
            use_container_width=True
        ):

            st.session_state.page = "new"
            st.rerun()

    with col2:

        if st.button(
            "📊 مشروع قائم",
            use_container_width=True
        ):

            st.session_state.page = "existing"
            st.rerun()


# ============================================
# NEW BUSINESS
# ============================================

def show_new_business():

    if st.button("⬅️ العودة"):
        st.session_state.page = "home"
        st.rerun()

    st.title("🚀 تحليل مشروع جديد")

    st.subheader("بيانات المشروع")

    governorate = st.selectbox(
        "المحافظة المفضلة",
        list(governorate_map.values()),
        index=4
    )

    can_relocate = st.radio(
        "هل يمكن تغيير الموقع؟",
        ["نعم", "لا"],
        horizontal=True
    )

    operating_model = st.selectbox(
        "نموذج التشغيل",
        [
            "small_physical",
            "online",
            "hybrid",
            "home_based"
        ]
    )

    available_capital = st.number_input(
        "رأس المال المتوفر - اختياري (JD)",
        min_value=0.0,
        value=0.0,
        step=100.0
    )

    plan = st.radio(
        "نوع التحليل",
        ["Free", "Premium"],
        horizontal=True
    )

    if st.button(
        "تحليل المشروع 📊",
        use_container_width=True
    ):

        st.session_state.new_project = {
            "ISIC Code": "4771",
            "Economic Activity":
                "Retail sale of clothing, footwear and leather articles in specialized stores",
            "Business Family": "retail_trade",
            "Preferred Governorate": governorate,
            "Can Relocate": can_relocate == "نعم",
            "Operating Model": operating_model,
            "Available Capital":
                available_capital if available_capital > 0 else None,
            "Plan": plan
        }


    if st.session_state.new_project:

        project = st.session_state.new_project

        st.divider()

        st.header("📋 ملف المشروع")

        c1, c2, c3 = st.columns(3)

        c1.metric(
            "ISIC",
            project["ISIC Code"]
        )

        c2.metric(
            "المحافظة",
            project["Preferred Governorate"]
        )

        c3.metric(
            "الخطة",
            project["Plan"]
        )

        st.write(
            "**النشاط:**",
            project["Economic Activity"]
        )

        # ====================================
        # LOCATION
        # ====================================

        st.divider()

        st.header("📍 تحليل ملاءمة الموقع")

        try:

            demand_category = (
                find_clothing_demand_category()
            )

            location_df, components = (
                calculate_location_scores(
                    "4771",
                    demand_category
                )
            )

            selected = location_df[
                location_df["Governorate"]
                == project["Preferred Governorate"]
            ].iloc[0]

            top = location_df.iloc[0]

            c1, c2, c3 = st.columns(3)

            c1.metric(
                "Location Suitability",
                f"{selected['Location Suitability Score']:.1f} / 100"
            )

            c2.metric(
                "ترتيب المحافظة",
                f"{int(selected['Rank'])} / {len(location_df)}"
            )

            c3.metric(
                "أعلى محافظة",
                top["Governorate"]
            )

            st.caption(
                "مؤشر ملاءمة الموقع هو Screening Score "
                "لدعم المقارنة بين المحافظات، وليس احتمالًا لنجاح المشروع."
            )

            # Premium details
            if project["Plan"] == "Premium":

                st.subheader(
                    "أفضل المحافظات حسب المؤشر"
                )

                display_df = location_df[
                    [
                        "Rank",
                        "Governorate",
                        "Location Suitability Score"
                    ]
                ].head(5).copy()

                display_df[
                    "Location Suitability Score"
                ] = display_df[
                    "Location Suitability Score"
                ].round(1)

                st.dataframe(
                    display_df,
                    use_container_width=True,
                    hide_index=True
                )

                st.subheader(
                    "مكونات تحليل المحافظة المختارة"
                )

                component_cols = [
                    "Market Score",
                    "Trend Score",
                    "Sector Score"
                ]

                if (
                    "Demand Score" in selected.index
                    and pd.notna(
                        selected["Demand Score"]
                    )
                ):
                    component_cols.append(
                        "Demand Score"
                    )

                component_data = pd.DataFrame({
                    "المؤشر": component_cols,
                    "النتيجة": [
                        round(
                            float(selected[col]),
                            1
                        )
                        for col in component_cols
                    ]
                })

                st.dataframe(
                    component_data,
                    use_container_width=True,
                    hide_index=True
                )

                if demand_category is not None:

                    st.info(
                        "مؤشر الطلب المستخدم يعتمد على "
                        "الإنفاق الأسري التاريخي المرتبط "
                        "بالملابس والأحذية من HEIS 2017، "
                        "ولا يمثل الطلب الحالي بشكل مباشر."
                    )

                if (
                    selected["Governorate"]
                    != top["Governorate"]
                ):

                    gap = (
                        top[
                            "Location Suitability Score"
                        ]
                        -
                        selected[
                            "Location Suitability Score"
                        ]
                    )

                    st.write(
                        f"أعلى نتيجة كانت في "
                        f"**{top['Governorate']}** "
                        f"بفارق **{gap:.1f} نقطة** "
                        f"عن المحافظة المختارة."
                    )

        except Exception as e:

            st.error(
                f"تعذر إجراء تحليل الموقع: {e}"
            )


# ============================================
# EXISTING BUSINESS
# ============================================

def show_existing_business():

    if st.button("⬅️ العودة"):
        st.session_state.page = "home"
        st.rerun()

    st.title("📊 تحليل مشروع قائم")

    st.info(
        "أدخل بيانات الأداء الشهرية لمتابعة المشروع."
    )


# ============================================
# ROUTER
# ============================================

if st.session_state.page == "home":
    show_home()

elif st.session_state.page == "new":
    show_new_business()

elif st.session_state.page == "existing":
    show_existing_business()

Overwriting app.py


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")

code = code.replace(
'''            c1.metric(
                "Location Suitability",
                f"{selected['Location Suitability Score']:.1f} / 100"
            )
''',
'''            c1.metric(
                "ملاءمة الموقع",
                f"{selected['Location Suitability Score']:.1f}",
                help="النتيجة من 100"
            )
'''
)

code = code.replace(
'''            c2.metric(
                "ترتيب المحافظة",
                f"{int(selected['Rank'])} / {len(location_df)}"
            )
''',
'''            c2.metric(
                "ترتيب المحافظة",
                f"{int(selected['Rank'])}",
                help=f"من أصل {len(location_df)} محافظة"
            )
'''
)

path.write_text(code, encoding="utf-8")

print("✅ Display fixed")

✅ Display fixed


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# 1) CAPITAL PLANNER FUNCTION
# ============================================

capital_function = r'''
# ============================================
# CAPITAL PLANNER
# ============================================

def render_capital_planner(project):

    st.header("💰 تخطيط رأس المال")

    st.caption(
        "هذا تقدير تخطيطي مبني على التكاليف التي تدخلها، "
        "وليس تقديرًا سوقيًا مؤكدًا لرأس المال المطلوب."
    )

    st.subheader("تكاليف التأسيس")

    col1, col2 = st.columns(2)

    with col1:

        equipment = st.number_input(
            "معدات وأجهزة (JD)",
            min_value=0.0,
            value=0.0,
            step=50.0,
            key="cap_equipment"
        )

        inventory = st.number_input(
            "مخزون أولي (JD)",
            min_value=0.0,
            value=0.0,
            step=50.0,
            key="cap_inventory"
        )

        premises = st.number_input(
            "موقع / إيجار / تأمين أولي (JD)",
            min_value=0.0,
            value=0.0,
            step=50.0,
            key="cap_premises"
        )

        setup = st.number_input(
            "تجهيز وديكور (JD)",
            min_value=0.0,
            value=0.0,
            step=50.0,
            key="cap_setup"
        )

        licensing = st.number_input(
            "ترخيص وإجراءات قانونية (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="cap_licensing"
        )

    with col2:

        packaging = st.number_input(
            "تغليف (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="cap_packaging"
        )

        delivery_setup = st.number_input(
            "تجهيز التوصيل (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="cap_delivery"
        )

        launch_marketing = st.number_input(
            "تسويق الإطلاق (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="cap_launch_marketing"
        )

        technology = st.number_input(
            "تقنية / أنظمة / برامج (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="cap_technology"
        )

        other_startup = st.number_input(
            "تكاليف تأسيس أخرى (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="cap_other_startup"
        )


    st.subheader("التكاليف الشهرية")

    col1, col2 = st.columns(2)

    with col1:

        salaries = st.number_input(
            "رواتب شهرية (JD)",
            min_value=0.0,
            value=0.0,
            step=50.0,
            key="monthly_salaries"
        )

        rent = st.number_input(
            "إيجار شهري (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="monthly_rent"
        )

        utilities = st.number_input(
            "مياه / كهرباء / خدمات (JD)",
            min_value=0.0,
            value=0.0,
            step=10.0,
            key="monthly_utilities"
        )

    with col2:

        marketing = st.number_input(
            "تسويق شهري (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="monthly_marketing"
        )

        logistics = st.number_input(
            "نقل وتوصيل شهري (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="monthly_logistics"
        )

        other_monthly = st.number_input(
            "مصاريف شهرية أخرى (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0,
            key="monthly_other"
        )


    st.subheader("افتراضات التخطيط")

    c1, c2 = st.columns(2)

    with c1:

        months = st.number_input(
            "عدد أشهر رأس المال التشغيلي",
            min_value=0,
            value=3,
            step=1,
            key="working_months"
        )

    with c2:

        reserve_percent = st.number_input(
            "نسبة الاحتياطي %",
            min_value=0.0,
            value=10.0,
            step=1.0,
            key="reserve_percent"
        )

    st.caption(
        "القيم الافتراضية 3 أشهر و10% احتياطي هي افتراضات "
        "قابلة للتعديل وليست قواعد مالية ثابتة."
    )


    if st.button(
        "احسب رأس المال 💰",
        use_container_width=True
    ):

        startup_cost = sum([
            equipment,
            inventory,
            premises,
            setup,
            licensing,
            packaging,
            delivery_setup,
            launch_marketing,
            technology,
            other_startup
        ])

        monthly_cost = sum([
            salaries,
            rent,
            utilities,
            marketing,
            logistics,
            other_monthly
        ])

        operating_capital = (
            monthly_cost * months
        )

        cumulative = (
            startup_cost
            + operating_capital
        )

        reserve_amount = (
            cumulative
            * reserve_percent
            / 100
        )

        total_required = (
            cumulative
            + reserve_amount
        )

        available = project.get(
            "Available Capital"
        )

        st.session_state.capital_result = {
            "startup": startup_cost,
            "monthly": monthly_cost,
            "operating": operating_capital,
            "reserve": reserve_amount,
            "total": total_required,
            "available": available
        }


    result = st.session_state.get(
        "capital_result"
    )

    if result:

        if result["total"] <= 0:

            st.warning(
                "أدخل بعض التكاليف أولًا حتى يتم حساب "
                "رأس المال التخطيطي."
            )

        else:

            st.subheader("نتيجة التخطيط")

            c1, c2, c3 = st.columns(3)

            c1.metric(
                "تكاليف الإطلاق",
                f"{result['startup']:,.0f} JD"
            )

            c2.metric(
                "التكلفة الشهرية",
                f"{result['monthly']:,.0f} JD"
            )

            c3.metric(
                "رأس المال التشغيلي",
                f"{result['operating']:,.0f} JD"
            )

            c1, c2 = st.columns(2)

            c1.metric(
                "الاحتياطي",
                f"{result['reserve']:,.0f} JD"
            )

            c2.metric(
                "إجمالي رأس المال التخطيطي",
                f"{result['total']:,.0f} JD"
            )


            if (
                result["available"] is not None
                and result["available"] > 0
            ):

                difference = (
                    result["available"]
                    - result["total"]
                )

                st.subheader(
                    "مقارنة برأس المال المتوفر"
                )

                if difference >= 0:

                    st.success(
                        f"رأس المال المتوفر أعلى من التقدير "
                        f"التخطيطي بحوالي "
                        f"{difference:,.0f} JD."
                    )

                else:

                    st.warning(
                        f"يوجد فرق تخطيطي قدره "
                        f"{abs(difference):,.0f} JD "
                        f"بين رأس المال المتوفر والتقدير الحالي."
                    )
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def render_capital_planner(project):" not in code:
    code = code.replace(
        marker,
        capital_function + "\n\n" + marker
    )


# ============================================
# 2) SHOW CAPITAL AFTER LOCATION ANALYSIS
# ============================================

old = '''        except Exception as e:

            st.error(
                f"تعذر إجراء تحليل الموقع: {e}"
            )
'''

new = '''        except Exception as e:

            st.error(
                f"تعذر إجراء تحليل الموقع: {e}"
            )

        # ====================================
        # CAPITAL PLANNING
        # ====================================

        st.divider()

        render_capital_planner(
            project
        )
'''

if "render_capital_planner(\n            project\n        )" not in code:
    code = code.replace(
        old,
        new
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Capital planner added to Streamlit")

✅ Capital planner added to Streamlit


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# 1) RISK ANALYSIS FUNCTION
# ============================================

risk_function = r'''
# ============================================
# PRE-LAUNCH RISK ANALYSIS
# ============================================

def render_prelaunch_risks(project):

    st.header("⚠️ تحليل مخاطر ما قبل الإطلاق")

    st.caption(
        "يعتمد هذا التحليل على قواعد Screening واضحة "
        "ومدخلات المشروع الحالية، ولا يمثل احتمالًا إحصائيًا للفشل أو النجاح."
    )

    # التفاصيل متاحة في Premium
    if project.get("Plan") != "Premium":

        st.info(
            "🔒 تحليل المخاطر التفصيلي متاح ضمن Premium."
        )

        return


    high_priority = []
    watch_points = []
    data_gaps = []
    positive_factors = []


    # ========================================
    # CAPITAL
    # ========================================

    capital = st.session_state.get(
        "capital_result"
    )

    available = project.get(
        "Available Capital"
    )

    required = None

    if capital and capital.get("total", 0) > 0:

        required = float(
            capital["total"]
        )


    if (
        available is not None
        and required is not None
    ):

        gap = (
            float(available)
            - required
        )

        if gap < 0:

            high_priority.append(
                f"رأس المال المتوفر أقل من التقدير "
                f"التخطيطي الحالي بحوالي "
                f"{abs(gap):,.0f} JD."
            )

        else:

            positive_factors.append(
                "رأس المال المتوفر يغطي التقدير "
                "التخطيطي الحالي لرأس المال."
            )

    elif available is None:

        data_gaps.append(
            "لم يتم تحديد رأس المال المتوفر، "
            "لذلك لا يمكن تقييم فجوة التمويل."
        )

    elif required is None:

        data_gaps.append(
            "لم يتم حساب خطة رأس المال بعد، "
            "لذلك لا يمكن تقييم فجوة التمويل."
        )


    # ========================================
    # LOCATION
    # ========================================

    location = st.session_state.get(
        "location_summary"
    )

    if location:

        rank = int(
            location["Rank"]
        )

        total_governorates = int(
            location.get(
                "Total Governorates",
                12
            )
        )

        if rank <= 3:

            positive_factors.append(
                f"المحافظة المختارة ضمن أعلى "
                f"{rank} نتائج في Screening الموقع."
            )

        elif rank >= 8:

            watch_points.append(
                f"ترتيب المحافظة المختارة "
                f"{rank} من {total_governorates} "
                f"في Screening الموقع."
            )

            if not project.get(
                "Can Relocate",
                False
            ):

                high_priority.append(
                    "الموقع المختار منخفض نسبيًا "
                    "في Screening الموقع، "
                    "ولا توجد مرونة حالية لتغييره."
                )


        # ====================================
        # DEMAND EVIDENCE
        # ====================================

        if not location.get(
            "Demand Category"
        ):

            data_gaps.append(
                "لا يوجد مؤشر طلب مباشر موثوق "
                "مرتبط بهذا النشاط ضمن البيانات الحالية."
            )

    else:

        data_gaps.append(
            "تحليل الموقع غير متوفر حاليًا."
        )


    # ========================================
    # OPERATING MODEL
    # ========================================

    operating_model = project.get(
        "Operating Model"
    )

    if operating_model in [
        "home",
        "home_based"
    ]:

        watch_points.append(
            "نموذج التشغيل من المنزل يحتاج إلى "
            "التحقق من ملاءمته لطبيعة النشاط "
            "ومتطلبات الترخيص قبل الإطلاق."
        )


    # ========================================
    # SECTOR INTERPRETATION
    # ========================================

    family = project.get(
        "Business Family"
    )

    strategy_map = {
        "manufacturing": "cluster",
        "construction": "cluster",
        "transport_logistics": "cluster",
        "real_estate": "ambiguous_market",
        "retail_trade": "competition",
        "hospitality_food": "competition",
        "personal_services": "competition",
        "professional_services": "competition",
        "business_support": "competition",
        "education": "competition",
        "entertainment": "competition",
        "information_technology": "competition",
        "financial_services": "competition",
        "health_social": "competition",
        "agriculture": "resource_based",
        "mining": "resource_based",
        "utilities_environment": "resource_based",
        "other": "competition"
    }

    strategy = strategy_map.get(
        family,
        "competition"
    )

    if strategy == "cluster":

        watch_points.append(
            "ارتفاع نشاط القطاع قد يدل على سوق نشط "
            "أو تجمع أعمال، لكنه قد يعني أيضًا "
            "منافسة أعلى؛ لذلك لا يتم تفسيره وحده "
            "كميزة مؤكدة."
        )

    elif strategy == "resource_based":

        data_gaps.append(
            "هذا القطاع يحتاج بيانات موارد ومتغيرات "
            "متخصصة للحصول على تحليل موقع أدق."
        )


    # ========================================
    # DISPLAY
    # ========================================

    col1, col2 = st.columns(2)

    with col1:

        st.subheader("🚨 مخاطر عالية الأولوية")

        if high_priority:

            for item in high_priority:
                st.error(item)

        else:

            st.success(
                "لا توجد مخاطر عالية الأولوية "
                "ضمن البيانات الحالية."
            )


        st.subheader("👀 نقاط تحتاج متابعة")

        if watch_points:

            for item in watch_points:
                st.warning(item)

        else:

            st.write(
                "لا توجد نقاط متابعة إضافية حاليًا."
            )


    with col2:

        st.subheader("📌 فجوات البيانات")

        if data_gaps:

            for item in data_gaps:
                st.info(item)

        else:

            st.write(
                "لا توجد فجوات أساسية ظاهرة "
                "ضمن البيانات المستخدمة حاليًا."
            )


        st.subheader("✅ عوامل إيجابية")

        if positive_factors:

            for item in positive_factors:
                st.success(item)

        else:

            st.write(
                "لا توجد عوامل إيجابية إضافية "
                "مسجلة حاليًا."
            )
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def render_prelaunch_risks(project):" not in code:

    code = code.replace(
        marker,
        risk_function + "\n\n" + marker
    )


# ============================================
# 2) SAVE LOCATION RESULT FOR RISK ENGINE
# ============================================

old_location = '''            top = location_df.iloc[0]
'''

new_location = '''            top = location_df.iloc[0]

            st.session_state.location_summary = {
                "Governorate":
                    selected["Governorate"],

                "Rank":
                    int(selected["Rank"]),

                "Score":
                    float(
                        selected[
                            "Location Suitability Score"
                        ]
                    ),

                "Top Governorate":
                    top["Governorate"],

                "Total Governorates":
                    len(location_df),

                "Demand Category":
                    demand_category
            }
'''

if "st.session_state.location_summary" not in code:

    code = code.replace(
        old_location,
        new_location
    )


# ============================================
# 3) DISPLAY RISK ANALYSIS AFTER CAPITAL
# ============================================

old_capital = '''        render_capital_planner(
            project
        )
'''

new_capital = '''        render_capital_planner(
            project
        )

        st.divider()

        render_prelaunch_risks(
            project
        )
'''

if "render_prelaunch_risks(\n            project\n        )" not in code:

    code = code.replace(
        old_capital,
        new_capital
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Pre-launch risk analysis added")

✅ Pre-launch risk analysis added


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# 1) TARGET CUSTOMER + RECOMMENDATIONS
# ============================================

new_functions = r'''
# ============================================
# TARGET CUSTOMER
# ============================================

def build_target_customer_profile(project):

    family = project.get(
        "Business Family",
        "other"
    )

    operating_model = project.get(
        "Operating Model",
        "unknown"
    )

    isic_code = str(
        project.get(
            "ISIC Code",
            ""
        )
    )

    # ----------------------------------------
    # Customer Segments + Needs
    # ----------------------------------------

    if family == "retail_trade":

        segments = [
            "مستهلكون أفراد يبحثون عن المنتج للاستخدام الشخصي.",
            "عملاء محليون ضمن منطقة المشروع.",
            "عملاء يقارنون بين السعر والجودة والتوفر."
        ]

        needs = [
            "سعر مناسب",
            "توفر المنتج",
            "سهولة الشراء",
            "جودة موثوقة"
        ]

    elif family == "hospitality_food":

        segments = [
            "عملاء أفراد يبحثون عن الطعام أو خدمات الضيافة.",
            "عملاء محليون ومتكررون.",
            "عملاء التوصيل والطلبات الرقمية."
        ]

        needs = [
            "الجودة",
            "السعر",
            "سرعة الخدمة",
            "النظافة",
            "التوصيل"
        ]

    elif family == "manufacturing":

        segments = [
            "تجار ومحلات قد تشترى بالجملة.",
            "شركات تحتاج المنتجات ضمن سلسلة التوريد.",
            "عملاء أفراد إذا كان المشروع يبيع مباشرة."
        ]

        needs = [
            "ثبات الجودة",
            "السعر",
            "القدرة على التوريد",
            "الالتزام بالمواعيد"
        ]

    elif family == "real_estate":

        if isic_code == "6810":

            segments = [
                "أفراد يبحثون عن شراء أو استئجار عقار.",
                "أسر تبحث عن سكن مناسب.",
                "مستثمرون يبحثون عن عقارات للاستثمار."
            ]

            needs = [
                "السعر",
                "الموقع",
                "حالة العقار",
                "وضوح المعلومات",
                "الثقة والشفافية"
            ]

        else:

            segments = [
                "أفراد يحتاجون خدمات عقارية.",
                "أصحاب أو مستثمرون في العقارات."
            ]

            needs = [
                "الثقة",
                "الموقع",
                "السعر",
                "وضوح المعلومات"
            ]

    elif family == "information_technology":

        segments = [
            "مشاريع صغيرة ومتوسطة تحتاج حلولًا تقنية.",
            "شركات تبحث عن تطوير أو دعم أنظمة رقمية.",
            "عملاء أفراد حسب طبيعة الخدمة."
        ]

        needs = [
            "حل واضح للمشكلة",
            "سهولة الاستخدام",
            "الدعم الفني",
            "السعر"
        ]

    elif family == "professional_services":

        segments = [
            "أفراد يحتاجون خدمات متخصصة.",
            "شركات ومؤسسات تحتاج خدمات مهنية.",
            "مشاريع صغيرة تحتاج خبرات خارجية."
        ]

        needs = [
            "الثقة",
            "الخبرة",
            "وضوح الخدمة",
            "سرعة الاستجابة"
        ]

    elif family == "education":

        segments = [
            "طلاب أو متعلمون يحتاجون الخدمة التعليمية.",
            "أولياء أمور حسب طبيعة الخدمة.",
            "مؤسسات تحتاج تدريبًا أو تطوير مهارات."
        ]

        needs = [
            "جودة التعليم",
            "النتائج",
            "المرونة",
            "السعر"
        ]

    elif family == "health_social":

        segments = [
            "أفراد يحتاجون الخدمة الصحية أو الاجتماعية.",
            "أسر تبحث عن خدمات مناسبة لأفرادها.",
            "جهات تحتاج خدمات متخصصة."
        ]

        needs = [
            "الثقة",
            "سهولة الوصول",
            "جودة الخدمة",
            "الاستجابة"
        ]

    else:

        segments = [
            "عملاء أفراد محتملون حسب طبيعة النشاط.",
            "شركات أو مؤسسات قد تستفيد من الخدمة.",
            "عملاء ضمن السوق المحلي للمشروع."
        ]

        needs = [
            "القيمة مقابل السعر",
            "الجودة",
            "الثقة",
            "سهولة الوصول"
        ]


    # ----------------------------------------
    # Marketing Channels
    # ----------------------------------------

    if operating_model in [
        "online",
        "home_online"
    ]:

        channels = [
            "وسائل التواصل الاجتماعي",
            "الإعلانات الرقمية",
            "التواصل المباشر عبر الإنترنت"
        ]

    elif operating_model in [
        "home",
        "home_based"
    ]:

        channels = [
            "وسائل التواصل الاجتماعي",
            "التوصيات والإحالات",
            "التسويق المحلي"
        ]

    elif operating_model in [
        "small_physical",
        "full_physical"
    ]:

        channels = [
            "الموقع الفعلي",
            "Google Maps والمنصات المحلية",
            "وسائل التواصل الاجتماعي",
            "التسويق المحلي"
        ]

    elif operating_model == "production":

        channels = [
            "التواصل مع التجار والشركات",
            "المعارض والفعاليات التجارية",
            "التسويق الرقمي B2B",
            "المبيعات المباشرة"
        ]

    else:

        channels = [
            "التسويق الرقمي",
            "التسويق المحلي"
        ]


    return {
        "segments": segments,
        "needs": needs,
        "channels": channels
    }


# ============================================
# CUSTOMER + RECOMMENDATIONS DISPLAY
# ============================================

def render_customer_and_recommendations(project):

    if project.get("Plan") != "Premium":

        st.info(
            "🔒 تحليل العميل المستهدف والتوصيات "
            "التفصيلية متاح ضمن Premium."
        )

        return


    profile = build_target_customer_profile(
        project
    )


    # ========================================
    # TARGET CUSTOMER
    # ========================================

    st.header("👥 العميل المستهدف")

    st.caption(
        "هذا توصيف أولي مبني على نوع النشاط "
        "وطريقة التشغيل، ويحتاج إلى التحقق "
        "من خلال بيانات فعلية من السوق."
    )

    col1, col2, col3 = st.columns(3)

    with col1:

        st.subheader("🎯 الشرائح المحتملة")

        for item in profile["segments"]:
            st.write("•", item)

    with col2:

        st.subheader("💭 احتياجات العملاء")

        for item in profile["needs"]:
            st.write("•", item)

    with col3:

        st.subheader("📣 قنوات الوصول")

        for item in profile["channels"]:
            st.write("•", item)


    # ========================================
    # RECOMMENDATIONS
    # ========================================

    st.divider()

    st.header("💡 التوصيات")

    priorities = []
    recommendations = []
    validation_steps = []


    # ----------------------------------------
    # Capital
    # ----------------------------------------

    capital = st.session_state.get(
        "capital_result"
    )

    available = project.get(
        "Available Capital"
    )

    if (
        capital
        and capital.get("total", 0) > 0
        and available is not None
    ):

        difference = (
            float(available)
            - float(capital["total"])
        )

        if difference < 0:

            priorities.append(
                f"يوجد فرق تخطيطي في رأس المال "
                f"بحوالي {abs(difference):,.0f} JD. "
                f"يمكن مراجعة تكاليف البداية أو "
                f"تقسيم الإنفاق إلى مراحل."
            )

        else:

            recommendations.append(
                "رأس المال المتوفر يغطي التقدير "
                "التخطيطي الحالي."
            )


    # ----------------------------------------
    # Location
    # ----------------------------------------

    location = st.session_state.get(
        "location_summary"
    )

    if location:

        rank = int(
            location["Rank"]
        )

        if rank == 1:

            recommendations.append(
                "المحافظة المختارة حصلت على أعلى "
                "نتيجة في Screening الموقع الحالي."
            )

        elif rank <= 3:

            recommendations.append(
                f"المحافظة المختارة ضمن أعلى "
                f"{rank} نتائج في Screening الموقع."
            )

        else:

            validation_steps.append(
                "مقارنة المحافظة المختارة مع "
                "المحافظات الأعلى ترتيبًا قبل "
                "اتخاذ القرار النهائي."
            )


    # ----------------------------------------
    # Risk summary
    # ----------------------------------------

    risks = st.session_state.get(
        "prelaunch_risk_summary",
        {}
    )

    for item in risks.get(
        "high_priority",
        []
    ):

        if (
            "رأس المال" in item
            and any(
                "رأس المال" in x
                for x in priorities
            )
        ):
            continue

        if item not in priorities:
            priorities.append(item)


    for item in risks.get(
        "watch_points",
        []
    ):

        if item not in validation_steps:
            validation_steps.append(item)


    for item in risks.get(
        "data_gaps",
        []
    ):

        if item not in validation_steps:
            validation_steps.append(item)


    # ----------------------------------------
    # Customer validation
    # ----------------------------------------

    recommendations.append(
        "اختبار الجمهور المستهدف مبدئيًا قبل "
        "الإطلاق للتأكد من وجود اهتمام فعلي."
    )

    recommendations.append(
        "بدء تجربة تسويقية صغيرة باستخدام "
        "قنوات الوصول المقترحة قبل زيادة "
        "الإنفاق التسويقي."
    )

    validation_steps.append(
        "جمع بيانات فعلية من السوق قبل الالتزام "
        "الكامل بالمشروع، مثل استبيان أو مقابلات "
        "مع عملاء محتملين أو تجربة أولية صغيرة."
    )


    # ========================================
    # DISPLAY
    # ========================================

    col1, col2 = st.columns(2)

    with col1:

        st.subheader("🚩 إجراءات ذات أولوية")

        if priorities:

            for item in priorities:
                st.warning(item)

        else:

            st.success(
                "لا توجد إجراءات عاجلة واضحة "
                "ضمن البيانات الحالية."
            )


        st.subheader("✅ توصيات")

        for item in recommendations:
            st.success(item)


    with col2:

        st.subheader("🔎 خطوات التحقق")

        for item in validation_steps:
            st.info(item)
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def build_target_customer_profile(project):" not in code:

    code = code.replace(
        marker,
        new_functions + "\n\n" + marker
    )


# ============================================
# 2) SAVE RISK RESULTS
# ============================================

risk_marker = '''    # ========================================
    # DISPLAY
    # ========================================

    col1, col2 = st.columns(2)
'''

risk_replacement = '''    # ========================================
    # SAVE RISK RESULTS
    # ========================================

    st.session_state.prelaunch_risk_summary = {
        "high_priority": high_priority,
        "watch_points": watch_points,
        "data_gaps": data_gaps,
        "positive_factors": positive_factors
    }


    # ========================================
    # DISPLAY
    # ========================================

    col1, col2 = st.columns(2)
'''

if "st.session_state.prelaunch_risk_summary" not in code:

    code = code.replace(
        risk_marker,
        risk_replacement,
        1
    )


# ============================================
# 3) DISPLAY AFTER RISK ANALYSIS
# ============================================

old = '''        render_prelaunch_risks(
            project
        )
'''

new = '''        render_prelaunch_risks(
            project
        )

        st.divider()

        render_customer_and_recommendations(
            project
        )
'''

if "render_customer_and_recommendations(" not in code:

    code = code.replace(
        old,
        new
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Target customer and recommendations added")

✅ Target customer and recommendations added


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")

old = '''        render_prelaunch_risks(
            project
        )
'''

new = '''        render_prelaunch_risks(
            project
        )

        st.divider()

        render_customer_and_recommendations(
            project
        )
'''

if new not in code:
    code = code.replace(
        old,
        new,
        1
    )

path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Customer and recommendations display fixed")

✅ Customer and recommendations display fixed


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# EXISTING BUSINESS HELPERS
# ============================================

existing_helpers = r'''
# ============================================
# EXISTING BUSINESS HELPERS
# ============================================

def safe_percent_change(current, previous):

    current = float(current)
    previous = float(previous)

    if previous == 0 and current == 0:
        return 0.0

    if previous == 0:
        return None

    return (
        (current - previous)
        / abs(previous)
        * 100
    )


def calculate_existing_kpis(snapshot):

    revenue = float(snapshot["Revenue"])
    expenses = float(snapshot["Expenses"])
    marketing = float(snapshot["Marketing Spend"])

    customers = int(snapshot["Customers"])
    transactions = int(snapshot["Transactions"])
    complaints = int(snapshot["Complaints"])

    profit = revenue - expenses

    profit_margin = (
        profit / revenue * 100
        if revenue > 0
        else 0
    )

    average_order_value = (
        revenue / transactions
        if transactions > 0
        else 0
    )

    complaint_rate = (
        complaints / customers * 100
        if customers > 0
        else 0
    )

    marketing_ratio = (
        marketing / revenue * 100
        if revenue > 0
        else 0
    )

    return {
        "Profit": profit,
        "Profit Margin %": profit_margin,
        "Average Order Value": average_order_value,
        "Complaint Rate %": complaint_rate,
        "Marketing / Revenue %": marketing_ratio
    }


def save_existing_month(month_name, snapshot):

    if "performance_history" not in st.session_state:
        st.session_state.performance_history = []

    record = {
        "Month": month_name,
        **snapshot,
        **calculate_existing_kpis(snapshot)
    }

    # إذا الشهر موجود، نحدثه بدل تكراره
    updated = False

    for i, old_record in enumerate(
        st.session_state.performance_history
    ):

        if old_record["Month"] == month_name:

            st.session_state.performance_history[i] = record
            updated = True
            break

    if not updated:
        st.session_state.performance_history.append(record)
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def calculate_existing_kpis(snapshot):" not in code:

    code = code.replace(
        marker,
        existing_helpers + "\n\n" + marker
    )


# ============================================
# REPLACE EXISTING BUSINESS PAGE
# ============================================

start_marker = """
# ============================================
# EXISTING BUSINESS
# ============================================
"""

end_marker = """
# ============================================
# ROUTER
# ============================================
"""

new_existing_section = r'''
# ============================================
# EXISTING BUSINESS
# ============================================

def show_existing_business():

    if st.button("⬅️ العودة"):
        st.session_state.page = "home"
        st.rerun()

    st.title("📊 تحليل مشروع قائم")

    st.caption(
        "أدخل بيانات الأداء الشهرية لمتابعة التغيرات "
        "ومقارنة الأداء بين الفترات."
    )


    # ========================================
    # MONTH INPUT
    # ========================================

    st.header("➕ إضافة أو تحديث شهر")

    month_name = st.text_input(
        "اسم الشهر",
        placeholder="مثال: Month 1"
    )

    c1, c2 = st.columns(2)

    with c1:

        revenue = st.number_input(
            "الإيرادات (JD)",
            min_value=0.0,
            value=0.0,
            step=50.0
        )

        expenses = st.number_input(
            "المصروفات (JD)",
            min_value=0.0,
            value=0.0,
            step=50.0
        )

        marketing = st.number_input(
            "الإنفاق التسويقي (JD)",
            min_value=0.0,
            value=0.0,
            step=20.0
        )

        customers = st.number_input(
            "عدد العملاء",
            min_value=0,
            value=0,
            step=1
        )

    with c2:

        transactions = st.number_input(
            "عدد العمليات / المبيعات",
            min_value=0,
            value=0,
            step=1
        )

        employees = st.number_input(
            "عدد الموظفين",
            min_value=0,
            value=0,
            step=1
        )

        complaints = st.number_input(
            "عدد الشكاوى",
            min_value=0,
            value=0,
            step=1
        )

        rating = st.number_input(
            "تقييم العملاء",
            min_value=0.0,
            max_value=5.0,
            value=0.0,
            step=0.1
        )


    if st.button(
        "حفظ الشهر ✅",
        use_container_width=True
    ):

        if not month_name.strip():

            st.warning(
                "اكتب اسم الشهر أولًا."
            )

        elif (
            revenue == 0
            and expenses == 0
            and customers == 0
            and transactions == 0
        ):

            st.warning(
                "أدخل بيانات فعلية للشهر قبل الحفظ."
            )

        else:

            snapshot = {
                "Revenue": revenue,
                "Expenses": expenses,
                "Marketing Spend": marketing,
                "Customers": customers,
                "Transactions": transactions,
                "Employees": employees,
                "Complaints": complaints,
                "Customer Rating": rating
            }

            save_existing_month(
                month_name.strip(),
                snapshot
            )

            st.success(
                f"تم حفظ {month_name.strip()} ✅"
            )


    # ========================================
    # SAVED MONTHS
    # ========================================

    history = st.session_state.get(
        "performance_history",
        []
    )

    if history:

        st.divider()

        st.header("📋 الأشهر المحفوظة")

        df = pd.DataFrame(history)

        display_columns = [
            "Month",
            "Revenue",
            "Expenses",
            "Profit",
            "Customers",
            "Transactions",
            "Complaints",
            "Customer Rating"
        ]

        st.dataframe(
            df[display_columns].round(2),
            use_container_width=True,
            hide_index=True
        )


        # ====================================
        # BASIC COMPARISON
        # ====================================

        if len(df) >= 2:

            previous = df.iloc[-2]
            latest = df.iloc[-1]

            st.divider()

            st.header("📈 مقارنة آخر شهرين")

            st.caption(
                f"{latest['Month']} مقارنة مع "
                f"{previous['Month']}"
            )

            revenue_change = safe_percent_change(
                latest["Revenue"],
                previous["Revenue"]
            )

            profit_change = safe_percent_change(
                latest["Profit"],
                previous["Profit"]
            )

            customer_change = safe_percent_change(
                latest["Customers"],
                previous["Customers"]
            )

            transaction_change = safe_percent_change(
                latest["Transactions"],
                previous["Transactions"]
            )


            c1, c2, c3, c4 = st.columns(4)

            c1.metric(
                "الإيرادات",
                f"{latest['Revenue']:,.0f} JD",
                None if revenue_change is None
                else f"{revenue_change:.1f}%"
            )

            c2.metric(
                "الربح",
                f"{latest['Profit']:,.0f} JD",
                None if profit_change is None
                else f"{profit_change:.1f}%"
            )

            c3.metric(
                "العملاء",
                f"{int(latest['Customers'])}",
                None if customer_change is None
                else f"{customer_change:.1f}%"
            )

            c4.metric(
                "العمليات",
                f"{int(latest['Transactions'])}",
                None if transaction_change is None
                else f"{transaction_change:.1f}%"
            )

        else:

            st.info(
                "أدخل بيانات شهر ثانٍ حتى تظهر مقارنة الأداء."
            )
'''


if (
    start_marker in code
    and end_marker in code
):

    before = code.split(start_marker)[0]
    after = code.split(end_marker)[1]

    code = (
        before
        + start_marker
        + new_existing_section
        + "\n"
        + end_marker
        + after
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Existing business monthly tracking added")

✅ Existing business monthly tracking added


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# 1) PERFORMANCE ANALYSIS FUNCTION
# ============================================

analysis_function = r'''
# ============================================
# EXISTING BUSINESS CHANGE ANALYSIS
# ============================================

def analyze_existing_changes(previous, latest):

    warnings = []
    observations = []
    positives = []


    # ----------------------------------------
    # Percentage changes
    # ----------------------------------------

    revenue_change = safe_percent_change(
        latest["Revenue"],
        previous["Revenue"]
    )

    profit_change = safe_percent_change(
        latest["Profit"],
        previous["Profit"]
    )

    customer_change = safe_percent_change(
        latest["Customers"],
        previous["Customers"]
    )

    transaction_change = safe_percent_change(
        latest["Transactions"],
        previous["Transactions"]
    )

    complaint_rate_change = safe_percent_change(
        latest["Complaint Rate %"],
        previous["Complaint Rate %"]
    )

    expenses_change = safe_percent_change(
        latest["Expenses"],
        previous["Expenses"]
    )

    average_order_change = safe_percent_change(
        latest["Average Order Value"],
        previous["Average Order Value"]
    )


    # ----------------------------------------
    # Absolute differences
    # ----------------------------------------

    rating_diff = (
        float(latest["Customer Rating"])
        - float(previous["Customer Rating"])
    )

    margin_diff = (
        float(latest["Profit Margin %"])
        - float(previous["Profit Margin %"])
    )


    # ========================================
    # Revenue
    # ========================================

    if revenue_change is not None:

        if revenue_change <= -20:

            warnings.append(
                f"الإيرادات انخفضت بنسبة "
                f"{abs(revenue_change):.1f}%، "
                f"وهو تغير يحتاج مراجعة ذات أولوية."
            )

        elif revenue_change <= -10:

            observations.append(
                f"الإيرادات انخفضت بنسبة "
                f"{abs(revenue_change):.1f}%."
            )

        elif revenue_change >= 10:

            positives.append(
                f"الإيرادات ارتفعت بنسبة "
                f"{revenue_change:.1f}%."
            )


    # ========================================
    # Profit
    # ========================================

    if profit_change is not None:

        if profit_change <= -20:

            warnings.append(
                f"الربح انخفض بنسبة "
                f"{abs(profit_change):.1f}%، "
                f"ويحتاج تحليل أسباب الانخفاض."
            )

        elif profit_change <= -10:

            observations.append(
                f"الربح انخفض بنسبة "
                f"{abs(profit_change):.1f}%."
            )

        elif profit_change >= 10:

            positives.append(
                f"الربح ارتفع بنسبة "
                f"{profit_change:.1f}%."
            )


    # ========================================
    # Profit Margin
    # ========================================

    if margin_diff <= -5:

        warnings.append(
            f"هامش الربح انخفض بحوالي "
            f"{abs(margin_diff):.1f} نقطة مئوية."
        )

    elif margin_diff >= 3:

        positives.append(
            f"هامش الربح تحسن بحوالي "
            f"{margin_diff:.1f} نقطة مئوية."
        )


    # ========================================
    # Customers
    # ========================================

    if customer_change is not None:

        if customer_change <= -15:

            warnings.append(
                f"عدد العملاء انخفض بنسبة "
                f"{abs(customer_change):.1f}%."
            )

        elif customer_change <= -5:

            observations.append(
                f"عدد العملاء انخفض بنسبة "
                f"{abs(customer_change):.1f}%."
            )

        elif customer_change >= 10:

            positives.append(
                f"عدد العملاء ارتفع بنسبة "
                f"{customer_change:.1f}%."
            )


    # ========================================
    # Transactions
    # ========================================

    if transaction_change is not None:

        if transaction_change <= -20:

            warnings.append(
                f"عدد عمليات البيع انخفض بنسبة "
                f"{abs(transaction_change):.1f}%."
            )

        elif transaction_change <= -10:

            observations.append(
                f"عدد عمليات البيع انخفض بنسبة "
                f"{abs(transaction_change):.1f}%."
            )

        elif transaction_change >= 10:

            positives.append(
                f"عدد عمليات البيع ارتفع بنسبة "
                f"{transaction_change:.1f}%."
            )


    # ========================================
    # Complaint Rate
    # ========================================

    if complaint_rate_change is not None:

        if complaint_rate_change >= 25:

            warnings.append(
                f"معدل الشكاوى ارتفع بنسبة "
                f"{complaint_rate_change:.1f}%."
            )

        elif complaint_rate_change >= 10:

            observations.append(
                f"معدل الشكاوى ارتفع بنسبة "
                f"{complaint_rate_change:.1f}%."
            )

        elif complaint_rate_change <= -20:

            positives.append(
                f"معدل الشكاوى انخفض بنسبة "
                f"{abs(complaint_rate_change):.1f}%."
            )


    # ========================================
    # Customer Rating
    # ========================================

    if rating_diff <= -0.5:

        warnings.append(
            f"تقييم العملاء انخفض بمقدار "
            f"{abs(rating_diff):.1f} نقطة."
        )

    elif rating_diff >= 0.3:

        positives.append(
            f"تقييم العملاء تحسن بمقدار "
            f"{rating_diff:.1f} نقطة."
        )


    # ========================================
    # Combined Context
    # ========================================

    if (
        transaction_change is not None
        and average_order_change is not None
        and transaction_change < 0
        and average_order_change > 0
    ):

        observations.append(
            "انخفض عدد عمليات البيع بينما ارتفع "
            "متوسط قيمة الطلب؛ قد يكون التغير ناتجًا "
            "عن عدد أقل من العمليات بقيم أعلى."
        )


    if (
        expenses_change is not None
        and revenue_change is not None
        and expenses_change > 10
        and revenue_change <= 0
    ):

        observations.append(
            "المصاريف ارتفعت بينما الإيرادات لم ترتفع، "
            "لذلك يُنصح بمراجعة بنود التكلفة."
        )


    return {
        "warnings": list(dict.fromkeys(warnings)),
        "observations": list(dict.fromkeys(observations)),
        "positives": list(dict.fromkeys(positives))
    }
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def analyze_existing_changes(previous, latest):" not in code:

    code = code.replace(
        marker,
        analysis_function + "\n\n" + marker
    )


# ============================================
# 2) SHOW ANALYSIS AFTER MONTH COMPARISON
# ============================================

old = '''            c4.metric(
                "العمليات",
                f"{int(latest['Transactions'])}",
                None if transaction_change is None
                else f"{transaction_change:.1f}%"
            )
'''

new = '''            c4.metric(
                "العمليات",
                f"{int(latest['Transactions'])}",
                None if transaction_change is None
                else f"{transaction_change:.1f}%"
            )


            # ====================================
            # PERFORMANCE INTERPRETATION
            # ====================================

            insights = analyze_existing_changes(
                previous,
                latest
            )

            st.session_state.existing_base_insights = insights

            st.divider()

            st.header("🧭 قراءة الأداء")

            st.caption(
                "التنبيهات التالية تعتمد على حدود تشغيلية "
                "شفافة في النسخة التجريبية، وهي قابلة للتعديل "
                "وليست حدودًا علمية ثابتة لجميع المشاريع."
            )

            col1, col2, col3 = st.columns(3)


            with col1:

                st.subheader("🚨 تنبيهات ذات أولوية")

                if insights["warnings"]:

                    for item in insights["warnings"]:
                        st.error(item)

                else:

                    st.success(
                        "لا توجد إشارات ذات أولوية مرتفعة "
                        "ضمن التغيرات الحالية."
                    )


            with col2:

                st.subheader("🔎 سياق الأداء")

                if insights["observations"]:

                    for item in insights["observations"]:
                        st.info(item)

                else:

                    st.write(
                        "لا توجد ملاحظات سياقية إضافية حاليًا."
                    )


            with col3:

                st.subheader("✅ إشارات إيجابية")

                if insights["positives"]:

                    for item in insights["positives"]:
                        st.success(item)

                else:

                    st.write(
                        "لا توجد إشارات إيجابية إضافية حاليًا."
                    )
'''

if "st.session_state.existing_base_insights" not in code:

    code = code.replace(
        old,
        new,
        1
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Existing business alerts added")

✅ Existing business alerts added


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# 1) ADD IMPORTS
# ============================================

if "import re" not in code:
    code = code.replace(
        "import os\n",
        "import os\nimport re\n"
    )

if "from transformers import pipeline" not in code:
    code = code.replace(
        "import streamlit as st\n",
        "import streamlit as st\nfrom transformers import pipeline\n"
    )


# ============================================
# 2) NLP FUNCTIONS
# ============================================

nlp_code = r'''
# ============================================
# CUSTOMER REVIEW NLP
# ============================================

aspect_keywords = {

    "الجودة": [
        "جودة", "الخامة", "خامة",
        "القماش", "قماش",
        "تشطيب", "خياطة",
        "quality", "fabric"
    ],

    "المقاس": [
        "مقاس", "قياس",
        "ضيق", "واسع",
        "size"
    ],

    "السعر": [
        "سعر", "السعر",
        "غالي", "رخيص",
        "مرتفع", "price"
    ],

    "التأخير": [
        "تأخر", "تأخير",
        "متأخر", "التوصيل",
        "توصيل", "delivery",
        "late"
    ],

    "الخدمة": [
        "خدمة", "الخدمة",
        "تعامل", "موظف",
        "موظفين", "استجابة",
        "service"
    ],

    "التغليف": [
        "تغليف", "التغليف",
        "باكج", "package",
        "packaging"
    ]
}


@st.cache_resource
def load_sentiment_model():

    return pipeline(
        "sentiment-analysis",
        model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
    )


def analyze_review_sentiment(review):

    model = load_sentiment_model()

    result = model(
        str(review)
    )[0]

    label_map = {
        "negative": "سلبي",
        "neutral": "محايد",
        "positive": "إيجابي"
    }

    return {
        "Sentiment":
            label_map.get(
                result["label"].lower(),
                result["label"]
            ),

        "Confidence":
            round(
                float(result["score"]),
                3
            )
    }


def detect_review_aspects(review):

    text = str(review).lower()

    found = []

    for aspect, keywords in aspect_keywords.items():

        for keyword in keywords:

            if keyword.lower() in text:

                found.append(aspect)
                break

    return found


def split_review_into_clauses(review):

    clauses = re.split(
        r"\s*(?:بس|لكن|ولكن|،|,|؛|;)\s*|\s+و(?=ال)",
        str(review).strip()
    )

    return [
        x.strip()
        for x in clauses
        if x.strip()
    ]


def analyze_aspect_sentiment(review):

    results = []

    for clause in split_review_into_clauses(review):

        aspects = detect_review_aspects(
            clause
        )

        if not aspects:
            continue

        sentiment = analyze_review_sentiment(
            clause
        )

        for aspect in aspects:

            results.append({
                "Review": review,
                "Aspect": aspect,
                "Context": clause,
                "Aspect Sentiment":
                    sentiment["Sentiment"],
                "Confidence":
                    sentiment["Confidence"]
            })

    return results


def build_streamlit_aspect_summary(reviews):

    rows = []

    for review in reviews:

        rows.extend(
            analyze_aspect_sentiment(
                review
            )
        )

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    summary = (
        df
        .groupby(
            ["Aspect", "Aspect Sentiment"]
        )
        .size()
        .unstack(fill_value=0)
    )

    for sentiment in [
        "إيجابي",
        "محايد",
        "سلبي"
    ]:

        if sentiment not in summary.columns:
            summary[sentiment] = 0

    summary = summary[
        ["إيجابي", "محايد", "سلبي"]
    ]

    summary["Total Mentions"] = (
        summary[
            ["إيجابي", "محايد", "سلبي"]
        ]
        .sum(axis=1)
    )

    summary["Positive %"] = (
        summary["إيجابي"]
        / summary["Total Mentions"]
        * 100
    )

    summary["Negative %"] = (
        summary["سلبي"]
        / summary["Total Mentions"]
        * 100
    )

    confidence = (
        df
        .groupby("Aspect")["Confidence"]
        .mean()
    )

    summary["Average Confidence"] = confidence

    return (
        summary
        .sort_values(
            ["Negative %", "Total Mentions"],
            ascending=[False, False]
        )
        .reset_index()
    )


def render_customer_review_nlp():

    st.header("🗣️ تحليل آراء العملاء")

    st.caption(
        "يستخدم النظام نموذج NLP متعدد اللغات لتحليل "
        "المشاعر وربطها بجوانب محددة من تجربة العميل."
    )

    if "customer_reviews" not in st.session_state:
        st.session_state.customer_reviews = []

    review = st.text_area(
        "رأي العميل",
        placeholder="مثال: الجودة ممتازة لكن التوصيل تأخر."
    )

    if st.button(
        "تحليل وإضافة الرأي 🔍",
        use_container_width=True
    ):

        if not review.strip():

            st.warning(
                "اكتب رأي العميل أولًا."
            )

        else:

            with st.spinner(
                "جاري تحليل رأي العميل..."
            ):

                overall = analyze_review_sentiment(
                    review.strip()
                )

                st.session_state.customer_reviews.append(
                    review.strip()
                )

                st.success(
                    f"التصنيف العام: "
                    f"{overall['Sentiment']} "
                    f"— ثقة "
                    f"{overall['Confidence']:.2f}"
                )


    reviews = st.session_state.customer_reviews

    if reviews:

        st.subheader("آراء العملاء المدخلة")

        for i, item in enumerate(
            reviews,
            start=1
        ):
            st.write(
                f"{i}. {item}"
            )

        with st.spinner(
            "جاري بناء ملخص الجوانب..."
        ):

            summary = (
                build_streamlit_aspect_summary(
                    reviews
                )
            )

        if not summary.empty:

            st.subheader(
                "📊 ملخص تحليل الجوانب"
            )

            display_summary = summary[
                [
                    "Aspect",
                    "Total Mentions",
                    "Positive %",
                    "Negative %",
                    "Average Confidence"
                ]
            ].copy()

            display_summary[
                "Positive %"
            ] = display_summary[
                "Positive %"
            ].round(1)

            display_summary[
                "Negative %"
            ] = display_summary[
                "Negative %"
            ].round(1)

            display_summary[
                "Average Confidence"
            ] = display_summary[
                "Average Confidence"
            ].round(2)

            st.dataframe(
                display_summary,
                use_container_width=True,
                hide_index=True
            )

            st.session_state.aspect_summary = summary
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def render_customer_review_nlp():" not in code:

    code = code.replace(
        marker,
        nlp_code + "\n\n" + marker
    )


# ============================================
# 3) SHOW NLP ON EXISTING BUSINESS PAGE
# ============================================

old = '''        else:

            st.info(
                "أدخل بيانات شهر ثانٍ حتى تظهر مقارنة الأداء."
            )
'''

new = '''        else:

            st.info(
                "أدخل بيانات شهر ثانٍ حتى تظهر مقارنة الأداء."
            )

    st.divider()

    render_customer_review_nlp()
'''

if "render_customer_review_nlp()\n" not in code.split(
    "# ============================================\n# EXISTING BUSINESS\n"
)[-1]:

    code = code.replace(
        old,
        new,
        1
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Customer review NLP added")

✅ Customer review NLP added


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# 1) EXPAND KEYWORDS
# ============================================

code = code.replace(
'''        "جودة", "الخامة", "خامة",
''',
'''        "جودة", "جوده", "الخامة", "خامة",
'''
)

code = code.replace(
'''        "تأخر", "تأخير",
        "متأخر", "التوصيل",
        "توصيل", "delivery",
        "late"
''',
'''        "تأخر", "تأخير",
        "تاخير", "متأخر",
        "التوصيل", "توصيل",
        "أخذت وقت", "اخذت وقت",
        "بطء", "بطيء",
        "delivery", "late"
'''
)


# ============================================
# 2) ADD NEUTRAL %
# ============================================

old = '''    summary["Negative %"] = (
        summary["سلبي"]
        / summary["Total Mentions"]
        * 100
    )

    confidence = (
'''

new = '''    summary["Negative %"] = (
        summary["سلبي"]
        / summary["Total Mentions"]
        * 100
    )

    summary["Neutral %"] = (
        summary["محايد"]
        / summary["Total Mentions"]
        * 100
    )

    confidence = (
'''

code = code.replace(old, new)


# ============================================
# 3) SHOW NEUTRAL COLUMN
# ============================================

old = '''                    "Positive %",
                    "Negative %",
                    "Average Confidence"
'''

new = '''                    "Positive %",
                    "Neutral %",
                    "Negative %",
                    "Average Confidence"
'''

code = code.replace(old, new)


old = '''            display_summary[
                "Negative %"
            ] = display_summary[
                "Negative %"
            ].round(1)
'''

new = '''            display_summary[
                "Neutral %"
            ] = display_summary[
                "Neutral %"
            ].round(1)

            display_summary[
                "Negative %"
            ] = display_summary[
                "Negative %"
            ].round(1)
'''

code = code.replace(old, new)


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ NLP summary improved")

✅ NLP summary improved


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


hybrid_patch = r'''
# ============================================
# HYBRID ASPECT SENTIMENT
# ============================================

def normalize_review_text(text):

    text = str(text).lower().strip()

    replacements = {
        "أ": "ا",
        "إ": "ا",
        "آ": "ا",
        "ى": "ي"
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


def keyword_is_negated(text, keyword):

    text = normalize_review_text(text)
    keyword = normalize_review_text(keyword)

    negation_patterns = [
        f"مش {keyword}",
        f"مو {keyword}",
        f"ليس {keyword}",
        f"غير {keyword}"
    ]

    return any(
        pattern in text
        for pattern in negation_patterns
    )


def rule_based_aspect_sentiment(
    aspect,
    clause
):

    text = normalize_review_text(
        clause
    )


    # ========================================
    # DELAY / DELIVERY
    # ========================================

    if aspect == "التأخير":

        negative_terms = [
            "تاخر",
            "تاخير",
            "متاخر",
            "اخذت وقت",
            "اخد وقت",
            "طولت",
            "طول",
            "بطيء",
            "بطيئه",
            "بطء"
        ]

        positive_terms = [
            "سريع",
            "سريعه",
            "وصل بسرعه",
            "التوصيل ممتاز"
        ]

        if any(
            term in text
            for term in negative_terms
        ):
            return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    # ========================================
    # PRICE
    # ========================================

    elif aspect == "السعر":

        negative_terms = [
            "غالي",
            "غاليه",
            "مرتفع",
            "مرتفعه",
            "مبالغ",
            "مكلف"
        ]

        positive_terms = [
            "رخيص",
            "رخيصه",
            "سعر مناسب",
            "السعر مناسب",
            "سعر معقول",
            "السعر معقول"
        ]

        for term in negative_terms:

            if (
                term in text
                and not keyword_is_negated(
                    text,
                    term
                )
            ):
                return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    # ========================================
    # QUALITY
    # ========================================

    elif aspect == "الجودة":

        positive_terms = [
            "ممتاز",
            "ممتازه",
            "جيد",
            "جيده",
            "رائعه",
            "رائع",
            "مرتبه",
            "الخامة حلوه",
            "الخامه حلوه"
        ]

        negative_terms = [
            "سيئ",
            "سيئه",
            "رديء",
            "رديئه",
            "ضعيف",
            "ضعيفه",
            "الخامة سيئه",
            "الخامه سيئه"
        ]

        if any(
            term in text
            for term in negative_terms
        ):
            return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    # ========================================
    # SIZE
    # ========================================

    elif aspect == "المقاس":

        negative_terms = [
            "غير مناسب",
            "مش مناسب",
            "ضيق",
            "واسع",
            "المقاس غلط",
            "القياس غلط"
        ]

        positive_terms = [
            "مناسب",
            "مظبوط",
            "مضبوط",
            "المقاس ممتاز"
        ]

        if any(
            term in text
            for term in negative_terms
        ):
            return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    # ========================================
    # SERVICE
    # ========================================

    elif aspect == "الخدمة":

        negative_terms = [
            "سيئ",
            "سيئه",
            "بطيء",
            "بطيئه",
            "غير متعاون",
            "مش متعاون",
            "تعامل سيئ"
        ]

        positive_terms = [
            "ممتاز",
            "ممتازه",
            "متعاون",
            "متعاونين",
            "سريع",
            "سريعه",
            "تعامل رائع"
        ]

        if any(
            term in text
            for term in negative_terms
        ):
            return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    # ========================================
    # PACKAGING
    # ========================================

    elif aspect == "التغليف":

        negative_terms = [
            "سيئ",
            "سيئه",
            "ممزق",
            "تالف",
            "مخرب"
        ]

        positive_terms = [
            "جميل",
            "جميله",
            "مرتب",
            "مرتبه",
            "ممتاز",
            "ممتازه"
        ]

        if any(
            term in text
            for term in negative_terms
        ):
            return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    return None


# ============================================
# OVERRIDE ASPECT ANALYSIS
# ============================================

def analyze_aspect_sentiment(review):

    results = []

    for clause in split_review_into_clauses(
        review
    ):

        aspects = detect_review_aspects(
            clause
        )

        if not aspects:
            continue


        for aspect in aspects:

            rule_sentiment = (
                rule_based_aspect_sentiment(
                    aspect,
                    clause
                )
            )


            # --------------------------------
            # Clear rule
            # --------------------------------

            if rule_sentiment is not None:

                sentiment_label = (
                    rule_sentiment
                )

                confidence = np.nan

                method = "Rule-based"


            # --------------------------------
            # NLP fallback
            # --------------------------------

            else:

                model_result = (
                    analyze_review_sentiment(
                        clause
                    )
                )

                sentiment_label = (
                    model_result[
                        "Sentiment"
                    ]
                )

                confidence = (
                    model_result[
                        "Confidence"
                    ]
                )

                method = "NLP Model"


            results.append({
                "Review":
                    review,

                "Aspect":
                    aspect,

                "Context":
                    clause,

                "Aspect Sentiment":
                    sentiment_label,

                "Confidence":
                    confidence,

                "Method":
                    method
            })


    return results


# ============================================
# OVERRIDE SUMMARY
# ============================================

def build_streamlit_aspect_summary(
    reviews
):

    rows = []

    for review in reviews:

        rows.extend(
            analyze_aspect_sentiment(
                review
            )
        )


    if not rows:
        return pd.DataFrame()


    df = pd.DataFrame(
        rows
    )


    summary = (
        df
        .groupby(
            [
                "Aspect",
                "Aspect Sentiment"
            ]
        )
        .size()
        .unstack(fill_value=0)
    )


    for sentiment in [
        "إيجابي",
        "محايد",
        "سلبي"
    ]:

        if sentiment not in summary.columns:
            summary[sentiment] = 0


    summary = summary[
        [
            "إيجابي",
            "محايد",
            "سلبي"
        ]
    ]


    summary["Total Mentions"] = (
        summary[
            [
                "إيجابي",
                "محايد",
                "سلبي"
            ]
        ]
        .sum(axis=1)
    )


    summary["Positive %"] = (
        summary["إيجابي"]
        / summary["Total Mentions"]
        * 100
    )


    summary["Neutral %"] = (
        summary["محايد"]
        / summary["Total Mentions"]
        * 100
    )


    summary["Negative %"] = (
        summary["سلبي"]
        / summary["Total Mentions"]
        * 100
    )


    # ثقة النموذج فقط
    model_rows = df[
        df["Method"]
        == "NLP Model"
    ]

    if not model_rows.empty:

        model_confidence = (
            model_rows
            .groupby("Aspect")[
                "Confidence"
            ]
            .mean()
        )

        summary[
            "Average Model Confidence"
        ] = model_confidence

    else:

        summary[
            "Average Model Confidence"
        ] = np.nan


    # عدد الحالات التي حُسمت بالقواعد
    rule_counts = (
        df[
            df["Method"]
            == "Rule-based"
        ]
        .groupby("Aspect")
        .size()
    )

    summary[
        "Rule-based Mentions"
    ] = (
        rule_counts
        .reindex(
            summary.index,
            fill_value=0
        )
    )


    return (
        summary
        .sort_values(
            [
                "Negative %",
                "Total Mentions"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index()
    )
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

# نحط النسخة الجديدة بعد وظائف NLP القديمة
# حتى تعمل Override عليها
if "def rule_based_aspect_sentiment(" not in code:

    code = code.replace(
        marker,
        hybrid_patch
        + "\n\n"
        + marker
    )


# ============================================
# UPDATE DISPLAY COLUMNS
# ============================================

old_columns = '''                    "Positive %",
                    "Neutral %",
                    "Negative %",
                    "Average Confidence"
'''

new_columns = '''                    "Positive %",
                    "Neutral %",
                    "Negative %",
                    "Rule-based Mentions",
                    "Average Model Confidence"
'''

code = code.replace(
    old_columns,
    new_columns
)


# إزالة تقريب العمود القديم إذا كان موجودًا
code = code.replace(
'''            display_summary[
                "Average Confidence"
            ] = display_summary[
                "Average Confidence"
            ].round(2)
''',
'''            if "Average Model Confidence" in display_summary.columns:
                display_summary[
                    "Average Model Confidence"
                ] = display_summary[
                    "Average Model Confidence"
                ].round(2)
'''
)


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Hybrid NLP activated")

✅ Hybrid NLP activated


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")

# إضافة صيغة "أخذ وقت"
code = code.replace(
'''            "اخذت وقت",
            "اخد وقت",
''',
'''            "اخذت وقت",
            "اخذ وقت",
            "اخد وقت",
'''
)

# نخلي الثقة الفارغة تظهر بشكل أنظف
old = '''            if "Average Model Confidence" in display_summary.columns:
                display_summary[
                    "Average Model Confidence"
                ] = display_summary[
                    "Average Model Confidence"
                ].round(2)
'''

new = '''            if "Average Model Confidence" in display_summary.columns:

                display_summary[
                    "Average Model Confidence"
                ] = display_summary[
                    "Average Model Confidence"
                ].apply(
                    lambda x:
                    "—"
                    if pd.isna(x)
                    else round(float(x), 2)
                )
'''

code = code.replace(old, new)

path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Delay rule and confidence display fixed")

✅ Delay rule and confidence display fixed


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# INTEGRATED PERFORMANCE + NLP INSIGHTS
# ============================================

integrated_function = r'''
# ============================================
# INTEGRATED BUSINESS INSIGHTS
# ============================================

def build_integrated_existing_insights(
    base_insights,
    aspect_summary
):

    priority_alerts = list(
        base_insights.get(
            "warnings",
            []
        )
    )

    business_context = list(
        base_insights.get(
            "observations",
            []
        )
    )

    positive_signals = list(
        base_insights.get(
            "positives",
            []
        )
    )


    if (
        isinstance(
            aspect_summary,
            pd.DataFrame
        )
        and not aspect_summary.empty
    ):

        for _, row in aspect_summary.iterrows():

            aspect = row["Aspect"]

            mentions = int(
                row["Total Mentions"]
            )

            negative = float(
                row["Negative %"]
            )

            positive = float(
                row["Positive %"]
            )


            # -------------------------------
            # Repeated negative feedback
            # -------------------------------

            if (
                mentions >= 2
                and negative >= 60
            ):

                priority_alerts.append(
                    f"آراء العملاء حول «{aspect}» "
                    f"تظهر اتجاهًا سلبيًا "
                    f"({negative:.0f}% من "
                    f"{mentions} إشارات)."
                )


            # -------------------------------
            # Repeated positive feedback
            # -------------------------------

            if (
                mentions >= 2
                and positive >= 60
            ):

                positive_signals.append(
                    f"آراء العملاء حول «{aspect}» "
                    f"تميل إلى الإيجابية "
                    f"({positive:.0f}% من الإشارات)."
                )


    return {
        "Priority Alerts":
            list(
                dict.fromkeys(
                    priority_alerts
                )
            ),

        "Business Context":
            list(
                dict.fromkeys(
                    business_context
                )
            ),

        "Positive Signals":
            list(
                dict.fromkeys(
                    positive_signals
                )
            )
    }


def render_integrated_existing_insights():

    base = st.session_state.get(
        "existing_base_insights"
    )

    summary = st.session_state.get(
        "aspect_summary"
    )

    if not base:
        return

    if (
        not isinstance(
            summary,
            pd.DataFrame
        )
        or summary.empty
    ):
        return


    integrated = (
        build_integrated_existing_insights(
            base,
            summary
        )
    )


    st.divider()

    st.header(
        "🔗 قراءة متكاملة للأداء وآراء العملاء"
    )

    st.caption(
        "يتم دمج تغيرات الأداء الشهرية مع "
        "أنماط آراء العملاء المتكررة. "
        "حدود التنبيه في هذه النسخة التجريبية "
        "قابلة للتعديل وليست قواعد عالمية ثابتة."
    )


    col1, col2, col3 = st.columns(3)


    with col1:

        st.subheader(
            "🚨 تنبيهات ذات أولوية"
        )

        if integrated[
            "Priority Alerts"
        ]:

            for item in integrated[
                "Priority Alerts"
            ]:

                st.error(item)

        else:

            st.success(
                "لا توجد تنبيهات مرتفعة الأولوية."
            )


    with col2:

        st.subheader(
            "🔎 سياق الأداء"
        )

        if integrated[
            "Business Context"
        ]:

            for item in integrated[
                "Business Context"
            ]:

                st.info(item)

        else:

            st.write(
                "لا توجد ملاحظات سياقية إضافية."
            )


    with col3:

        st.subheader(
            "✅ إشارات إيجابية"
        )

        if integrated[
            "Positive Signals"
        ]:

            for item in integrated[
                "Positive Signals"
            ]:

                st.success(item)

        else:

            st.write(
                "لا توجد إشارات إيجابية إضافية."
            )
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def build_integrated_existing_insights(" not in code:

    code = code.replace(
        marker,
        integrated_function
        + "\n\n"
        + marker
    )


# ============================================
# SHOW AFTER NLP
# ============================================

old = '''    render_customer_review_nlp()
'''

new = '''    render_customer_review_nlp()

    render_integrated_existing_insights()
'''

if (
    "render_integrated_existing_insights()"
    not in code
):

    code = code.replace(
        old,
        new,
        1
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ NLP linked with business alerts")

✅ NLP linked with business alerts


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")

old = '''    render_customer_review_nlp()
'''

new = '''    render_customer_review_nlp()

    render_integrated_existing_insights()
'''

# نضيف الاستدعاء حتى لو كانت الدالة نفسها معرفة مسبقًا
if new not in code:
    code = code.replace(
        old,
        new,
        1
    )

path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Integrated insights display fixed")

✅ Integrated insights display fixed


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")

old = '''            st.session_state.aspect_summary = summary
'''

new = '''            st.session_state.aspect_summary = summary

            # ====================================
            # INTEGRATED PERFORMANCE + NLP
            # ====================================

            render_integrated_existing_insights()
'''

# نضيفه مرة واحدة فقط
if new not in code:
    code = code.replace(
        old,
        new,
        1
    )

path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Integrated insights attached directly to NLP summary")

✅ Integrated insights attached directly to NLP summary


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


force_function = r'''
# ============================================
# FINAL INTEGRATED EXISTING BUSINESS VIEW
# ============================================

def render_final_integrated_insights():

    history = st.session_state.get(
        "performance_history",
        []
    )

    summary = st.session_state.get(
        "aspect_summary"
    )

    # لازم يكون في شهرين
    if len(history) < 2:
        return

    # لازم يكون في تحليل آراء
    if (
        not isinstance(summary, pd.DataFrame)
        or summary.empty
    ):
        return


    performance = pd.DataFrame(
        history
    )

    previous = performance.iloc[-2]
    latest = performance.iloc[-1]


    # تحليل الأداء من آخر شهرين
    base = analyze_existing_changes(
        previous,
        latest
    )


    # دمج الأداء مع آراء العملاء
    integrated = (
        build_integrated_existing_insights(
            base,
            summary
        )
    )


    st.divider()

    st.header(
        "🔗 قراءة متكاملة للأداء وآراء العملاء"
    )

    st.caption(
        "يجمع هذا القسم بين تغير مؤشرات الأداء "
        "والأنماط المتكررة في آراء العملاء."
    )


    col1, col2, col3 = st.columns(3)


    # ========================================
    # ALERTS
    # ========================================

    with col1:

        st.subheader(
            "🚨 تنبيهات ذات أولوية"
        )

        alerts = integrated[
            "Priority Alerts"
        ]

        if alerts:

            for item in alerts:
                st.error(item)

        else:

            st.success(
                "لا توجد تنبيهات مرتفعة الأولوية."
            )


    # ========================================
    # CONTEXT
    # ========================================

    with col2:

        st.subheader(
            "🔎 سياق الأداء"
        )

        context = integrated[
            "Business Context"
        ]

        if context:

            for item in context:
                st.info(item)

        else:

            st.write(
                "لا توجد ملاحظات سياقية إضافية."
            )


    # ========================================
    # POSITIVE SIGNALS
    # ========================================

    with col3:

        st.subheader(
            "✅ إشارات إيجابية"
        )

        positives = integrated[
            "Positive Signals"
        ]

        if positives:

            for item in positives:
                st.success(item)

        else:

            st.write(
                "لا توجد إشارات إيجابية إضافية."
            )
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def render_final_integrated_insights():" not in code:

    code = code.replace(
        marker,
        force_function
        + "\n\n"
        + marker
    )


# ============================================
# ADD DIRECTLY AFTER ASPECT SUMMARY
# ============================================

target = '''            st.session_state.aspect_summary = summary
'''

replacement = '''            st.session_state.aspect_summary = summary

            render_final_integrated_insights()
'''

# شيل أي استدعاء قديم حتى ما يتكرر
code = code.replace(
    '''            render_integrated_existing_insights()
''',
    ""
)

if replacement not in code:
    code = code.replace(
        target,
        replacement,
        1
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Final integrated analysis fixed")

✅ Final integrated analysis fixed


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


groupby_code = r'''
# ============================================
# GROUPBY REVIEW INSIGHTS
# ============================================

def build_grouped_review_insights(reviews):

    rows = []

    for review in reviews:

        results = analyze_aspect_sentiment(
            review
        )

        rows.extend(results)


    if not rows:
        return pd.DataFrame()


    df = pd.DataFrame(rows)


    # ========================================
    # GROUP BY ASPECT
    # ========================================

    grouped = (
        df
        .groupby("Aspect")
        .agg(
            Total_Mentions=(
                "Aspect",
                "size"
            ),

            Negative_Mentions=(
                "Aspect Sentiment",
                lambda x:
                    (x == "سلبي").sum()
            ),

            Positive_Mentions=(
                "Aspect Sentiment",
                lambda x:
                    (x == "إيجابي").sum()
            ),

            Neutral_Mentions=(
                "Aspect Sentiment",
                lambda x:
                    (x == "محايد").sum()
            )
        )
        .reset_index()
    )


    grouped["Negative %"] = (
        grouped["Negative_Mentions"]
        / grouped["Total_Mentions"]
        * 100
    )


    grouped["Positive %"] = (
        grouped["Positive_Mentions"]
        / grouped["Total_Mentions"]
        * 100
    )


    grouped["Neutral %"] = (
        grouped["Neutral_Mentions"]
        / grouped["Total_Mentions"]
        * 100
    )


    return grouped


def render_grouped_review_alerts():

    reviews = st.session_state.get(
        "customer_reviews",
        []
    )

    if not reviews:
        return


    grouped = build_grouped_review_insights(
        reviews
    )

    if grouped.empty:
        return


    st.divider()

    st.header(
        "🔗 قراءة متكاملة لآراء العملاء"
    )


    alerts = []
    positives = []


    for _, row in grouped.iterrows():

        aspect = row["Aspect"]

        mentions = int(
            row["Total_Mentions"]
        )

        negative = float(
            row["Negative %"]
        )

        positive = float(
            row["Positive %"]
        )


        # ====================================
        # NEGATIVE REPEATED SIGNAL
        # ====================================

        if (
            mentions >= 2
            and negative >= 60
        ):

            alerts.append(
                f"تكررت ملاحظات سلبية حول «{aspect}» "
                f"في {mentions} إشارات، "
                f"بنسبة سلبية {negative:.0f}%."
            )


        # ====================================
        # POSITIVE REPEATED SIGNAL
        # ====================================

        if (
            mentions >= 2
            and positive >= 60
        ):

            positives.append(
                f"تكررت ملاحظات إيجابية حول «{aspect}» "
                f"في {mentions} إشارات، "
                f"بنسبة إيجابية {positive:.0f}%."
            )


    col1, col2 = st.columns(2)


    with col1:

        st.subheader(
            "🚨 تنبيهات متكررة"
        )

        if alerts:

            for item in alerts:
                st.error(item)

        else:

            st.success(
                "لا توجد أنماط سلبية متكررة "
                "تتجاوز حد التنبيه الحالي."
            )


    with col2:

        st.subheader(
            "✅ أنماط إيجابية"
        )

        if positives:

            for item in positives:
                st.success(item)

        else:

            st.write(
                "لا توجد أنماط إيجابية متكررة حاليًا."
            )
'''


# نحط الدوال قبل Session State
marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def build_grouped_review_insights(" not in code:

    code = code.replace(
        marker,
        groupby_code
        + "\n\n"
        + marker
    )


# ============================================
# DISPLAY DIRECTLY AFTER NLP SUMMARY
# ============================================

target = '''            st.session_state.aspect_summary = summary
'''

replacement = '''            st.session_state.aspect_summary = summary

            render_grouped_review_alerts()
'''

if replacement not in code:

    code = code.replace(
        target,
        replacement,
        1
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ GroupBy review alerts added")

✅ GroupBy review alerts added


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


recommendation_code = r'''
# ============================================
# EXISTING BUSINESS RECOMMENDATIONS
# ============================================

def build_existing_recommendations():

    recommendations = []
    investigation = []

    history = st.session_state.get(
        "performance_history",
        []
    )

    # ========================================
    # PERFORMANCE-BASED RECOMMENDATIONS
    # ========================================

    if len(history) >= 2:

        df = pd.DataFrame(history)

        previous = df.iloc[-2]
        latest = df.iloc[-1]

        revenue_change = safe_percent_change(
            latest["Revenue"],
            previous["Revenue"]
        )

        profit_change = safe_percent_change(
            latest["Profit"],
            previous["Profit"]
        )

        transaction_change = safe_percent_change(
            latest["Transactions"],
            previous["Transactions"]
        )

        customer_change = safe_percent_change(
            latest["Customers"],
            previous["Customers"]
        )

        complaint_change = safe_percent_change(
            latest["Complaint Rate %"],
            previous["Complaint Rate %"]
        )

        margin_diff = (
            float(latest["Profit Margin %"])
            - float(previous["Profit Margin %"])
        )


        # ------------------------------------
        # Revenue
        # ------------------------------------

        if (
            revenue_change is not None
            and revenue_change <= -10
        ):

            recommendations.append(
                "مراجعة أسباب انخفاض الإيرادات، "
                "مثل حجم الطلب، قنوات البيع، "
                "الأسعار والعروض التسويقية."
            )


        # ------------------------------------
        # Profit
        # ------------------------------------

        if (
            profit_change is not None
            and profit_change <= -10
        ):

            recommendations.append(
                "مراجعة هيكل التكاليف ومصادر الربح "
                "لتحديد البنود الأكثر تأثيرًا "
                "على انخفاض الربحية."
            )


        # ------------------------------------
        # Profit Margin
        # ------------------------------------

        if margin_diff <= -3:

            recommendations.append(
                "تحليل هامش الربح حسب المنتج أو الخدمة، "
                "ومراجعة التسعير والتكاليف المباشرة."
            )


        # ------------------------------------
        # Transactions
        # ------------------------------------

        if (
            transaction_change is not None
            and transaction_change <= -10
        ):

            recommendations.append(
                "فحص أسباب انخفاض عدد عمليات البيع، "
                "مع مراجعة سهولة الشراء، "
                "معدل التحويل وإعادة الشراء."
            )

            investigation.append(
                "تحقق مما إذا كان انخفاض العمليات "
                "متركزًا في منتج أو قناة بيع "
                "أو فترة زمنية محددة."
            )


        # ------------------------------------
        # Customers
        # ------------------------------------

        if (
            customer_change is not None
            and customer_change <= -5
        ):

            recommendations.append(
                "مراجعة اكتساب العملاء والاحتفاظ بهم، "
                "ومقارنة العملاء الجدد بالعملاء المتكررين."
            )


        # ------------------------------------
        # Complaints
        # ------------------------------------

        if (
            complaint_change is not None
            and complaint_change >= 10
        ):

            recommendations.append(
                "تصنيف الشكاوى حسب الموضوع "
                "وتحديد المشكلات الأكثر تكرارًا "
                "قبل اتخاذ إجراء تصحيحي."
            )

            investigation.append(
                "قارن ارتفاع الشكاوى مع نتائج "
                "تحليل آراء العملاء، دون افتراض "
                "أن أحدهما سبب مباشر للآخر."
            )


    # ========================================
    # REVIEW-BASED RECOMMENDATIONS
    # ========================================

    reviews = st.session_state.get(
        "customer_reviews",
        []
    )

    if reviews:

        grouped = build_grouped_review_insights(
            reviews
        )

        if not grouped.empty:

            for _, row in grouped.iterrows():

                aspect = row["Aspect"]

                mentions = int(
                    row["Total_Mentions"]
                )

                negative = float(
                    row["Negative %"]
                )


                # لا نعتبر التعليق الواحد نمطًا متكررًا
                if not (
                    mentions >= 2
                    and negative >= 60
                ):
                    continue


                # --------------------------------
                # Delay
                # --------------------------------

                if aspect == "التأخير":

                    recommendations.append(
                        "مراجعة مسار تنفيذ الطلب والتوصيل، "
                        "من وقت استلام الطلب حتى التسليم، "
                        "وتحديد المرحلة التي يحدث فيها التأخير."
                    )

                    investigation.append(
                        "تحقق مما إذا كانت ملاحظات التأخير "
                        "مرتبطة بأوقات أو مناطق أو "
                        "قنوات توصيل محددة."
                    )


                # --------------------------------
                # Price
                # --------------------------------

                elif aspect == "السعر":

                    recommendations.append(
                        "مراجعة تصور العملاء للقيمة مقابل السعر، "
                        "مع اختبار وضوح القيمة المقدمة "
                        "قبل اتخاذ قرار بتغيير الأسعار."
                    )

                    investigation.append(
                        "تحقق مما إذا كانت ملاحظات السعر "
                        "مرتبطة بمنتجات محددة أو بفئة معينة "
                        "من العملاء."
                    )


                # --------------------------------
                # Quality
                # --------------------------------

                elif aspect == "الجودة":

                    recommendations.append(
                        "مراجعة خطوات ضبط الجودة "
                        "وتحديد أكثر المنتجات أو الخدمات "
                        "ارتباطًا بالملاحظات السلبية."
                    )


                # --------------------------------
                # Size
                # --------------------------------

                elif aspect == "المقاس":

                    recommendations.append(
                        "مراجعة دليل المقاسات ووضوح معلومات القياس، "
                        "ومتابعة حالات الاستبدال المرتبطة بالمقاس."
                    )


                # --------------------------------
                # Service
                # --------------------------------

                elif aspect == "الخدمة":

                    recommendations.append(
                        "مراجعة نقاط التواصل مع العميل "
                        "وزمن الاستجابة وجودة التعامل "
                        "في مراحل الخدمة المختلفة."
                    )


                # --------------------------------
                # Packaging
                # --------------------------------

                elif aspect == "التغليف":

                    recommendations.append(
                        "مراجعة طريقة التغليف "
                        "والتحقق من المشكلات المتكررة "
                        "قبل تعديل مواد أو إجراءات التغليف."
                    )


    return {
        "Recommendations":
            list(
                dict.fromkeys(
                    recommendations
                )
            ),

        "Needs Investigation":
            list(
                dict.fromkeys(
                    investigation
                )
            )
    }


def render_existing_recommendations():

    result = build_existing_recommendations()

    recommendations = result[
        "Recommendations"
    ]

    investigation = result[
        "Needs Investigation"
    ]

    st.divider()

    st.header(
        "💡 توصيات للتحسين"
    )

    st.caption(
        "هذه التوصيات ناتجة عن مؤشرات الأداء "
        "والأنماط المتكررة في آراء العملاء. "
        "هي نقاط دعم للقرار ولا تثبت سبب المشكلة بشكل مباشر."
    )

    col1, col2 = st.columns(2)


    with col1:

        st.subheader(
            "🛠️ إجراءات مقترحة"
        )

        if recommendations:

            for item in recommendations:
                st.success(item)

        else:

            st.info(
                "لا توجد توصيات تلقائية إضافية "
                "ضمن المؤشرات الحالية."
            )


    with col2:

        st.subheader(
            "🔎 أمور تحتاج تحقق"
        )

        if investigation:

            for item in investigation:
                st.warning(item)

        else:

            st.write(
                "لا توجد نقاط تحقق إضافية حاليًا."
            )
'''


# ============================================
# ADD FUNCTIONS BEFORE SESSION STATE
# ============================================

marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if "def build_existing_recommendations():" not in code:

    code = code.replace(
        marker,
        recommendation_code
        + "\n\n"
        + marker
    )


# ============================================
# DISPLAY AFTER CUSTOMER REVIEW SECTION
# ============================================

target = "\n    render_customer_review_nlp()\n"

call = "\n    render_existing_recommendations()\n"

if call not in code:

    position = code.rfind(target)

    if position != -1:

        insert_at = (
            position
            + len(target)
        )

        code = (
            code[:insert_at]
            + "\n    render_existing_recommendations()\n"
            + code[insert_at:]
        )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ Existing business recommendations added")

✅ Existing business recommendations added


In [ ]:
from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================
# 1) ADD NEW ASPECTS
# ============================================

insert_after = '''    "التغليف": [
        "تغليف", "التغليف",
        "باكج", "package",
        "packaging"
    ]
'''

replacement = '''    "التغليف": [
        "تغليف", "التغليف",
        "باكج", "package",
        "packaging"
    ],

    "التجربة الرقمية": [
        "التطبيق",
        "تطبيق",
        "استخدام التطبيق",
        "واجهة التطبيق",
        "سهولة الاستخدام",
        "app",
        "application"
    ],

    "تنوع المنتجات": [
        "تشكيلة",
        "تنوع",
        "خيارات",
        "موديلات",
        "قطع",
        "منتجات",
        "اصناف",
        "أصناف",
        "variety",
        "collection"
    ],

    "العلامات التجارية": [
        "ماركات",
        "الماركات",
        "براند",
        "براندات",
        "علامات تجارية",
        "brand",
        "brands"
    ]
'''

if '"التجربة الرقمية"' not in code:
    code = code.replace(
        insert_after,
        replacement,
        1
    )


# ============================================
# 2) ADD HYBRID RULES FOR NEW ASPECTS
# ============================================

target = '''    return None


# ============================================
# OVERRIDE ASPECT ANALYSIS
# ============================================
'''

new_rules = '''    # ========================================
    # DIGITAL EXPERIENCE
    # ========================================

    elif aspect == "التجربة الرقمية":

        negative_terms = [
            "صعب",
            "صعبه",
            "معقد",
            "معقده",
            "بطيء",
            "بطيئ",
            "يعلق",
            "تعليق",
            "ما بشتغل",
            "مش شغال",
            "لا يعمل"
        ]

        positive_terms = [
            "سهل",
            "سهله",
            "سلس",
            "سلسه",
            "واضح",
            "واضحه",
            "سريع",
            "سريعه",
            "سهولة الاستخدام"
        ]

        if any(
            term in text
            for term in negative_terms
        ):
            return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    # ========================================
    # PRODUCT VARIETY
    # ========================================

    elif aspect == "تنوع المنتجات":

        negative_terms = [
            "محدود",
            "محدوده",
            "قليل",
            "قليله",
            "ما في خيارات",
            "خيارات قليلة",
            "نفس القطع",
            "نفس المنتجات"
        ]

        positive_terms = [
            "متنوع",
            "متنوعه",
            "تنوع",
            "فريد",
            "فريده",
            "قطع فريدة",
            "خيارات كثيرة",
            "تشكيلة كبيرة",
            "تشكيله كبيره"
        ]

        if any(
            term in text
            for term in negative_terms
        ):
            return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    # ========================================
    # BRANDS
    # ========================================

    elif aspect == "العلامات التجارية":

        negative_terms = [
            "تقليد",
            "مقلد",
            "مقلده",
            "غير اصلي",
            "غير اصليه",
            "مش اصلي",
            "مش اصليه"
        ]

        positive_terms = [
            "عالميه",
            "عالمية",
            "معروفه",
            "معروفة",
            "اصليه",
            "أصلية",
            "اصلية",
            "ماركات عالمية"
        ]

        if any(
            term in text
            for term in negative_terms
        ):
            return "سلبي"

        if any(
            term in text
            for term in positive_terms
        ):
            return "إيجابي"


    return None


# ============================================
# OVERRIDE ASPECT ANALYSIS
# ============================================
'''

if 'elif aspect == "التجربة الرقمية":' not in code:
    code = code.replace(
        target,
        new_rules,
        1
    )


# ============================================
# 3) ADD RECOMMENDATIONS FOR NEW ASPECTS
# ============================================

recommendation_target = '''                elif aspect == "التغليف":

                    recommendations.append(
                        "مراجعة طريقة التغليف "
                        "والتحقق من المشكلات المتكررة "
                        "قبل تعديل مواد أو إجراءات التغليف."
                    )
'''

recommendation_new = '''                elif aspect == "التغليف":

                    recommendations.append(
                        "مراجعة طريقة التغليف "
                        "والتحقق من المشكلات المتكررة "
                        "قبل تعديل مواد أو إجراءات التغليف."
                    )


                # --------------------------------
                # Digital Experience
                # --------------------------------

                elif aspect == "التجربة الرقمية":

                    recommendations.append(
                        "مراجعة خطوات استخدام التطبيق "
                        "وتحديد النقاط التي تسبب صعوبة "
                        "أو بطئًا للمستخدم."
                    )

                    investigation.append(
                        "تحقق مما إذا كانت المشكلة مرتبطة "
                        "بصفحة أو جهاز أو خطوة محددة "
                        "داخل التطبيق."
                    )


                # --------------------------------
                # Product Variety
                # --------------------------------

                elif aspect == "تنوع المنتجات":

                    recommendations.append(
                        "مراجعة تنوع التشكيلة والمنتجات "
                        "ومقارنة الخيارات المتاحة "
                        "مع أكثر ما يطلبه العملاء."
                    )

                    investigation.append(
                        "حدد المنتجات أو الفئات التي يطلب "
                        "العملاء تنوعًا أكبر فيها قبل "
                        "زيادة المخزون."
                    )


                # --------------------------------
                # Brands
                # --------------------------------

                elif aspect == "العلامات التجارية":

                    recommendations.append(
                        "مراجعة العلامات التجارية التي يهتم "
                        "بها العملاء وتحديد الأكثر ارتباطًا "
                        "بالطلب الفعلي."
                    )

                    investigation.append(
                        "تحقق من تفضيلات العملاء للعلامات "
                        "التجارية قبل توسيع التشكيلة."
                    )
'''

if 'elif aspect == "التجربة الرقمية":' not in code.split(
    "def build_existing_recommendations"
)[-1]:
    code = code.replace(
        recommendation_target,
        recommendation_new,
        1
    )


path.write_text(
    code,
    encoding="utf-8"
)

print("✅ NLP aspects expanded successfully")

✅ NLP aspects expanded successfully


In [ ]:
from pathlib import Path
import re

path = Path("app.py")
code = path.read_text(encoding="utf-8")


# ============================================================
# 1) REQUIRED IMPORTS
# ============================================================

extra_imports = '''
import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
'''

if "from sentence_transformers import SentenceTransformer" not in code:

    code = code.replace(
        "import streamlit as st\n",
        "import streamlit as st\n" + extra_imports + "\n",
        1
    )


# ============================================================
# 2) UNIVERSAL ACTIVITY SEARCH + LOCATION MODEL
# ============================================================

universal_code = r'''
# ============================================================
# UNIVERSAL NEW-BUSINESS ENGINE
# ============================================================

def get_business_family(isic_code):

    code = normalize_isic4(isic_code)

    if not code or not code[:2].isdigit():
        return "other"

    division = int(code[:2])

    if 1 <= division <= 3:
        return "agriculture"

    elif 5 <= division <= 9:
        return "mining"

    elif 10 <= division <= 33:
        return "manufacturing"

    elif 35 <= division <= 39:
        return "utilities_environment"

    elif 41 <= division <= 43:
        return "construction"

    elif 45 <= division <= 47:
        return "retail_trade"

    elif 49 <= division <= 53:
        return "transport_logistics"

    elif 55 <= division <= 56:
        return "hospitality_food"

    elif 58 <= division <= 63:
        return "information_technology"

    elif 64 <= division <= 66:
        return "financial_services"

    elif division == 68:
        return "real_estate"

    elif 69 <= division <= 75:
        return "professional_services"

    elif 77 <= division <= 82:
        return "business_support"

    elif division == 85:
        return "education"

    elif 86 <= division <= 88:
        return "health_social"

    elif 90 <= division <= 93:
        return "entertainment"

    elif 94 <= division <= 96:
        return "personal_services"

    return "other"


business_family_labels = {

    "التجارة والبيع":
        "retail_trade",

    "المطاعم والطعام والضيافة":
        "hospitality_food",

    "التصنيع":
        "manufacturing",

    "العقارات":
        "real_estate",

    "تكنولوجيا المعلومات":
        "information_technology",

    "الخدمات المهنية":
        "professional_services",

    "خدمات الأعمال":
        "business_support",

    "التعليم":
        "education",

    "الصحة والخدمات الاجتماعية":
        "health_social",

    "الخدمات الشخصية":
        "personal_services",

    "الترفيه":
        "entertainment",

    "النقل والخدمات اللوجستية":
        "transport_logistics",

    "الإنشاءات":
        "construction",

    "الخدمات المالية":
        "financial_services",

    "الزراعة":
        "agriculture",

    "التعدين":
        "mining",

    "المياه والطاقة والبيئة":
        "utilities_environment"
}


# ============================================================
# ACTIVITY CATALOG
# ============================================================

activity_catalog = (
    activities[
        [
            "ISIC Code",
            "Economic Activity"
        ]
    ]
    .dropna()
    .drop_duplicates(
        subset=["ISIC Code"]
    )
    .copy()
)

activity_catalog[
    "Business Family"
] = activity_catalog[
    "ISIC Code"
].apply(
    get_business_family
)

activity_catalog[
    "Search Text"
] = activity_catalog[
    "Economic Activity"
].astype(str)


# ============================================================
# AI SEARCH MODELS
# ============================================================

@st.cache_resource(show_spinner=False)
def load_activity_search_models():

    semantic_model = SentenceTransformer(
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    )

    translation_name = (
        "Helsinki-NLP/opus-mt-ar-en"
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            translation_name
        )
    )

    translation_model = (
        AutoModelForSeq2SeqLM
        .from_pretrained(
            translation_name
        )
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    translation_model = (
        translation_model.to(device)
    )

    return (
        semantic_model,
        tokenizer,
        translation_model,
        device
    )


def translate_activity_query(text):

    (
        semantic_model,
        tokenizer,
        translation_model,
        device
    ) = load_activity_search_models()

    text = str(text).strip()

    if not text:
        return ""

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        generated = (
            translation_model.generate(
                **inputs,
                max_length=128
            )
        )

    return tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )


def search_activities_streamlit(
    business_family,
    search_text,
    top_k=5
):

    search_text = str(
        search_text
    ).strip()

    if not search_text:
        return pd.DataFrame()


    (
        semantic_model,
        _,
        _,
        _
    ) = load_activity_search_models()


    translated_text = (
        translate_activity_query(
            search_text
        )
    )


    candidates = activity_catalog[
        activity_catalog[
            "Business Family"
        ] == business_family
    ].copy()


    if candidates.empty:
        return pd.DataFrame()


    documents = (
        candidates[
            "Search Text"
        ]
        .fillna("")
        .astype(str)
        .tolist()
    )


    document_vectors = (
        semantic_model.encode(
            documents,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
    )


    arabic_vector = (
        semantic_model.encode(
            search_text,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
    )


    english_vector = (
        semantic_model.encode(
            translated_text,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
    )


    arabic_semantic = (
        util.cos_sim(
            arabic_vector,
            document_vectors
        )[0]
        .cpu()
        .numpy()
    )


    english_semantic = (
        util.cos_sim(
            english_vector,
            document_vectors
        )[0]
        .cpu()
        .numpy()
    )


    char_vectorizer = (
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            lowercase=True
        )
    )


    char_matrix = (
        char_vectorizer.fit_transform(
            [translated_text]
            + documents
        )
    )


    char_scores = (
        cosine_similarity(
            char_matrix[0:1],
            char_matrix[1:]
        )[0]
    )


    candidates[
        "Arabic Semantic"
    ] = arabic_semantic

    candidates[
        "English Semantic"
    ] = english_semantic

    candidates[
        "Text Match"
    ] = char_scores


    # Prototype hybrid weights
    candidates[
        "Final Search Score"
    ] = (
        0.40
        * candidates[
            "Arabic Semantic"
        ]

        + 0.30
        * candidates[
            "English Semantic"
        ]

        + 0.30
        * candidates[
            "Text Match"
        ]
    )


    return (
        candidates
        .sort_values(
            "Final Search Score",
            ascending=False
        )
        .head(top_k)[
            [
                "ISIC Code",
                "Economic Activity",
                "Business Family",
                "Final Search Score"
            ]
        ]
        .reset_index(drop=True)
    )


# ============================================================
# DEMAND MATCHING
# ============================================================

consumer_families = [
    "retail_trade",
    "hospitality_food",
    "personal_services",
    "education",
    "health_social",
    "entertainment"
]


def match_demand_for_activity(
    activity_name,
    business_family
):

    if business_family not in consumer_families:
        return None


    (
        semantic_model,
        _,
        _,
        _
    ) = load_activity_search_models()


    categories = (
        demand["Category (EN)"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )


    if not categories:
        return None


    query_vector = (
        semantic_model.encode(
            str(activity_name),
            convert_to_tensor=True,
            normalize_embeddings=True
        )
    )


    demand_vectors = (
        semantic_model.encode(
            categories,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
    )


    scores = util.cos_sim(
        query_vector,
        demand_vectors
    )[0]


    best_index = int(
        scores.argmax()
    )

    similarity = float(
        scores[best_index]
    )


    # Prototype confidence threshold
    if similarity < 0.45:
        return None


    return categories[
        best_index
    ]


# ============================================================
# LOCATION STRATEGY
# ============================================================

location_strategy = {

    "manufacturing":
        "cluster",

    "construction":
        "cluster",

    "transport_logistics":
        "cluster",

    "real_estate":
        "ambiguous_market",

    "retail_trade":
        "competition",

    "hospitality_food":
        "competition",

    "personal_services":
        "competition",

    "professional_services":
        "competition",

    "business_support":
        "competition",

    "education":
        "competition",

    "entertainment":
        "competition",

    "information_technology":
        "competition",

    "financial_services":
        "competition",

    "health_social":
        "competition",

    "agriculture":
        "resource_based",

    "mining":
        "resource_based",

    "utilities_environment":
        "resource_based",

    "other":
        "competition"
}


# ============================================================
# OVERRIDE LOCATION MODEL
# ============================================================

def calculate_location_scores(
    isic_code,
    demand_category=None,
    business_family=None
):

    if business_family is None:

        business_family = (
            get_business_family(
                isic_code
            )
        )


    business = activities[
        activities[
            "ISIC Code"
        ] == str(isic_code)
    ]


    if business.empty:

        raise ValueError(
            "لم يتم العثور على النشاط "
            "في بيانات المنشآت."
        )


    business = business.iloc[0]


    strategy = (
        location_strategy.get(
            business_family,
            "competition"
        )
    )


    demand_row = None

    if demand_category:

        match = demand[
            demand[
                "Category (EN)"
            ] == demand_category
        ]

        if not match.empty:
            demand_row = (
                match.iloc[0]
            )


    rows = []


    for (
        arabic_name,
        english_name
    ) in governorate_map.items():


        gov_match = governorates[
            governorates[
                "Governorate"
            ] == english_name
        ]


        if gov_match.empty:
            continue


        gov_row = (
            gov_match.iloc[0]
        )


        population = float(
            gov_row[
                "Population 2025"
            ]
        )


        registration_growth = float(
            gov_row[
                "H1 Growth 2023→2024"
            ]
        )


        similar_businesses = (
            pd.to_numeric(
                business.get(
                    english_name,
                    0
                ),
                errors="coerce"
            )
        )


        if pd.isna(
            similar_businesses
        ):
            similar_businesses = 0


        similar_businesses = float(
            similar_businesses
        )


        businesses_per_10k = (
            similar_businesses
            / population
            * 10000
        )


        if demand_row is not None:

            historical_demand = (
                pd.to_numeric(
                    demand_row.get(
                        arabic_name,
                        np.nan
                    ),
                    errors="coerce"
                )
            )

        else:

            historical_demand = (
                np.nan
            )


        rows.append({

            "Governorate":
                english_name,

            "Population":
                population,

            "Historical Demand":
                historical_demand,

            "Similar Businesses":
                similar_businesses,

            "Businesses per 10k":
                businesses_per_10k,

            "Registration Growth":
                registration_growth
        })


    result = pd.DataFrame(
        rows
    )


    # ----------------------------------------
    # Market
    # ----------------------------------------

    result[
        "Log Population"
    ] = np.log1p(
        result[
            "Population"
        ]
    )


    result[
        "Market Score"
    ] = normalize(
        result[
            "Log Population"
        ]
    )


    # ----------------------------------------
    # Historical demand
    # ----------------------------------------

    if result[
        "Historical Demand"
    ].notna().any():

        result[
            "Demand Score"
        ] = normalize(
            result[
                "Historical Demand"
            ]
        )

    else:

        result[
            "Demand Score"
        ] = np.nan


    # ----------------------------------------
    # Similar-business indicator
    # ----------------------------------------

    result[
        "Sector Activity Score"
    ] = normalize(
        result[
            "Businesses per 10k"
        ]
    )


    if strategy == "cluster":

        result[
            "Sector Score"
        ] = result[
            "Sector Activity Score"
        ]


    elif strategy == "competition":

        result[
            "Sector Score"
        ] = normalize(
            result[
                "Businesses per 10k"
            ],
            reverse=True
        )


    else:

        # Real estate / resource-based:
        # keep for context only.
        result[
            "Sector Score"
        ] = np.nan


    # ----------------------------------------
    # Trend
    # ----------------------------------------

    result[
        "Trend Score"
    ] = normalize(
        result[
            "Registration Growth"
        ]
    )


    # ----------------------------------------
    # Final supported components only
    # ----------------------------------------

    components = [
        "Market Score",
        "Trend Score"
    ]


    if result[
        "Demand Score"
    ].notna().any():

        components.append(
            "Demand Score"
        )


    if (
        strategy
        in [
            "cluster",
            "competition"
        ]
        and result[
            "Sector Score"
        ].notna().any()
    ):

        components.append(
            "Sector Score"
        )


    result[
        "Location Suitability Score"
    ] = (
        result[
            components
        ].mean(
            axis=1
        )
    )


    result = (
        result
        .sort_values(
            "Location Suitability Score",
            ascending=False
        )
        .reset_index(drop=True)
    )


    result["Rank"] = (
        result.index + 1
    )

    result["Strategy"] = (
        strategy
    )

    result[
        "Components Used"
    ] = ", ".join(
        components
    )


    return (
        result,
        components
    )
'''


marker = """
# ============================================
# SESSION STATE
# ============================================
"""

if (
    "def search_activities_streamlit("
    not in code
):

    code = code.replace(
        marker,
        universal_code
        + "\n\n"
        + marker,
        1
    )


# ============================================================
# 3) ADD ACTIVITY SEARCH UI
# ============================================================

old_heading = '''    st.subheader("بيانات المشروع")

    governorate = st.selectbox(
'''

new_heading = '''    st.subheader("بيانات المشروع")

    # ========================================
    # ACTIVITY SEARCH
    # ========================================

    sector_label = st.selectbox(
        "القطاع الرئيسي",
        list(
            business_family_labels.keys()
        )
    )

    selected_family = (
        business_family_labels[
            sector_label
        ]
    )

    activity_query = st.text_input(
        "اكتب وصف المشروع أو النشاط",
        placeholder=(
            "مثال: محل ملابس، شركة برمجيات، "
            "مكتب عقاري، مركز تدريب..."
        )
    )

    if st.button(
        "بحث عن النشاط 🔎",
        use_container_width=True,
        key="activity_search_button"
    ):

        if not activity_query.strip():

            st.warning(
                "اكتب وصف النشاط أولًا."
            )

        else:

            with st.spinner(
                "جاري البحث عن الأنشطة الأقرب..."
            ):

                results = (
                    search_activities_streamlit(
                        selected_family,
                        activity_query,
                        top_k=5
                    )
                )

                st.session_state[
                    "activity_search_results"
                ] = results


    activity_results = (
        st.session_state.get(
            "activity_search_results",
            pd.DataFrame()
        )
    )


    selected_activity = None


    if (
        isinstance(
            activity_results,
            pd.DataFrame
        )
        and not activity_results.empty
    ):

        st.markdown(
            "#### 🔎 أفضل الأنشطة المقترحة"
        )

        show_results = (
            activity_results[
                [
                    "ISIC Code",
                    "Economic Activity",
                    "Final Search Score"
                ]
            ]
            .copy()
        )

        show_results[
            "Final Search Score"
        ] = show_results[
            "Final Search Score"
        ].round(3)

        st.dataframe(
            show_results,
            use_container_width=True,
            hide_index=True
        )


        activity_options = {}

        for _, row in (
            activity_results
            .iterrows()
        ):

            label = (
                f"{row['ISIC Code']} - "
                f"{row['Economic Activity']}"
            )

            activity_options[
                label
            ] = row[
                "ISIC Code"
            ]


        selected_label = (
            st.selectbox(
                "اختر النشاط الأنسب",
                list(
                    activity_options.keys()
                )
            )
        )


        selected_code = (
            activity_options[
                selected_label
            ]
        )


        selected_row = (
            activity_results[
                activity_results[
                    "ISIC Code"
                ] == selected_code
            ]
            .iloc[0]
        )


        selected_activity = {

            "ISIC Code":
                str(
                    selected_row[
                        "ISIC Code"
                    ]
                ),

            "Economic Activity":
                selected_row[
                    "Economic Activity"
                ],

            "Business Family":
                selected_row[
                    "Business Family"
                ]
        }


        st.success(
            "✅ تم اختيار النشاط: "
            + str(
                selected_activity[
                    "Economic Activity"
                ]
            )
        )


    governorate = st.selectbox(
'''

if old_heading in code:

    code = code.replace(
        old_heading,
        new_heading,
        1
    )


# ============================================================
# 4) REMOVE HARDCODED CLOTHING PROJECT
# ============================================================

old_project = '''        st.session_state.new_project = {
            "ISIC Code": "4771",
            "Economic Activity":
                "Retail sale of clothing, footwear and leather articles in specialized stores",
            "Business Family": "retail_trade",
            "Preferred Governorate": governorate,
            "Can Relocate": can_relocate == "نعم",
            "Operating Model": operating_model,
            "Available Capital":
                available_capital if available_capital > 0 else None,
            "Plan": plan
        }
'''

new_project = '''        if selected_activity is None:

            st.warning(
                "ابحث عن النشاط واختر النشاط "
                "الأنسب أولًا."
            )

        else:

            st.session_state.new_project = {

                "ISIC Code":
                    selected_activity[
                        "ISIC Code"
                    ],

                "Economic Activity":
                    selected_activity[
                        "Economic Activity"
                    ],

                "Business Family":
                    selected_activity[
                        "Business Family"
                    ],

                "Preferred Governorate":
                    governorate,

                "Can Relocate":
                    can_relocate == "نعم",

                "Operating Model":
                    operating_model,

                "Available Capital":
                    available_capital
                    if available_capital > 0
                    else None,

                "Plan":
                    plan
            }
'''

if old_project in code:

    code = code.replace(
        old_project,
        new_project,
        1
    )


# ============================================================
# 5) DYNAMIC DEMAND CATEGORY
# ============================================================

code = code.replace(
'''            demand_category = (
                find_clothing_demand_category()
            )
''',
'''            demand_category = (
                match_demand_for_activity(
                    project[
                        "Economic Activity"
                    ],
                    project[
                        "Business Family"
                    ]
                )
            )
''',
1
)


# ============================================================
# 6) DYNAMIC ISIC LOCATION ANALYSIS
# ============================================================

code = code.replace(
'''                calculate_location_scores(
                    "4771",
                    demand_category
                )
''',
'''                calculate_location_scores(
                    project[
                        "ISIC Code"
                    ],
                    demand_category,
                    project[
                        "Business Family"
                    ]
                )
''',
1
)


# ============================================================
# 7) DYNAMIC LOCATION COMPONENT DISPLAY
# ============================================================

old_components = '''                component_cols = [
                    "Market Score",
                    "Trend Score",
                    "Sector Score"
                ]

                if (
                    "Demand Score" in selected.index
                    and pd.notna(
                        selected["Demand Score"]
                    )
                ):
                    component_cols.append(
                        "Demand Score"
                    )
'''

new_components = '''                component_cols = [
                    col
                    for col in components
                    if (
                        col in selected.index
                        and pd.notna(
                            selected[col]
                        )
                    )
                ]
'''

if old_components in code:

    code = code.replace(
        old_components,
        new_components,
        1
    )


# ============================================================
# 8) GENERAL DEMAND NOTE
# ============================================================

old_note = '''                    st.info(
                        "مؤشر الطلب المستخدم يعتمد على "
                        "الإنفاق الأسري التاريخي المرتبط "
                        "بالملابس والأحذية من HEIS 2017، "
                        "ولا يمثل الطلب الحالي بشكل مباشر."
                    )
'''

new_note = '''                    st.info(
                        f"مؤشر الطلب المستخدم يعتمد على "
                        f"فئة الإنفاق الأسري التاريخية "
                        f"«{demand_category}» من HEIS 2017، "
                        f"ولا يمثل الطلب الحالي بشكل مباشر."
                    )
'''

if old_note in code:

    code = code.replace(
        old_note,
        new_note,
        1
    )


# ============================================================
# SAVE
# ============================================================

path.write_text(
    code,
    encoding="utf-8"
)

print("✅ New-business activity selection is now universal")
print("✅ Hardcoded clothing activity removed")
print("✅ Location analysis now follows the selected ISIC activity")

✅ New-business activity selection is now universal
✅ Hardcoded clothing activity removed
✅ Location analysis now follows the selected ISIC activity


In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=900
)

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("/content/app.py")

In [ ]:
import subprocess
import time
import requests

log_file = open("/content/bunya_8502.log", "w")

bunya_process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app_BUNYA.py",
        "--server.port=8502",
        "--server.address=127.0.0.1",
        "--server.headless=true"
    ],
    cwd="/content",
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)

try:
    r = requests.get("http://127.0.0.1:8502/_stcore/health", timeout=10)
    print("بُنْيَة الجديدة:", r.status_code, r.text)
except Exception as e:
    print("في مشكلة:", e)

في مشكلة: HTTPConnectionPool(host='127.0.0.1', port=8502): Max retries exceeded with url: /_stcore/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7afa26838e90>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
import subprocess, time, re

# تشغيل Cloudflare على النسخة الجديدة فقط
cf = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8502"
    ],
    stdout=open("/content/cloudflare_bunya_8502.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(8)

log = open("/content/cloudflare_bunya_8502.log", "r").read()

match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)

if match:
    print("رابط بُنْيَة الجديدة:")
    print(match.group(0))
else:
    print("لسا الرابط ما ظهر، شغلي الخلية مرة ثانية بعد شوي.")

رابط بُنْيَة الجديدة:
https://tumor-indicators-merely-casting.trycloudflare.com


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving bunya_logo.png to bunya_logo (1).png
Saving app_BUNYA_v2.py to app_BUNYA_v2 (2).py
Saving SME_AI_Master_Data_v4_Location_Scoring.xlsx to SME_AI_Master_Data_v4_Location_Scoring (2).xlsx


In [ ]:
import os

check_files = [
    "/content/app_BUNYA_v2 (2).py",
    "/content/bunya_logo.png",
    "/content/SME_AI_Master_Data_v4_Location_Scoring.xlsx"
]

for f in check_files:
    print("موجود" if os.path.exists(f) else "مش موجود", ":", f)

موجود : /content/app_BUNYA_v2 (2).py
موجود : /content/bunya_logo.png
موجود : /content/SME_AI_Master_Data_v4_Location_Scoring.xlsx


In [ ]:
import subprocess
import time
import requests

log_file = open("/content/bunya_v2_8503.log", "w")

bunya_v2_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/app_BUNYA_v2 (2).py",
        "--server.port=8503",
        "--server.address=127.0.0.1",
        "--server.headless=true"
    ],
    cwd="/content",
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)

try:
    r = requests.get(
        "http://127.0.0.1:8503/_stcore/health",
        timeout=10
    )
    print("بُنْيَة v2:", r.status_code, r.text)
except Exception as e:
    print("في مشكلة:", e)

بُنْيَة v2: 200 ok


In [ ]:
import subprocess
import time
import re
import os

cloudflared_path = "/content/cloudflared"

log_path = "/content/cloudflare_bunya_v2_8503.log"

cf_v2 = subprocess.Popen(
    [
        cloudflared_path,
        "tunnel",
        "--url",
        "http://127.0.0.1:8503"
    ],
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT
)

time.sleep(8)

log = open(log_path, "r").read()

match = re.search(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
    log
)

if match:
    print("رابط بُنْيَة v2:")
    print(match.group(0))
else:
    print("الرابط لسا ما ظهر، استني 5 ثواني وشغّلي نفس الخلية مرة ثانية.")

رابط بُنْيَة v2:
https://solve-affiliates-alive-par.trycloudflare.com


In [ ]:
import os
import re
import time
import socket
import subprocess
import requests

# 1) نتأكد إن بُنْيَة نفسها شغالة
try:
    health = requests.get(
        "http://127.0.0.1:8503/_stcore/health",
        timeout=5
    )
    print("Streamlit:", health.status_code, health.text)
except Exception as e:
    print("Streamlit مش شغال على 8503")
    print(e)
    raise SystemExit

# 2) نوقف Cloudflare القديم فقط
subprocess.run(
    ["pkill", "-f", "cloudflared"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

log_path = "/content/cloudflare_bunya_v2.log"

# 3) نعمل Tunnel جديد
cf = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8503"
    ],
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT
)

print("جاري تجهيز الرابط...")

valid_url = None

# 4) ننتظر الرابط ونتأكد أن DNS شغال فعليًا
for _ in range(30):

    time.sleep(2)

    if not os.path.exists(log_path):
        continue

    log = open(
        log_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ).read()

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log
    )

    if not match:
        continue

    url = match.group(0)
    host = url.replace("https://", "")

    try:
        socket.gethostbyname(host)

        r = requests.get(
            url,
            timeout=10
        )

        if r.status_code < 500:
            valid_url = url
            break

    except:
        pass

if valid_url:
    print("\nرابط بُنْيَة الشغال:")
    print(valid_url)
else:
    print("\nالرابط ما ثبت لسا. ابعتيلي آخر السجل:")
    print(open(log_path, "r", errors="ignore").read()[-1500:])

Streamlit: 200 ok
جاري تجهيز الرابط...

رابط بُنْيَة الشغال:
https://disc-exec-testing-april.trycloudflare.com


In [ ]:
from pathlib import Path

app_path = Path("/content/app_BUNYA_v2 (2).py")

text = app_path.read_text(encoding="utf-8")

text = text.replace("بُنْيَة", "بينه")
text = text.replace("بُنْيَة", "بينه")

app_path.write_text(text, encoding="utf-8")

print("تم تغيير اسم المشروع إلى: بينه")

تم تغيير اسم المشروع إلى: بينه
